# Core symbolic experiment suite

Cleaned from the author-supplied `v4-results(3).ipynb`. Environment-transfer, backup, and superseded visual cells were removed; the experiment/model/training logic in the retained cells is unchanged.

Use `PRESET="smoke"` first for a pipeline check, then `PRESET="full"` for paper runs. Results are append-only and resumable.


In [ ]:
# !pip install torch torchvision numpy pandas matplotlib

import copy, glob, hashlib, itertools, json, math, os, pathlib, platform, random, time
from dataclasses import dataclass, replace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================== CHOOSE HERE ==============================
MODALITY = "abstract"        # "abstract" | "mnist" | "cifar10" | "cifar100"
TARGET   = "encoder"         # visual only: "encoder" | "true_label"
PRESET   = "full"            # "smoke" | "quick" | "full"
RESUME   = True              # skip any unit of work already in results/
WORKDIR  = os.environ.get("PAPER_WORKDIR", ".")   # point at Drive to survive Colab
# =========================================================================
#
# PRESET = "smoke" runs every block on the full grid with tiny budgets. It is
# a correctness check, not a result: ~15 minutes on a GPU, and it proves the
# whole pipeline executes before you commit a day of compute to "full".
#
# RESUME = True means a killed session loses at most one run. Every finished
# run is appended to results/<block>__<modality>.jsonl and skipped on restart.
# Delete a block's .jsonl to force it to recompute.

RUN = {"sanity": True, "free": True, "boundary": True, "costforms": True,
       "width": True, "ood": True, "replication": True, "cores": True,
       "schedule": True, "inversion": True, "switch": True, "silencing": True,
       "reversibility": True, "negatives": True, "lamstar": True, "swap": True,
       "sharebias": True, "figures": True, "derived": True, "appendix": True,
       "mamba": True}

IS_VISUAL = MODALITY != "abstract"
TAG = MODALITY

WORK = pathlib.Path(WORKDIR).expanduser().resolve()
# smoke and quick outputs live in their own tree, so a pipeline check can never
# be mixed into the paper's numbers by a resumed sweep picking up stale records
_sfx = "" if PRESET == "full" else f"_{PRESET}"
RESDIR, CKPTDIR = WORK / f"results{_sfx}", WORK / f"checkpoints{_sfx}"
FIGDIR, TABDIR = WORK / f"figures{_sfx}", WORK / f"tables{_sfx}"
for _d in (RESDIR, CKPTDIR, FIGDIR, TABDIR):
    _d.mkdir(parents=True, exist_ok=True)


def S(full, quick=None, smoke=None):
    """Budget for the current preset. Defaults keep the ratios sane."""
    return {"full": full,
            "quick": quick if quick is not None else max(full // 3, 100),
            "smoke": smoke if smoke is not None else max(full // 40, 100)}[PRESET]


# Headline results (boundary, inversion, OOD, switch, reversibility, width) get
# five seeds; the large factorial grids get three. Both drop to one under smoke.
SEEDS_MAIN = {"full": [0, 1, 2, 3, 4], "quick": [0, 1], "smoke": [0]}[PRESET]
SEEDS_AUX  = {"full": [0, 1, 2],       "quick": [0, 1], "smoke": [0]}[PRESET]

# One accuracy floor for every crossing in the paper. A crossing is only
# meaningful where the model still solves the task, and the floor decides which
# cells are excluded — so it has to be the same number everywhere or the
# crossings are not comparable across sections. State this value in the paper.
ACC_FLOOR = 0.90

STEPS = S(2500, 1200, 150)     # default per-run budget
LONG  = S(16000, 4000, 400)    # drift / equilibrium budget
XLONG = S(24000, 6000, 600)    # the long-budget arm of the Delta measurement (S6.1)

print(f"device {DEVICE} | modality {MODALITY} | preset {PRESET} | resume {RESUME}")
print(f"work dir {WORK}")
print(f"seeds: main {SEEDS_MAIN}, aux {SEEDS_AUX} | steps {STEPS} / {LONG} / {XLONG}")
if DEVICE.type == "cpu":
    print("!! CPU — roughly 10x slower. Colab: Runtime -> Change runtime type -> GPU.")
if PRESET == "smoke":
    print("!! SMOKE preset: numbers are meaningless. Use it to check the pipeline runs.")


def csv(name):
    return str(RESDIR / f"{name}__{TAG}.csv")


def set_seed(s):
    torch.manual_seed(s); np.random.seed(s); random.seed(s)


## Persistence and resume

A full run is several million optimiser steps, which is longer than a Colab
session. Every unit of work is content-hashed on its specification and appended
to a JSONL file the moment it finishes, so restarting the notebook re-reads what
is already done and runs only what is missing.

Two granularities. Sweeps resume at the level of one `(condition, seed)` run.
Runs of 8k steps and up also checkpoint model and optimiser state mid-flight, so
a 24,000-step run survives a disconnect. A resumed run restarts its data stream
from a fresh RNG state: since trials are generated i.i.d. online this is
statistically equivalent, but it is not bit-identical, and initialisation is
still exactly seeded either way.

To force a recomputation, delete that block's `.jsonl`.


In [ ]:
def _jsonable(o):
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, torch.Tensor):
        return o.detach().cpu().tolist()
    if isinstance(o, pathlib.Path):
        return str(o)
    return str(o)


def keyof(spec):
    """Content hash of a run specification. Two runs with the same spec are the
    same experiment, so the second one is skipped."""
    return hashlib.md5(
        json.dumps(spec, sort_keys=True, default=_jsonable).encode()).hexdigest()[:16]


def cfg_tag(cfg=None):
    if cfg is None:
        return "default"
    return ",".join(f"{k}={getattr(cfg, k)}" for k in sorted(cfg.__dataclass_fields__))


class Store:
    """Append-only JSONL of finished units of work, one file per block.

    A unit is whatever a block can afford to lose: usually one (condition, seed)
    training run. Records are flushed immediately, so a killed kernel loses at
    most the run in flight. A truncated final line is discarded on load.
    """

    def __init__(self, block):
        self.block = block
        self.path = RESDIR / f"{block}__{TAG}.jsonl"
        self.recs, bad = {}, 0
        if RESUME and self.path.exists():
            for line in self.path.read_text().splitlines():
                if not line.strip():
                    continue
                try:
                    r = json.loads(line)
                    self.recs[r["_key"]] = r
                except (json.JSONDecodeError, KeyError):
                    bad += 1
            if bad:
                print(f"  [{block}] discarded {bad} unreadable record(s)")

    def has(self, spec):
        return RESUME and keyof(spec) in self.recs

    def get(self, spec):
        return self.recs[keyof(spec)]

    def put(self, spec, payload):
        r = {"_key": keyof(spec), "_spec": spec, "_t": time.time(), **payload}
        self.recs[r["_key"]] = r
        with open(self.path, "a") as f:
            f.write(json.dumps(r, default=_jsonable) + "\n")
            f.flush(); os.fsync(f.fileno())
        return r

    def frame(self):
        """Flatten to one row per result. A record either carries metric fields
        directly or a `rows` list, in which case each entry becomes a row."""
        out = []
        for r in self.recs.values():
            spec = {k: v for k, v in r["_spec"].items()}
            body = {k: v for k, v in r.items()
                    if not k.startswith("_") and k not in ("trace", "rows")}
            if "rows" in r:
                for row in r["rows"]:
                    out.append({**spec, **body, **row})
            else:
                out.append({**spec, **body})
        return pd.DataFrame(out)

    def traces(self):
        out = []
        for r in self.recs.values():
            for t in r.get("trace", []):
                out.append({**r["_spec"], **t})
        return pd.DataFrame(out)

    def elapsed(self):
        return sum(r.get("_elapsed", 0.0) for r in self.recs.values())


def cached(store, spec, fn, label=None):
    """Run `fn()` unless this exact spec is already on disk. `fn` returns the
    payload dict; anything not JSON-serialisable must be checkpointed separately."""
    if store.has(spec):
        return store.get(spec)
    t0 = time.time()
    payload = fn()
    payload["_elapsed"] = time.time() - t0
    r = store.put(spec, payload)
    if label:
        print(f"  [{store.block}] {label}  {payload['_elapsed']/60:.1f}m")
    return r


def ckpt_path(name):
    return CKPTDIR / f"{name}__{TAG}.pt"


def ckpt_save(name, obj):
    p = ckpt_path(name)
    torch.save(obj, str(p) + ".tmp")
    os.replace(str(p) + ".tmp", str(p))


def ckpt_load(name):
    p = ckpt_path(name)
    if not (RESUME and p.exists()):
        return None
    try:
        return torch.load(str(p), map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(str(p), map_location=DEVICE)
    except Exception as e:
        print(f"  checkpoint {p.name} unreadable ({e}) — recomputing")
        return None


def save_model(name, m):
    ckpt_save(name, {"state": m.state_dict()})


def load_model_into(name, m):
    ck = ckpt_load(name)
    if ck is None:
        return False
    m.load_state_dict(ck["state"])
    return True


print(f"persistence: results -> {RESDIR}   checkpoints -> {CKPTDIR}")


## Configuration

`n_out` is the readout width and therefore chance level: 8 for symbols, 10 for
MNIST/CIFAR-10, 100 for CIFAR-100. Dependency scores are normalised by each
model's own headroom above its own chance, so the measures are comparable
across modalities without rescaling.

In [ ]:
PAD, DELAY, CUE, MARK = 0, 1, 2, 3
N_SPECIAL = 4
MAX_POS = 24
D_FEAT = 64


@dataclass
class Config:
    n_symbols: int = 8            # abstract alphabet
    max_query_pos: int = MAX_POS
    base_delay: int = 3
    d_model: int = 64
    lr: float = 3e-3
    batch_size: int = 128
    weight_decay: float = 0.0
    grad_clip: float = 1.0


CFG = Config()
N_CLASSES = {"abstract": CFG.n_symbols, "mnist": 10, "cifar10": 10,
             "cifar100": 100}[MODALITY]
# visual packing: [features | query one-hot | is_delay | is_query | is_mark]
IN_DIM = D_FEAT + MAX_POS + 3


def vocab_size(cfg):
    return N_SPECIAL + cfg.n_symbols + cfg.max_query_pos


def token_str(t, cfg=CFG):
    t = int(t)
    return {PAD: "_", DELAY: ".", CUE: "?", MARK: "*"}.get(
        t, chr(ord("A") + t - N_SPECIAL) if t < N_SPECIAL + cfg.n_symbols
        else f"Q{t - N_SPECIAL - cfg.n_symbols}")

## Frozen encoder (visual modalities only)

`AdaptiveAvgPool2d(3)` rather than `(1)`: collapsing the feature map to one
number per channel discards the spatial structure digits and objects depend on,
and cost about seven points of MNIST accuracy in an earlier version.

The encoder is **frozen** after training. A jointly-trained encoder would be a
third pathway able to compress or preserve information, and the two-route
decomposition every measurement here relies on would no longer partition the
information flow.

In [ ]:
BANK = {}

if IS_VISUAL:
    import torchvision
    from torchvision import transforms as T

    class Encoder(nn.Module):
        def __init__(self, in_ch, width, d_feat=D_FEAT, n_classes=10):
            super().__init__()
            a, b, c = width
            self.conv = nn.Sequential(
                nn.Conv2d(in_ch, a, 3, padding=1), nn.BatchNorm2d(a), nn.ReLU(),
                nn.Conv2d(a, a, 3, padding=1), nn.BatchNorm2d(a), nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(),
                nn.Conv2d(b, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(b, c, 3, padding=1), nn.BatchNorm2d(c), nn.ReLU(),
                nn.AdaptiveAvgPool2d(3))
            self.fc = nn.Linear(c * 9, d_feat)
            self.head = nn.Linear(d_feat, n_classes)

        def features(self, x):
            return self.fc(self.conv(x).flatten(1))

        def forward(self, x):
            return self.head(F.relu(self.features(x)))

    SPEC = {
        "mnist":    dict(cls=torchvision.datasets.MNIST, ch=1, ep=4,
                         w=(32, 64, 128), mean=(0.1307,), std=(0.3081,), aug=False),
        "cifar10":  dict(cls=torchvision.datasets.CIFAR10, ch=3, ep=20,
                         w=(64, 128, 256), mean=(0.4914, 0.4822, 0.4465),
                         std=(0.247, 0.243, 0.261), aug=True),
        "cifar100": dict(cls=torchvision.datasets.CIFAR100, ch=3, ep=30,
                         w=(64, 128, 256), mean=(0.5071, 0.4865, 0.4409),
                         std=(0.2673, 0.2564, 0.2762), aug=True),
    }[MODALITY]

    base = [T.ToTensor(), T.Normalize(SPEC["mean"], SPEC["std"])]
    tf_tr = T.Compose(([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()]
                       if SPEC["aug"] else []) + base)
    tf_te = T.Compose(base)
    ds_tr = SPEC["cls"]("./data", train=True, download=True, transform=tf_tr)
    ds_te = SPEC["cls"]("./data", train=False, download=True, transform=tf_te)

    set_seed(0)
    enc = Encoder(SPEC["ch"], SPEC["w"], n_classes=N_CLASSES).to(DEVICE)
    opt_e = torch.optim.AdamW(enc.parameters(), lr=2e-3, weight_decay=5e-4)
    dl = torch.utils.data.DataLoader(ds_tr, batch_size=256, shuffle=True,
                                     num_workers=2, drop_last=True)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt_e, 2e-3, SPEC["ep"] * len(dl))
    for ep in range(SPEC["ep"]):
        enc.train()
        for xb, yb in dl:
            loss = F.cross_entropy(enc(xb.to(DEVICE)), yb.to(DEVICE))
            opt_e.zero_grad(set_to_none=True); loss.backward(); opt_e.step(); sch.step()
        if (ep + 1) % max(SPEC["ep"] // 5, 1) == 0:
            print(f"  encoder epoch {ep+1}/{SPEC['ep']}  loss {loss.item():.3f}")
    enc.eval()
    for p in enc.parameters():
        p.requires_grad_(False)

    @torch.no_grad()
    def encode_all(dataset, perturb=None, bs=512):
        dl = torch.utils.data.DataLoader(dataset, batch_size=bs, num_workers=2)
        fs, ys, ps = [], [], []
        for xb, yb in dl:
            xb = xb.to(DEVICE)
            if perturb is not None:
                xb = perturb(xb)
            f = enc.features(xb)
            fs.append(f); ys.append(yb.to(DEVICE))
            ps.append(enc.head(F.relu(f)).argmax(-1))
        return torch.cat(fs), torch.cat(ys), torch.cat(ps)

    f_tr, y_tr, p_tr = encode_all(ds_tr)
    f_te, y_te, p_te = encode_all(ds_te)
    ENC_ACC = (p_te == y_te).float().mean().item()
    BANK = {"f_tr": f_tr, "t_tr": p_tr if TARGET == "encoder" else y_tr,
            "f_te": f_te, "t_te": p_te if TARGET == "encoder" else y_te,
            "y_te": y_te, "p_te": p_te, "enc_acc": ENC_ACC, "test_ds": ds_te}
    print(f"\nfrozen encoder test accuracy: {ENC_ACC:.4f}  ({N_CLASSES} classes, "
          f"feature dim {D_FEAT})")
    print(f"target = {TARGET}  →  working-memory ceiling is "
          f"{'1.000 regardless of encoder quality' if TARGET=='encoder' else f'{ENC_ACC:.3f}'}")

## Tasks

Four paradigms, defined once and dispatched on modality. `selective_recall` is
primary: the query is unknown until the end, so nothing can be stored
selectively. `distractor` holds the retention interval fixed and varies
interference alone, dissociating two things that grow together elsewhere.

`affine_track` is the **positive control**. It composes maps `s ↦ (a·s+b) mod 8`
with `a` coprime to 8; composition does not commute, so a permutation-invariant
attention read provably cannot succeed and recurrence must be used. Before any
`hidden_dep = 0` elsewhere can be interpreted, the measure has to be shown
capable of reporting recurrence. In visual modalities each image's class
indexes the operation, so the control transfers.

In [ ]:
AFFINE_OPS = [(1, 1), (1, 3), (3, 0), (5, 0)]


def _pack(feats, n_delay, qpos=None, mark=False, device=DEVICE):
    """Pack a visual trial into one float tensor so every downstream metric works
    unchanged: [features | query one-hot | is_delay | is_query | is_mark]."""
    B, n, _ = feats.shape
    T_ = n + n_delay + 1 + (1 if mark else 0)
    x = torch.zeros(B, T_, IN_DIM, device=device)
    off = 1 if mark else 0
    if mark:
        x[:, 0, D_FEAT + MAX_POS + 2] = 1.0
    x[:, off:off + n, :D_FEAT] = feats
    x[:, off + n:off + n + n_delay, D_FEAT + MAX_POS] = 1.0
    x[:, -1, D_FEAT + MAX_POS + 1] = 1.0
    if qpos is not None:
        x[torch.arange(B, device=device), -1, D_FEAT + qpos] = 1.0
    return x


class Task:
    name, n_out = "base", None

    def __init__(self, cfg=None, split="train"):
        self.cfg = cfg or CFG
        self.split = split
        if IS_VISUAL:
            self.feats = BANK["f_tr"] if split == "train" else BANK["f_te"]
            self.targets = BANK["t_tr"] if split == "train" else BANK["t_te"]

    def _sample(self, B, n, device):
        """Returns (feats_or_none, symbol_labels) for n items."""
        if IS_VISUAL:
            idx = torch.randint(0, self.feats.shape[0], (B, n), device=device)
            return self.feats[idx], self.targets[idx]
        s = torch.randint(0, self.cfg.n_symbols, (B, n), device=device)
        return None, s

    def show(self, load, n=3):
        b = self.generate(n, load, device=DEVICE)
        if IS_VISUAL:
            print(f"  x {tuple(b['x'].shape)} float   y {b['y'][:n].tolist()}")
        else:
            for i in range(n):
                print("  " + " ".join(token_str(t) for t in b["x"][i].cpu())
                      + f"  →  {token_str(int(b['y'][i]) + N_SPECIAL)}")


class SelectiveRecall(Task):
    name = "selective_recall"

    @property
    def n_out(self):
        # honour a non-default alphabet (the swap-error analysis in S6.6 needs one)
        return N_CLASSES if IS_VISUAL else self.cfg.n_symbols

    def generate(self, B, load, extra_delay=0, device=DEVICE, extra_items=0):
        c, n = self.cfg, load + extra_items
        assert n <= c.max_query_pos
        feats, lab = self._sample(B, n, device)
        j = torch.randint(0, load, (B,), device=device)     # trained positions only
        ar = torch.arange(B, device=device)
        nd = c.base_delay + extra_delay
        if IS_VISUAL:
            x = _pack(feats, nd, qpos=j, device=device)
        else:
            delay = torch.full((B, nd), DELAY, device=device, dtype=torch.long)
            q = (j + N_SPECIAL + c.n_symbols).unsqueeze(1)
            x = torch.cat([lab + N_SPECIAL, delay, q], 1)
        return {"x": x, "y": lab[ar, j], "key_pos": j, "items": lab}


class DelayedRecall(Task):
    name = "delayed_recall"

    @property
    def n_out(self):
        # honour a non-default alphabet (the swap-error analysis in S6.6 needs one)
        return N_CLASSES if IS_VISUAL else self.cfg.n_symbols

    def generate(self, B, load, extra_delay=0, device=DEVICE):
        c = self.cfg
        feats, lab = self._sample(B, load, device)
        nd = c.base_delay + extra_delay
        if IS_VISUAL:
            x = _pack(feats, nd, qpos=None, device=device)
        else:
            delay = torch.full((B, nd), DELAY, device=device, dtype=torch.long)
            cue = torch.full((B, 1), CUE, device=device, dtype=torch.long)
            x = torch.cat([lab + N_SPECIAL, delay, cue], 1)
        return {"x": x, "y": lab[:, 0],
                "key_pos": torch.zeros(B, dtype=torch.long, device=device),
                "items": lab}


class Distractor(Task):
    name = "distractor"

    @property
    def n_out(self):
        # honour a non-default alphabet (the swap-error analysis in S6.6 needs one)
        return N_CLASSES if IS_VISUAL else self.cfg.n_symbols

    def generate(self, B, load, extra_delay=0, device=DEVICE):
        c = self.cfg
        feats, lab = self._sample(B, load + 1, device)
        nd = c.base_delay + extra_delay
        if IS_VISUAL:
            x = _pack(feats, nd, qpos=None, mark=True, device=device)
        else:
            mark = torch.full((B, 1), MARK, device=device, dtype=torch.long)
            delay = torch.full((B, nd), DELAY, device=device, dtype=torch.long)
            cue = torch.full((B, 1), CUE, device=device, dtype=torch.long)
            x = torch.cat([mark, lab + N_SPECIAL, delay, cue], 1)
        return {"x": x, "y": lab[:, 0],
                "key_pos": torch.ones(B, dtype=torch.long, device=device),
                "items": lab}


class AffineTrack(Task):
    """Positive control: non-commutative composition, so a single attention read
    over a bag of values cannot recover the answer."""
    name = "affine_track"
    n_out = 8

    def generate(self, B, load, extra_delay=0, device=DEVICE):
        c = self.cfg
        feats, lab = self._sample(B, load, device)
        ops = lab % len(AFFINE_OPS)
        A = torch.tensor([a for a, _ in AFFINE_OPS], device=device)
        Bv = torch.tensor([b for _, b in AFFINE_OPS], device=device)
        s = torch.zeros(B, dtype=torch.long, device=device)
        for t in range(load):
            o = ops[:, t]
            s = (A[o] * s + Bv[o]) % 8
        nd = c.base_delay + extra_delay
        if IS_VISUAL:
            x = _pack(feats, nd, qpos=None, device=device)
        else:
            delay = torch.full((B, nd), DELAY, device=device, dtype=torch.long)
            cue = torch.full((B, 1), CUE, device=device, dtype=torch.long)
            x = torch.cat([ops + N_SPECIAL, delay, cue], 1)
        return {"x": x, "y": s,
                "key_pos": torch.full((B,), load - 1, dtype=torch.long, device=device),
                "items": ops}


TASKS = {t.name: t for t in [SelectiveRecall, DelayedRecall, Distractor, AffineTrack]}


def make_task(name="selective_recall", cfg=None, split="train"):
    return TASKS[name](cfg, split)

## Models

The additive two-route readout is the whole method:

> **logits = W_h · f(h_T)  +  W_c · g(c)**

`h_T` is the final recurrent state; `c` is a single-query attention read with
the query from `h_T` and keys/values from `H[:, :−1]`. Excluding the final
timestep is necessary — otherwise `v(h_T)` smuggles the recurrent state through
the context path and window ablation measures the wrong thing.

The routes meet only as a sum, so each can be ablated or priced independently.
`attention=False` removes the retrieval route entirely, which is what makes the
§4.1 control an actual control.

In [ ]:
def ablate(v, spec):
    """Resample ablation: replace a trial's activation with another trial's, so the
    readout still sees an in-distribution vector that is uninformative about this
    trial. Zeroing would be off-manifold."""
    if v is None or spec is None:
        return v
    mode, strength = spec if isinstance(spec, (tuple, list)) else (spec, 1.0)
    if strength <= 0:
        return v
    if mode == "noise":
        return v + strength * v.std() * torch.randn_like(v)
    repl = {"shuffle": lambda: v[torch.randperm(v.shape[0], device=v.device)],
            "mean": lambda: v.mean(0, keepdim=True).expand_as(v),
            "zero": lambda: torch.zeros_like(v)}[mode]()
    return (1.0 - strength) * v + strength * repl


class DiagLinearRNN(nn.Module):
    """h_t = a ⊙ h_{t-1} + B x_t — the S4D / LRU core."""

    def __init__(self, d):
        super().__init__()
        self.a_logit = nn.Parameter(torch.linspace(1.0, 4.0, d))
        self.B, self.C, self.gate = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)

    def forward(self, x):
        a = torch.sigmoid(self.a_logit); u = self.B(x)
        h = x.new_zeros(x.shape[0], x.shape[2]); hs = []
        for t in range(x.shape[1]):
            h = a * h + u[:, t]; hs.append(h)
        H = torch.stack(hs, 1)
        return self.C(H) * torch.sigmoid(self.gate(x)), h


class SelectiveSSM(nn.Module):
    """Input-dependent decay and gain — Mamba's selectivity mechanism."""

    def __init__(self, d):
        super().__init__()
        self.a_base = nn.Parameter(torch.linspace(1.0, 4.0, d))
        self.a_sel, self.b_sel = nn.Linear(d, d), nn.Linear(d, d)
        self.B, self.C, self.gate = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)

    def forward(self, x):
        a = torch.sigmoid(self.a_base.view(1, 1, -1) + self.a_sel(x))
        u = torch.sigmoid(self.b_sel(x)) * self.B(x)
        h = x.new_zeros(x.shape[0], x.shape[2]); hs = []
        for t in range(x.shape[1]):
            h = a[:, t] * h + u[:, t]; hs.append(h)
        H = torch.stack(hs, 1)
        return self.C(H) * torch.sigmoid(self.gate(x)), h


EXTRA_CORES = {}          # late-registered cores (see the Mamba cell at the end)


class CausalTransformer(nn.Module):
    """Attention-only sequence core, capacity-matched against the recurrent ones
    for S3.5. It is registered with HAS_ATTN=False because it has no *retrieval
    route in the readout* — self-attention inside the core is not the priced
    pathway. Its role in the paper is the parameter-count reference point."""

    def __init__(self, d, n_layers=2, n_heads=4, max_len=256):
        super().__init__()
        self.pos = nn.Parameter(torch.zeros(1, max_len, d))
        nn.init.normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=n_heads, dim_feedforward=2 * d, dropout=0.0,
            batch_first=True, norm_first=True, activation="gelu")
        self.enc = nn.TransformerEncoder(layer, n_layers)

    def forward(self, x):
        T_ = x.shape[1]
        h = x + self.pos[:, :T_]
        mask = torch.triu(torch.ones(T_, T_, device=x.device, dtype=torch.bool), 1)
        H = self.enc(h, mask=mask)
        return H, H[:, -1]


class TwoRoute(nn.Module):
    CORES = {"gru": nn.GRU, "lstm": nn.LSTM, "rnn": nn.RNN}
    K = 10.0

    def __init__(self, cfg=None, core="gru", d_h=None, d_c=None, n_out=None,
                 attention=True, gate=False, exclude_last=True):
        super().__init__()
        cfg = cfg or CFG
        d_h = d_h or cfg.d_model
        d_c = d_c or d_h
        self.cfg, self.d_h, self.d_c = cfg, d_h, d_c
        self.attention, self.use_gate, self.exclude_last = attention, gate, exclude_last
        n_out = n_out or N_CLASSES
        self.emb = nn.Linear(IN_DIM, d_h) if IS_VISUAL else nn.Embedding(vocab_size(cfg), d_h)
        if core in self.CORES:
            self.rnn = self.CORES[core](d_h, d_h, batch_first=True)
        elif core == "linear":
            self.rnn = DiagLinearRNN(d_h)
        elif core == "selective":
            self.rnn = SelectiveSSM(d_h)
        elif core == "transformer":
            self.rnn = CausalTransformer(d_h)
        elif core in EXTRA_CORES:
            self.rnn = EXTRA_CORES[core](d_h)
        else:
            raise ValueError(core)
        self.proj_h = nn.Sequential(nn.Linear(d_h, d_h), nn.GELU())
        self.w_h = nn.Linear(d_h, n_out)
        if attention:
            self.q, self.k, self.v = (nn.Linear(d_h, d_c), nn.Linear(d_h, d_c),
                                      nn.Linear(d_h, d_c))
            self.scale = d_c ** -0.5
            self.proj_c = nn.Sequential(nn.Linear(d_c, d_c), nn.GELU())
            self.w_c = nn.Linear(d_c, n_out, bias=False)
        if gate:
            self.gate_logit = nn.Parameter(torch.tensor(2.0))

    @property
    def z(self):
        return torch.sigmoid(self.gate_logit) if self.use_gate else None

    def attend(self, hq, H, window=None):
        B, T_, _ = H.shape
        sc = (self.q(hq).unsqueeze(1) @ self.k(H).transpose(1, 2)).squeeze(1) * self.scale
        if window is not None:
            idx = torch.arange(T_, device=H.device)
            sc = sc.masked_fill((idx < T_ - window).unsqueeze(0), float("-inf"))
        w = sc.softmax(-1)
        return torch.einsum("bt,btd->bd", w, self.v(H)), w

    def forward(self, x, intervene=None, return_internals=False):
        iv = intervene or {}
        H, _ = self.rnn(self.emb(x))
        h_T = H[:, -1, :]
        h_r = ablate(h_T, iv.get("hidden"))
        if iv.get("erase") is not None:
            P = iv["erase"]; h_r = h_r - (h_r @ P) @ P.T
        lh = self.w_h(self.proj_h(h_r))
        if not self.attention:
            out = {"logits": lh, "parts": {"hidden": lh, "context": torch.zeros_like(lh)}}
            if return_internals:
                out.update({"H": H, "h_T": h_T, "ctx": None, "attn": None})
            return out
        H_ctx = H[:, :-1, :] if self.exclude_last else H
        ctx, aw = self.attend(ablate(h_T, iv.get("query")), H_ctx,
                              window=iv.get("attn_window"))
        ctx = ablate(ctx, iv.get("context"))
        lc = self.w_c(self.proj_c(ctx))
        if self.use_gate:      # mixture gate: normalise BOTH routes, gate the mixture,
            nrm = lambda u: u / (u.norm(dim=-1, keepdim=True) + 1e-6)
            lh, lc = self.K * nrm(lh), self.K * self.z * nrm(lc)   # so no rescaling undoes it
        out = {"logits": lh + lc, "parts": {"hidden": lh, "context": lc}}
        if return_internals:
            out.update({"H": H, "h_T": h_T, "ctx": ctx, "attn": aw})
        return out


def route_params(m):
    """Split the parameters into the two routes plus what they share.

    The mixture gate belongs to the retrieval route: it scales that route's
    contribution, and an LR handicap applied to "attention" should reach it.
    Leaving it out of every group meant it never reached the optimiser at all,
    so z stayed at sigmoid(2.0) for every lambda and the gate cost form
    produced a flat sweep. The assertion in `train` now makes that class of
    omission fail loudly instead of silently.
    """
    rec = list(m.proj_h.parameters()) + list(m.w_h.parameters())
    att = ((list(m.q.parameters()) + list(m.k.parameters()) + list(m.v.parameters())
            + list(m.proj_c.parameters()) + list(m.w_c.parameters()))
           if m.attention else [])
    if getattr(m, "use_gate", False):
        att = att + [m.gate_logit]
    shared = list(m.emb.parameters()) + list(m.rnn.parameters())
    return rec, att, shared


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


CORE_OF = {"rnn": "rnn", "gru": "gru", "lstm": "lstm", "gru_attn": "gru",
           "lstm_attn": "lstm", "linear_attn": "linear", "selective_attn": "selective",
           "transformer": "transformer"}
HAS_ATTN = {"rnn": False, "gru": False, "lstm": False, "gru_attn": True,
            "lstm_attn": True, "linear_attn": True, "selective_attn": True,
            "transformer": False}


CAP_TABLE = []


def match_capacity(cfg=None, verbose=True):
    """Reference is the attention-FREE GRU, so the target reflects a plain
    recurrent model and the attention arms pay for their extra parameters."""
    cfg = cfg or CFG
    target = count_params(TwoRoute(cfg, "gru", d_h=cfg.d_model, attention=False))
    dims, rows = {}, []
    for name in CORE_OF:
        best = min(range(8, 400, 4),
                   key=lambda d: abs(count_params(
                       TwoRoute(cfg, CORE_OF[name], d_h=d,
                                attention=HAS_ATTN[name])) - target))
        dims[name] = best
        rows.append({"model": name, "attention": HAS_ATTN[name], "d": best,
                     "params": count_params(TwoRoute(cfg, CORE_OF[name], d_h=best,
                                                     attention=HAS_ATTN[name]))})
    if verbose:
        print(f"target {target:,} params (attention-free gru @ d={cfg.d_model})")
        print(pd.DataFrame(rows).to_string(index=False))
    CAP_TABLE[:] = rows
    return dims


DIMS = match_capacity()


def build(name="gru_attn", cfg=None, d_h=None, d_c=None, gate=False, n_out=None):
    return TwoRoute(cfg or CFG, CORE_OF[name], d_h=d_h or DIMS[name], d_c=d_c,
                    n_out=n_out, attention=HAS_ATTN[name], gate=gate)

## Training, metrics, crossings

Ablation is by resampling across the batch. Dependencies are normalised by each
model's own headroom above its own chance level, so an 8-way symbolic readout
and a 100-way CIFAR-100 readout give comparable numbers.

`crossing_any` detects a sign change in either direction: on a load axis the
difference goes negative→positive, on a λ axis positive→negative. A
direction-specific version silently returned NaN for every width crossing.

In [ ]:
def train(task, load, lam=0.0, form="l2", seed=0, steps=2500, model=None,
          name="gru_attn", d_h=None, d_c=None, gate=False, schedule="cosine",
          p_drop=0.0, log_every=None, cfg=None, handicap=None, lam_after=None,
          switch=None, ckpt_key=None, ckpt_every=2000):
    """`schedule='constant'` wherever the drift itself is studied — cosine decay
    freezes it partway and confounds the measurement. `handicap` scales a group's
    LEARNING RATE; scaling its gradient is nearly a no-op under Adam.

    `ckpt_key` enables mid-run checkpointing, for the 12k–24k step runs where
    losing the whole run to a disconnect is expensive. Resuming restarts the
    data stream from a fresh RNG state, so a resumed run is not bit-identical to
    an uninterrupted one; it is statistically equivalent because trials are
    generated i.i.d. online. Seeds still fix initialisation exactly.
    """
    cfg = cfg or CFG
    set_seed(seed)
    m = (model or build(name, cfg, d_h, d_c, gate, task.n_out)).to(DEVICE)
    rec, att, shared = route_params(m)
    groups = [{"params": rec, "name": "recurrent"}, {"params": shared, "name": "shared"}]
    if att:
        groups.insert(1, {"params": att, "name": "attention"})
    opt = torch.optim.AdamW(groups, lr=cfg.lr, weight_decay=cfg.weight_decay)
    _grouped = sum(len(g["params"]) for g in groups)
    _total = len([p for p in m.parameters() if p.requires_grad])
    assert _grouped == _total, (
        f"{_total - _grouped} parameter tensor(s) are not in any optimiser group "
        f"and will never be updated. Add them to route_params.")
    trace, start = [], 1

    if ckpt_key is not None:
        ck = ckpt_load(f"run_{ckpt_key}")
        if ck is not None and ck.get("steps") == steps:
            m.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
            trace, start = ck["trace"], ck["step"] + 1
            if start <= steps:
                print(f"    resumed run {ckpt_key} at step {ck['step']}/{steps}")

    for step in range(start, steps + 1):
        cos = 0.5 * (1 + math.cos(math.pi * step / steps)) if schedule == "cosine" else 1.0
        for g in opt.param_groups:
            f = (handicap["factor"] if handicap and step <= handicap["until"]
                 and g["name"] == handicap["route"] else 1.0)
            g["lr"] = cfg.lr * cos * f
        cur = lam if (switch is None or step <= switch) else lam_after
        m.train()
        b = task.generate(cfg.batch_size, load)
        iv = ({"context": ("zero", 1.0)} if p_drop > 0 and random.random() < p_drop
              else None)
        o = m(b["x"], intervene=iv)
        loss = F.cross_entropy(o["logits"], b["y"])
        if cur and cur > 0 and m.attention:
            lc = o["parts"]["context"]
            loss = loss + cur * ({"l2": lambda: lc.norm(dim=-1).mean(),
                                  "l2sq": lambda: lc.pow(2).sum(-1).mean(),
                                  "gate": lambda: m.z}[form]())
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(m.parameters(), cfg.grad_clip)
        opt.step()
        if log_every and (step % log_every == 0 or step == 1):
            trace.append({"step": step, **deps(m, task, load, cfg)})
        if ckpt_key is not None and step % ckpt_every == 0 and step < steps:
            ckpt_save(f"run_{ckpt_key}",
                      {"model": m.state_dict(), "opt": opt.state_dict(),
                       "step": step, "steps": steps, "trace": trace})

    if ckpt_key is not None:
        p = ckpt_path(f"run_{ckpt_key}")
        if p.exists():
            p.unlink()          # the finished run lives in the results store now
    return m, pd.DataFrame(trace)


@torch.no_grad()
def accuracy(m, task, load, intervene=None, n_batches=8, cfg=None, **gen):
    cfg = cfg or CFG
    m.eval(); c = t = 0
    for _ in range(n_batches):
        b = task.generate(cfg.batch_size, load, **gen)
        c += (m(b["x"], intervene=intervene)["logits"].argmax(-1) == b["y"]).sum().item()
        t += b["y"].numel()
    m.train(); return c / t


@torch.no_grad()
def deps(m, task, load, cfg=None, n_batches=8, mode="shuffle"):
    cfg = cfg or CFG
    chance = 1.0 / m.w_h.out_features
    acc = accuracy(m, task, load, n_batches=n_batches, cfg=cfg)
    den = max(acc - chance, 1e-6)
    d = lambda a: float(np.clip((acc - a) / den, 0, 1))
    hd = d(accuracy(m, task, load, {"hidden": (mode, 1.0)}, n_batches, cfg))
    if not m.attention:
        return {"acc_clean": acc, "hidden_dep": hd, "attn_dep": 0.0, "context_share": 0.0}
    ad = d(accuracy(m, task, load, {"context": (mode, 1.0)}, n_batches, cfg))
    m.eval(); sh, sc = [], []
    for _ in range(4):
        p = m(task.generate(cfg.batch_size, load)["x"])["parts"]
        sh.append(p["hidden"].std(-1)); sc.append(p["context"].std(-1))
    m.train()
    h, c = torch.cat(sh).mean().item(), torch.cat(sc).mean().item()
    return {"acc_clean": acc, "hidden_dep": hd, "attn_dep": ad,
            "context_share": c / (h + c + 1e-9)}


def anneal(task, load, seed=0, stages=(0.05, 0.1, 0.2, 0.3, 0.5, 0.8),
           target=0.15, steps=2500, name="gru_attn", d_h=None, cfg=None):
    """Learn with retrieval available, then price it out. Training with the route
    masked from the start finds much worse recurrent solutions."""
    m, _ = train(task, load, lam=0.0, seed=seed, steps=steps, name=name, d_h=d_h, cfg=cfg)
    used = None
    for lam in stages:
        m, _ = train(task, load, lam=lam, seed=seed, steps=steps // 2, model=m, cfg=cfg)
        used = lam
        if deps(m, task, load, cfg, n_batches=4)["context_share"] < target:
            break
    return m, used


def crossing_any(x, hidden, attn):
    x = np.asarray(x, float)
    d = np.asarray(attn, float) - np.asarray(hidden, float)
    ok = ~np.isnan(d); x, d = x[ok], d[ok]
    o = np.argsort(x); x, d = x[o], d[o]
    for i in range(len(d) - 1):
        a, b = d[i], d[i + 1]
        if a == 0:
            return float(x[i])
        if a * b < 0:
            f = abs(a) / (abs(a) + abs(b))
            return float(x[i] + f * (x[i + 1] - x[i]))
    return np.nan


def boundary_fit(cross):
    pts = [(math.log2(k), v) for k, v in cross.items()
           if k > 0 and v is not None and not np.isnan(v)]
    if len(pts) < 3:
        return (np.nan, np.nan, np.nan)
    x = np.array([p[0] for p in pts]); y = np.array([p[1] for p in pts])
    b = np.cov(x, y, bias=True)[0, 1] / x.var(); a = y.mean() - b * x.mean()
    ss_t = ((y - y.mean()) ** 2).sum()
    return (a, b, 1 - ((y - (a + b * x)) ** 2).sum() / max(ss_t, 1e-12))


def crossings_by_seed(df, axis="load", keys=None, acc_floor=0.0):
    """Per-seed crossings. Averaging dependencies first and interpolating once
    hides exactly the variance that needs reporting."""
    if acc_floor > 0 and "acc_clean" in df.columns:
        df = df[df.acc_clean >= acc_floor]
    keys = [k for k in (keys or ["lam", "task", "model", "d_h", "d_c", "p_drop", "form"])
            if k in df.columns and k != axis and df[k].nunique() > 1]
    out = []
    for k, g in (df.groupby(keys + ["seed"]) if keys else df.groupby("seed")):
        gg = g.sort_values(axis)
        rec = dict(zip(keys + ["seed"], k if isinstance(k, tuple) else (k,)))
        rec["crossing"] = crossing_any(gg[axis], gg.hidden_dep, gg.attn_dep)
        out.append(rec)
    return pd.DataFrame(out)


def boot_ci(vals, n=10000, alpha=0.05, seed=0):
    v = np.asarray([x for x in vals if x == x], float)
    if len(v) < 2:
        return (float(v.mean()) if len(v) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    mu = rng.choice(v, size=(n, len(v)), replace=True).mean(1)
    return float(v.mean()), float(np.quantile(mu, alpha/2)), float(np.quantile(mu, 1-alpha/2))


def report_crossings(df, axis="load", label="", keys=None, acc_floor=0.0):
    C = crossings_by_seed(df, axis, keys, acc_floor)
    if not len(C):
        print(f"  {label}: no crossings"); return C
    gk = [c for c in C.columns if c not in ("seed", "crossing")]
    print(f"\n{label}  (axis {axis}" + (f", acc floor {acc_floor}" if acc_floor else "") + ")")
    for k, g in (C.groupby(gk) if gk else [((), C)]):
        mu, lo, hi = boot_ci(g.crossing.values)
        lbl = ", ".join(f"{a}={b}" for a, b in zip(gk, k if isinstance(k, tuple) else (k,))) or "all"
        n_ok = int(g.crossing.notna().sum())
        flag = "  <- off-grid for some seeds" if n_ok < len(g) else ""
        print(f"  {lbl:<34} {mu:7.3f}  95% CI [{lo:.3f}, {hi:.3f}]  n={n_ok}/{len(g)}{flag}")
    if gk and "lam" in gk and axis == "load":
        a, b, r2 = boundary_fit(C.dropna().groupby("lam").crossing.mean().to_dict())
        print(f"  fit: load* = {a:.2f} + {b:.2f}·log2(lambda),  R^2 = {r2:.3f}")
    return C


def sweep(name, conditions, seeds, steps, schedule="cosine", out=None,
          model_fn=None, cfg=None, log_every=None, ckpt=False):
    """Resumable factorial sweep. Every (condition, seed) is hashed and skipped
    if already on disk, so a killed session costs at most one run."""
    st = Store(name)
    units = list(itertools.product(conditions, seeds))
    todo = [(c, sd) for c, sd in units
            if not st.has({**c, "seed": sd, "steps": steps, "schedule": schedule,
                           "block": name, "cfg": cfg_tag(cfg)})]
    print(f"  [{name}] {len(units)} units, {len(units)-len(todo)} cached, {len(todo)} to run")
    t0, done = time.time(), 0
    try:
        for c, sd in todo:
            spec = {**c, "seed": sd, "steps": steps, "schedule": schedule,
                    "block": name, "cfg": cfg_tag(cfg)}
            t1 = time.time()
            tk = make_task(c.get("task", "selective_recall"), cfg)
            m, tr = train(tk, c["load"], lam=c.get("lam", 0.0), form=c.get("form", "l2"),
                          seed=sd, steps=steps, model=model_fn(c) if model_fn else None,
                          name=c.get("model", "gru_attn"), d_h=c.get("d_h"),
                          d_c=c.get("d_c"), gate=c.get("gate", False),
                          schedule=schedule, p_drop=c.get("p_drop", 0.0),
                          log_every=log_every, cfg=cfg,
                          ckpt_key=(keyof(spec) if ckpt else None))
            payload = dict(deps(m, tk, c["load"], cfg))
            if len(tr):
                payload["trace"] = tr.to_dict("records")
            payload["_elapsed"] = time.time() - t1
            st.put(spec, payload)
            done += 1
            if done % 15 == 0 or done == len(todo):
                el = (time.time() - t0) / 60
                print(f"  [{name}] {done}/{len(todo)}  {el:.1f}m, ~{el/done*(len(todo)-done):.1f}m left")
    except KeyboardInterrupt:
        print(f"  [{name}] interrupted at {done}/{len(todo)} — finished runs are saved, "
              f"rerun this cell to continue")
    df = st.frame()
    if out and len(df):
        df.to_csv(out, index=False)
        tf = st.traces()
        if len(tf):
            tf.to_csv(out.replace(".csv", "_traces.csv"), index=False)
    return df


if RUN["sanity"]:
    for nm in TASKS:
        print(f"\n{nm} (load 4, n_out={make_task(nm).n_out})")
        make_task(nm).show(4, 2)
    CAP = pd.DataFrame(CAP_TABLE)
    CAP.to_csv(csv("p0_capacity"), index=False)
    print("\ncapacity matching (S3.5):")
    print(CAP.to_string(index=False))


## §4.1 Free retrieval dominates  ·  §3.4 Positive control

Two questions. With no cost, does the hybrid ever use recurrence? And is
recurrence even *sufficient* — does a genuinely attention-free model of matched
capacity solve the task? The claim needs both: `hidden_dep` at zero for the
hybrid, and the plain recurrent model at ceiling.

The positive control gates everything else. If the measure cannot report
recurrence on a task where retrieval provably fails, no zero elsewhere means
anything.

In [ ]:
if RUN["free"]:
    conds = [{"task": t, "model": mm, "lam": 0.0, "load": L}
             for t in ["selective_recall", "delayed_recall"]
             for mm in ["gru_attn", "gru"] for L in [2, 4, 8, 12]]
    FREE = sweep("free", conds, SEEDS_AUX, STEPS, out=csv("p1_free"))
    print("\nhidden_dep, hybrid at λ=0 (rows task, cols load):")
    print(FREE[FREE.model == "gru_attn"].pivot_table(
        index="task", columns="load", values="hidden_dep").round(3).to_string())
    print("\naccuracy of the attention-free model — is recurrence sufficient?")
    print(FREE[FREE.model == "gru"].pivot_table(
        index="task", columns="load", values="acc_clean").round(3).to_string())

    conds = [{"task": "affine_track", "model": "gru_attn", "lam": 0.0, "load": L}
             for L in [2, 4, 6]]
    CTRL = sweep("control", conds, SEEDS_AUX, STEPS, out=csv("p1_control"))
    print("\npositive control (affine_track):")
    print(CTRL.pivot_table(index="load",
                           values=["acc_clean", "hidden_dep", "attn_dep"]).round(3).to_string())
    print("\n  hidden_dep well above zero here means the measure CAN report")
    print("  recurrence; the zeros above are therefore informative.")

## §4.1 addendum: sufficiency at a budget recurrence can reach

The attention-free model stalls at 0.900 (load 8) and 0.764 (load 12) at the
default budget. That is the recurrent solution being slow to find, not a
capacity limit — and the premise of §4.1 depends on the difference.


In [ ]:
if RUN["free"]:
    # S4.1's premise is that both algorithms are individually sufficient. At the
    # default budget the attention-free model reaches 1.000 at loads 2 and 4 but
    # only 0.900 at load 8 and 0.764 at load 12 on selective_recall, which would
    # make the hybrid's use of retrieval a requirement rather than a choice at
    # the loads the boundary sweep cares about.
    #
    # The recurrent solution is the slow one to find — S6.2 measures how slow —
    # so this rerun gives it three times the budget. Report the premise at
    # whichever budget it holds, and report the asymmetry: retrieval is found in
    # a few hundred steps, recurrence needs thousands. That asymmetry is S6.2's
    # finding appearing inside S4.1, not a weakness of either.
    conds = [{"task": t, "model": m, "load": L}
             for t in ["selective_recall", "delayed_recall"]
             for m in ["gru", "lstm"] for L in [2, 4, 8, 12]]
    FL = sweep("free_long", conds, SEEDS_AUX, 3 * STEPS, out=csv("p1_free_long"))
    print(f"\naccuracy of the attention-free models at {3 * STEPS} steps:")
    print(FL.pivot_table(index=["task", "model"], columns="load",
                         values="acc_clean").round(3).to_string())
    _lo = FL[FL.task == "selective_recall"].groupby("load").acc_clean.mean().min()
    print(f"\n  lowest selective_recall cell: {_lo:.3f}")
    if _lo >= 0.99:
        print("  The premise holds at this budget. State it with the budget, and")
        print("  state the asymmetry against retrieval's few hundred steps.")
    else:
        print("  Still short. The premise is load-bounded: say that both")
        print("  algorithms are sufficient up to the load where this holds, and")
        print("  flag the boundary points above it as measuring something else.")


## §4.2–4.3 The boundary, and its invisibility to accuracy

The primary result. The accuracy table is not a formality — the entire claim is
that the network crosses this boundary without any change in behaviour.

In [ ]:
if RUN["boundary"]:
    conds = [{"lam": l, "load": L}
             for l in [0.0, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.35]
             for L in [2, 4, 6, 8, 12, 16, 20]]
    BND = sweep("boundary", conds, SEEDS_MAIN, STEPS, out=csv("p2_boundary"))
    print("\ncontext_share (rows λ, cols load):")
    print(BND.pivot_table(index="lam", columns="load", values="context_share").round(3).to_string())
    print("\naccuracy:")
    print(BND.pivot_table(index="lam", columns="load", values="acc_clean").round(3).to_string())
    print(f"\n  minimum accuracy over {len(BND)} runs: {BND.acc_clean.min():.4f}")
    report_crossings(BND, "load", "§4.2 boundary", acc_floor=ACC_FLOOR)

## §4.4 Four cost formulations, including one with no penalty term

1. `λ‖lc‖` — non-squared, group-lasso geometry
2. `λ‖lc‖²` — smooth, no sparsity threshold
3. a **mixture gate**: both routes normalised, `z` sets the mixture, penalty
   `λ·z`. Earlier gate designs were void because normalising one route let the
   model rescale the other; normalising both closes that.
4. **availability**: retrieval simply absent on a fraction *p* of steps, with no
   penalty term in the objective at all.

The non-squared penalty degrades accuracy at high λ and high load, so an
accuracy floor is applied before any crossing is read.

In [ ]:
if RUN["costforms"]:
    # Gate records written before the optimiser fix are invalid: z was frozen,
    # so those runs measured an untrained gate. They are dropped once here, and
    # the surviving gate conditions carry `gate_fix` so the retired records can
    # never be matched again. l2 and l2sq records are untouched and still cache.
    _p = RESDIR / f"costforms__{TAG}.jsonl"
    if _p.exists():
        _lines = [l for l in _p.read_text().splitlines() if l.strip()]
        _keep = []
        for _l in _lines:
            try:
                _s = json.loads(_l)["_spec"]
            except (json.JSONDecodeError, KeyError):
                continue
            if _s.get("form") == "gate" and _s.get("gate_fix") != 1:
                continue
            _keep.append(_l)
        if len(_keep) != len(_lines):
            _p.write_text("\n".join(_keep) + "\n")
            print(f"  retired {len(_lines) - len(_keep)} pre-fix gate record(s); "
                  f"{len(_keep)} kept")

    LOADS = [2, 4, 8, 12, 16, 24]
    GRIDS = {"l2": [0.0, 0.01, 0.05, 0.1, 0.2, 0.35],
             "l2sq": [0.0, 0.0005, 0.002, 0.01, 0.05],
             "gate": [0.0, 0.1, 0.3, 1.0, 3.0, 6.0]}
    conds = [dict({"form": f, "lam": l, "load": L, "gate": f == "gate"},
                  **({"gate_fix": 1} if f == "gate" else {}))
             for f, g in GRIDS.items() for l in g for L in LOADS]
    CF = sweep("costforms", conds, SEEDS_AUX, STEPS, out=csv("p3_costforms"))
    for f in GRIDS:
        g = CF[CF.form == f]
        nz = g[g.context_share > 0].context_share
        print(f"\n--- {f} ---   min acc {g.acc_clean.min():.3f}"
              f"   min nonzero share {nz.min():.2e}" if len(nz) else "")
        print(g.pivot_table(index="lam", columns="load", values="context_share").round(3).to_string())
        below = g[g.acc_clean < ACC_FLOOR]
        if len(below):
            print(f"  {len(below)}/{len(g)} cells below {ACC_FLOOR} accuracy — excluded")
        report_crossings(g, "load", f"§4.4 {f}", acc_floor=ACC_FLOOR)

    conds = [{"p_drop": p, "load": L}
             for p in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8] for L in LOADS]
    AV = sweep("availability", conds, SEEDS_AUX, STEPS, out=csv("p3_availability"))
    print("\ncontext_share — availability (rows p, cols load):")
    print(AV.pivot_table(index="p_drop", columns="load", values="context_share").round(3).to_string())
    report_crossings(AV, "load", "§4.4 availability (no penalty term)",
                 keys=["p_drop"], acc_floor=ACC_FLOOR)

## §4.5 Storage capacity, not total width

`d_h` and `d_c` varied independently at fixed load. This contains a **null** on
the retrieval axis, so it gets five seeds — nulls need power. The crossing here
is in λ, which is why direction-agnostic detection matters.

In [ ]:
if RUN["width"]:
    CONFIGS = [(32, 52), (52, 52), (96, 52), (160, 52), (52, 32), (52, 96), (52, 160)]
    conds = [{"d_h": dh, "d_c": dc, "lam": l, "load": 8}
             for dh, dc in CONFIGS for l in [0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.35]]
    WID = sweep("width", conds, SEEDS_MAIN, STEPS, out=csv("p4_width"))
    WID["cfgstr"] = WID.d_h.astype(str) + "," + WID.d_c.astype(str)
    print("\ncontext_share at load 8 (rows d_h,d_c; cols λ):")
    print(WID.pivot_table(index="cfgstr", columns="lam", values="context_share").round(3).to_string())
    C = report_crossings(WID, "lam", "§4.5 width", keys=["d_h", "d_c"], acc_floor=ACC_FLOOR)
    C.to_csv(csv("p4_width_crossings"), index=False)
    sub = C.dropna()
    if len(sub):
        a = sub[sub.d_h == 52].groupby("d_c").crossing.mean()
        b = sub[sub.d_c == 52].groupby("d_h").crossing.mean()
        print(f"\n  fold-range varying retrieval width d_c: {a.max()/a.min():.2f}×")
        print(f"  fold-range varying storage   width d_h: {b.max()/b.min():.2f}×")
        print("  The dissociation is the ratio of these, with CIs overlapping on one")
        print("  axis and separated on the other.")

## §4.6 The algorithm predicts where the model fails

Matched pairs at equal in-distribution accuracy, differing only in route split.
The recurrence arm is built by annealing — no from-scratch price gives a
recurrent high-load model at ceiling.

Interference appends items beyond the trained load while keeping the query
inside the trained range, so a failure is interference and not an unseen
position. Pair validity is checked before any number is read.

In [ ]:
if RUN["ood"]:
    PAIR_LOADS = [4, 8, 12]

    # From-scratch recurrent controls.
    # load 12 is deliberately retried at lambda=0.20.
    SCRATCH_LAM = {4: 0.10, 8: 0.20, 12: 0.20}

    # load 12 needs the longer recurrent-learning budget.
    SCRATCH_MULT = {4: 1, 8: 1, 12: 3}

    ACC_VALID = 0.99

    st_p = Store("ood_pairs")
    pair_models = {}

    def _pair_key(L, arm, sd, mult=1, scratch_lam=None):
        mult_tag = f"_x{mult}" if mult != 1 else ""
        lam_tag = (
            f"_lam{scratch_lam:g}"
            if arm == "recurrent_scratch" and scratch_lam is not None
            else ""
        )
        return f"ood_{L}_{arm}_s{sd}{mult_tag}{lam_tag}"

    # ------------------------------------------------------------------
    # Fit / load paired models
    # ------------------------------------------------------------------
    for L, sd in itertools.product(PAIR_LOADS, SEEDS_MAIN):
        task = make_task("selective_recall")

        arms = ["retrieval", "recurrent"]
        if L in SCRATCH_LAM:
            arms.append("recurrent_scratch")

        for arm in arms:
            is_scratch = arm == "recurrent_scratch"

            mult = SCRATCH_MULT.get(L, 1) if is_scratch else 1
            scratch_lam = SCRATCH_LAM[L] if is_scratch else None
            steps_used = mult * STEPS if is_scratch else STEPS

            # IMPORTANT:
            # scratch_lam + scratch_mult are part of the cache identity.
            # Changing lambda from .30 -> .20 therefore creates a new run.
            spec = {
                "block": "ood_pairs",
                "load": L,
                "arm": arm,
                "seed": sd,
                "steps": steps_used,
            }

            if is_scratch:
                spec["scratch_lam"] = scratch_lam
                spec["scratch_mult"] = mult

            key = _pair_key(
                L,
                arm,
                sd,
                mult=mult,
                scratch_lam=scratch_lam,
            )

            mm = build("gru_attn", n_out=task.n_out).to(DEVICE)

            if st_p.has(spec) and load_model_into(key, mm):
                pair_models[(L, arm, sd)] = mm
                continue

            if st_p.has(spec):
                print(
                    f"  [ood_pairs] load {L} {arm} seed {sd}: "
                    f"weights missing, refitting"
                )
                st_p.recs.pop(keyof(spec), None)

            def _fit(
                L=L,
                arm=arm,
                sd=sd,
                task=task,
                mult=mult,
                scratch_lam=scratch_lam,
                key=key,
            ):
                if arm == "retrieval":
                    m, _ = train(
                        task,
                        L,
                        lam=0.0,
                        seed=sd,
                        steps=STEPS,
                    )
                    lam_u = 0.0
                    actual_steps = STEPS

                elif arm == "recurrent":
                    m, lam_u = anneal(
                        task,
                        L,
                        seed=sd,
                        steps=STEPS,
                    )
                    actual_steps = STEPS

                else:
                    m, _ = train(
                        task,
                        L,
                        lam=scratch_lam,
                        seed=sd,
                        steps=mult * STEPS,
                    )
                    lam_u = scratch_lam
                    actual_steps = mult * STEPS

                save_model(key, m)
                pair_models[(L, arm, sd)] = m

                return {
                    "build_lam": lam_u,
                    "steps_used": actual_steps,
                    "scratch_lam": scratch_lam if arm == "recurrent_scratch" else np.nan,
                    "scratch_mult": mult if arm == "recurrent_scratch" else np.nan,
                    **deps(m, task, L),
                }

            cached(
                st_p,
                spec,
                _fit,
                label=(
                    f"load {L} {arm} seed {sd}"
                    + (
                        f" ({mult}x steps, lambda={scratch_lam:g})"
                        if is_scratch
                        else ""
                    )
                ),
            )

    # ------------------------------------------------------------------
    # Pair validity
    #
    # Store may contain historical scratch runs, e.g. load12 lambda=.30.
    # Keep only the CURRENT requested scratch configuration.
    # ------------------------------------------------------------------
    P_all = st_p.frame()

    keep = pd.Series(True, index=P_all.index)

    if "arm" in P_all.columns:
        scratch_rows = P_all.arm == "recurrent_scratch"

        # Start by excluding every historical scratch row.
        keep.loc[scratch_rows] = False

        # Add back only the currently requested scratch configs.
        for L in PAIR_LOADS:
            if L not in SCRATCH_LAM:
                continue

            wanted_lam = SCRATCH_LAM[L]
            wanted_mult = SCRATCH_MULT.get(L, 1)

            mask = (
                scratch_rows
                & (P_all["load"] == L)
            )

            if "scratch_lam" in P_all.columns:
                mask &= np.isclose(
                    pd.to_numeric(
                        P_all["scratch_lam"],
                        errors="coerce",
                    ),
                    wanted_lam,
                    equal_nan=False,
                )
            else:
                mask &= False

            if "scratch_mult" in P_all.columns:
                mask &= (
                    pd.to_numeric(
                        P_all["scratch_mult"],
                        errors="coerce",
                    )
                    == wanted_mult
                )
            else:
                mask &= False

            keep.loc[mask] = True

    P = P_all.loc[keep].copy()
    P.to_csv(csv("p5_pairs"), index=False)

    print("\npair validity:")
    print(
        P.groupby(["load", "arm"])[
            ["acc_clean", "context_share", "hidden_dep"]
        ]
        .agg(["mean", "std"])
        .round(3)
        .to_string()
    )

    # ------------------------------------------------------------------
    # Explicit validity gate
    # ------------------------------------------------------------------
    VALID = P.groupby(["load", "arm"]).acc_clean.mean()

    INVALID = {
        k
        for k, v in VALID.items()
        if v < ACC_VALID
    }

    if INVALID:
        print(
            f"\n  !! below {ACC_VALID} clean accuracy, excluded from "
            f"the dissociation and the construction control:"
        )

        for L, arm in sorted(INVALID):
            print(
                f"     load {L:>2} {arm:<18} "
                f"accuracy {VALID[(L, arm)]:.3f}"
            )

        print(
            "     A priced model that never found the recurrent solution is a"
        )
        print(
            "     failed control, not evidence about recurrence."
        )

    # ------------------------------------------------------------------
    # Verify model checkpoints exist
    # ------------------------------------------------------------------
    missing = [
        k
        for k in itertools.product(
            PAIR_LOADS,
            ["retrieval", "recurrent"],
            SEEDS_MAIN,
        )
        if k not in pair_models
    ]

    if missing:
        print(
            f"\n  !! {len(missing)} pair model(s) could not be loaded "
            f"from checkpoints"
        )

    task = make_task("selective_recall")

    # ------------------------------------------------------------------
    # OOD helpers
    # ------------------------------------------------------------------
    @torch.no_grad()
    def acc_extra(m, load, extra, n=4):
        m.eval()
        c = 0
        t = 0

        for _ in range(n):
            b = task.generate(
                CFG.batch_size,
                load,
                extra_items=extra,
            )
            c += (
                m(b["x"])["logits"].argmax(-1) == b["y"]
            ).sum().item()
            t += b["y"].numel()

        m.train()
        return c / t

    @torch.no_grad()
    def acc_noise(m, load, sigma, n=4):
        m.eval()

        h = m.emb.register_forward_hook(
            lambda mod, i, o:
            o + sigma * o.std() * torch.randn_like(o)
        )

        try:
            c = 0
            t = 0

            for _ in range(n):
                b = task.generate(
                    CFG.batch_size,
                    load,
                )

                c += (
                    m(b["x"])["logits"].argmax(-1) == b["y"]
                ).sum().item()

                t += b["y"].numel()

            return c / t

        finally:
            h.remove()
            m.train()

    # ------------------------------------------------------------------
    # OOD evaluation
    # ------------------------------------------------------------------
    st_o = Store("ood_eval")

    for (L, arm, sd), m in pair_models.items():
        is_scratch = arm == "recurrent_scratch"

        mult = SCRATCH_MULT.get(L, 1) if is_scratch else 1
        scratch_lam = SCRATCH_LAM[L] if is_scratch else None
        steps_used = mult * STEPS if is_scratch else STEPS

        # Again, lambda is part of the evaluation cache key.
        spec = {
            "block": "ood_eval",
            "load": L,
            "arm": arm,
            "seed": sd,
            "steps": steps_used,
        }

        if is_scratch:
            spec["scratch_lam"] = scratch_lam
            spec["scratch_mult"] = mult

        def _eval(
            L=L,
            arm=arm,
            sd=sd,
            m=m,
        ):
            rows = []

            for d in [0, 10, 30, 70]:
                rows.append({
                    "axis": "delay",
                    "level": d,
                    "acc": accuracy(
                        m,
                        task,
                        L,
                        n_batches=4,
                        extra_delay=d,
                    ),
                })

            for e in [0, 2, 4, 8]:
                if L + e <= CFG.max_query_pos:
                    rows.append({
                        "axis": "interference",
                        "level": e,
                        "acc": acc_extra(
                            m,
                            L,
                            e,
                        ),
                    })

            for s in [0.0, 0.25, 0.5, 1.0]:
                rows.append({
                    "axis": "noise",
                    "level": s,
                    "acc": acc_noise(
                        m,
                        L,
                        s,
                    ),
                })

            return {"rows": rows}

        cached(
            st_o,
            spec,
            _eval,
            label=(
                f"OOD load {L} {arm} seed {sd}"
                + (
                    f" lambda={scratch_lam:g}"
                    if is_scratch
                    else ""
                )
            ),
        )

    # ------------------------------------------------------------------
    # Filter historical OOD scratch evaluations too
    # ------------------------------------------------------------------
    O_all = st_o.frame()

    keep = pd.Series(True, index=O_all.index)

    if "arm" in O_all.columns:
        scratch_rows = O_all.arm == "recurrent_scratch"
        keep.loc[scratch_rows] = False

        for L in PAIR_LOADS:
            if L not in SCRATCH_LAM:
                continue

            wanted_lam = SCRATCH_LAM[L]
            wanted_mult = SCRATCH_MULT.get(L, 1)

            mask = (
                scratch_rows
                & (O_all["load"] == L)
            )

            if "scratch_lam" in O_all.columns:
                mask &= np.isclose(
                    pd.to_numeric(
                        O_all["scratch_lam"],
                        errors="coerce",
                    ),
                    wanted_lam,
                    equal_nan=False,
                )
            else:
                mask &= False

            if "scratch_mult" in O_all.columns:
                mask &= (
                    pd.to_numeric(
                        O_all["scratch_mult"],
                        errors="coerce",
                    )
                    == wanted_mult
                )
            else:
                mask &= False

            keep.loc[mask] = True

    O = O_all.loc[keep].copy()
    O.to_csv(csv("p5_ood"), index=False)

    # ------------------------------------------------------------------
    # Dissociation summary
    # ------------------------------------------------------------------
    out = []

    for L, ax in itertools.product(
        PAIR_LOADS,
        ["delay", "interference", "noise"],
    ):
        g = O[
            (O.load == L)
            & (O.axis == ax)
        ]

        if not len(g):
            continue

        hard = g.level.max()

        r = g[
            (g.arm == "retrieval")
            & (g.level == hard)
        ].acc

        h = g[
            (g.arm == "recurrent")
            & (g.level == hard)
        ].acc

        rec = {
            "load": L,
            "axis": ax,
            "retrieval": r.mean(),
            "retrieval_sd": r.std(),
            "recurrent": h.mean(),
            "recurrent_sd": h.std(),
            "advantage": r.mean() - h.mean(),
            "separated": (
                abs(r.mean() - h.mean())
                > (r.std() + h.std())
            ),
        }

        s = g[
            (g.arm == "recurrent_scratch")
            & (g.level == hard)
        ].acc

        scratch_valid = (
            (L, "recurrent_scratch") not in INVALID
            and (L, "recurrent_scratch") in VALID.index
        )

        rec["recurrent_scratch"] = (
            s.mean()
            if len(s) and scratch_valid
            else np.nan
        )

        rec["scratch_sd"] = (
            s.std()
            if len(s) and scratch_valid
            else np.nan
        )

        out.append(rec)

    S5 = pd.DataFrame(out)
    S5.to_csv(
        csv("p5_dissociation"),
        index=False,
    )

    print(
        "\ndissociation at the hardest level of each axis:"
    )
    print(
        S5.round(3).to_string(index=False)
    )

    print(
        "\n  Opposite signs across OOD manipulations, with matched "
        "in-distribution performance, support the double dissociation."
    )

    # ------------------------------------------------------------------
    # §4.6 construction control
    # ------------------------------------------------------------------
    CC = S5.dropna(
        subset=["recurrent_scratch"]
    )

    if len(CC):
        print(
            "\nconstruction control "
            "(annealed vs from-scratch recurrent arm):"
        )

        print(
            CC[
                [
                    "load",
                    "axis",
                    "retrieval",
                    "recurrent",
                    "recurrent_scratch",
                ]
            ]
            .round(3)
            .to_string(index=False)
        )

        CC.to_csv(
            csv("p5_construction_control"),
            index=False,
        )

        gap = (
            CC.recurrent
            - CC.recurrent_scratch
        ).abs()

        print(
            f"  largest annealed-vs-scratch gap: "
            f"{gap.max():.3f} "
            f"({CC.loc[gap.idxmax(), 'axis']}, "
            f"load {int(CC.loc[gap.idxmax(), 'load'])})"
        )

        print(
            "  Where annealed and from-scratch agree, the effect is not "
            "an artefact of annealing; where they do not, the row is not claimed."
        )

    _covered = (
        sorted({
            int(L)
            for L in CC.load.unique()
        })
        if len(CC)
        else []
    )

    print(
        f"\n  construction control covers loads {_covered}; "
        f"any load absent from that list has no from-scratch arm "
        f"that solved the task.")

## §4.7 Replication: tasks, architectures, recurrent cores, budget

`delayed_recall` and `distractor` abandon retrieval at much lower prices, so
they need their own λ grid. Loads run to 24 — `lstm_attn` boundaries ran off a
grid stopping at 12 in an earlier attempt.

In [ ]:
# =============================================================================
# §4.7  Generality — tasks, attention architectures, recurrent cores, schedule
#
# Two grid problems surfaced as apparent findings and both are now handled.
#
# 1. Crossings pinned at the top of the load grid (n=1/3 seeds, all reporting
#    the same value) are ceilings, not measurements. Excluded from every fit.
#
# 2. At low lambda the cores arms put four consecutive lambdas inside the single
#    interval [2, 4]. `crossing_any` interpolates linearly within an interval, so
#    a 4x price change was being compressed into one segment, producing an
#    apparent slope of ~0.38 and a spurious "floor". Loads 3 and 6 are added to
#    resolve it. If the low-lambda slope rises toward the high-lambda slope once
#    the interval is split, the curvature was resolution; if it does not, the
#    boundary genuinely flattens and that is a finding.
#
# Every arm is reported four ways: full-range fit, lower and upper half fits,
# and a fit over a FIXED lambda range common to all arms. The last is the only
# one whose coefficients are comparable across arms, because it does not let
# each arm choose its own window.
# =============================================================================

FIT_LAM_MIN = 0.005      # common comparison range, matching §4.2's grid
ALL_FITS = []


def boundary_report(df, label, axis="load", keys=("lam",), acc_floor=None,
                    min_seeds=None, curvature_tol=0.35, fit_lam_min=FIT_LAM_MIN):
    """Crossings, then fits: full range, split halves, and a fixed lambda range.

    Points where fewer than `min_seeds` seeds produced a crossing are excluded —
    a crossing pinned at the edge of the load grid is a ceiling, not a
    measurement. `curvature_tol` is the fractional slope difference between
    halves above which the log-linear model is reported as not holding.
    """
    acc_floor = ACC_FLOOR if acc_floor is None else acc_floor
    min_seeds = (min_seeds if min_seeds is not None
                 else max(2, int(np.ceil(0.6 * df.seed.nunique()))))
    C = crossings_by_seed(df, axis, list(keys), acc_floor=acc_floor)
    if not len(C) or "lam" not in C:
        print(f"\n{label}: no crossings"); return None

    tab = (C.groupby("lam").crossing
            .agg(crossing="mean", sd="std", n_ok="count", n="size").reset_index())
    tab["usable"] = (tab.n_ok >= min_seeds) & tab.crossing.notna() & (tab.lam > 0)

    loads = sorted(df[axis].unique())
    print(f"\n{label}  (axis {axis}, acc floor {acc_floor}, "
          f"min {min_seeds}/{df.seed.nunique()} seeds)")
    for _, r in tab.iterrows():
        flag = "" if r.usable else ("  <- excluded, thin" if r.lam > 0 else "")
        mu = f"{r.crossing:7.3f}" if r.crossing == r.crossing else "    nan"
        sd = f" ± {r.sd:.3f}" if r.sd == r.sd else ""
        # name the grid interval the crossing falls in: several lambdas sharing
        # one interval is the signature of a resolution artefact
        seg = ""
        if r.crossing == r.crossing:
            below = [L for L in loads if L <= r.crossing]
            above = [L for L in loads if L > r.crossing]
            if below and above:
                seg = f"   in [{below[-1]}, {above[0]}]"
        print(f"  lam={r.lam:<9g} {mu}{sd}   n={int(r.n_ok)}/{int(r.n)}{flag}{seg}")

    use = tab[tab.usable].sort_values("lam")
    if len(use) >= 2:
        occ = {}
        for _, r in use.iterrows():
            below = [L for L in loads if L <= r.crossing]
            above = [L for L in loads if L > r.crossing]
            if below and above:
                occ.setdefault((below[-1], above[0]), []).append(r.lam)
        crowded = {k: v for k, v in occ.items() if len(v) >= 3}
        if crowded:
            print("  !! grid resolution: one interval holds three or more lambdas —")
            for k, v in crowded.items():
                print(f"     [{k[0]}, {k[1]}] holds lam {', '.join(f'{x:g}' for x in v)}")
            print("     Slopes fitted through such a region measure the")
            print("     interpolation, not the boundary. Add load points there.")

    if len(use) < 3:
        print("  too few usable points to fit")
        return tab

    def _fit(sub):
        return (boundary_fit(dict(zip(sub.lam, sub.crossing))) if len(sub) >= 3
                else (np.nan, np.nan, np.nan))

    a_f, b_f, r_f = _fit(use)
    mid = len(use) // 2 + (len(use) % 2)
    lo, hi = use.iloc[:mid], use.iloc[mid - 1:]      # halves overlap by one point
    a_l, b_l, _ = _fit(lo)
    a_h, b_h, _ = _fit(hi)
    common = use[use.lam >= fit_lam_min]
    a_c, b_c, r_c = _fit(common)

    print(f"  full range   load* = {a_f:6.2f} + {b_f:5.2f}·log2(lam)   "
          f"R^2 = {r_f:.3f}   lam {use.lam.min():g}–{use.lam.max():g}")
    bent, rel = False, np.nan
    if b_l == b_l and b_h == b_h:
        rel = abs(b_h - b_l) / max(abs(b_h), abs(b_l), 1e-9)
        print(f"  lower half   slope {b_l:5.2f}  (lam {lo.lam.min():g}–{lo.lam.max():g})")
        print(f"  upper half   slope {b_h:5.2f}  (lam {hi.lam.min():g}–{hi.lam.max():g})")
        bent = rel > curvature_tol
        print(f"  halves differ by {rel:.0%} — "
              + ("NOT log-linear over this range" if bent
                 else "consistent with log-linear across the range"))
    if b_c == b_c:
        print(f"  lam>={fit_lam_min:g}   load* = {a_c:6.2f} + {b_c:5.2f}·log2(lam)   "
              f"R^2 = {r_c:.3f}   <- the cross-arm comparable fit")
    else:
        print(f"  lam>={fit_lam_min:g}   too few points for the common-range fit")

    slopes = []
    for _, g in C[C.lam.isin(common.lam)].dropna().groupby("seed"):
        _, b_s, _ = boundary_fit(dict(zip(g.lam, g.crossing)))
        if b_s == b_s:
            slopes.append(b_s)
    mb, lb, hb = boot_ci(slopes) if len(slopes) > 1 else (np.nan,) * 3
    if len(slopes) > 1:
        print(f"  per-seed slope on the common range {mb:.2f}  "
              f"95% CI [{lb:.2f}, {hb:.2f}]  n={len(slopes)}")

    ALL_FITS.append({"arm": label, "n_points": len(use),
                     "lam_lo": use.lam.min(), "lam_hi": use.lam.max(),
                     "slope_full": b_f, "intercept_full": a_f, "r2_full": r_f,
                     "slope_lower_half": b_l, "slope_upper_half": b_h,
                     "half_gap": rel, "log_linear": not bent,
                     "slope_common": b_c, "intercept_common": a_c, "r2_common": r_c,
                     "seed_slope": mb, "seed_lo": lb, "seed_hi": hb,
                     "n_common": len(common)})
    return tab


if RUN["replication"]:
    conds = [{"task": t, "model": mm, "lam": l, "load": L}
             for t in ["delayed_recall", "distractor"]
             for mm in ["gru_attn", "lstm_attn"]
             for l in [0.0, 0.0002, 0.0005, 0.00075, 0.001, 0.0015, 0.002, 0.003, 0.005]
             for L in [2, 4, 8, 12, 16, 24]]
    REP = sweep("replication", conds, SEEDS_AUX, STEPS, out=csv("p6_replication"))
    for (t, mm), g in REP.groupby(["task", "model"]):
        boundary_report(g, f"§4.7 {t} · {mm}", fit_lam_min=0.0005)
    print("\n  These arms are swept at a much lower lambda range than §4.2, so their")
    print("  common-range fit uses lam>=0.0005. Compare them to each other, and")
    print("  compare §4.2 and the cores to each other; do not compare across the")
    print("  two groups on coefficients.")
    print("  LSTM crosses at a higher load than GRU at matched price on both")
    print("  tasks, which is §4.5's storage-capacity prediction arriving from a")
    print("  second, independent manipulation.")


if RUN["cores"]:
    # loads 3 and 6 added: at low lambda four consecutive prices were landing
    # inside the single interval [2, 4], so the reported slope there was an
    # artefact of interpolating across one wide segment
    conds = [{"model": mm, "lam": l, "load": L}
             for mm in ["linear_attn", "selective_attn"]
             for l in [0.0, 0.0002, 0.0005, 0.00075, 0.001, 0.002,
                       0.005, 0.01, 0.02, 0.03, 0.05]
             for L in [2, 3, 4, 6, 8, 12, 16]]
    COR = sweep("cores", conds, SEEDS_AUX, STEPS, out=csv("p6_cores"))
    for mm, g in COR.groupby("model"):
        print(f"\naccuracy — {mm}:")
        print(g.pivot_table(index="lam", columns="load", values="acc_clean").round(3).to_string())
        boundary_report(g, f"§4.7 {mm}")


if RUN["schedule"]:
    _sch_store = Store("schedule")
    for s, l, L, sd in itertools.product(["cosine", "constant"], [0.01, 0.02, 0.05, 0.1],
                                         [2, 4, 8, 12, 16], SEEDS_AUX):
        task = make_task("selective_recall")
        spec = {"block": "schedule", "sched": s, "lam": l, "load": L, "seed": sd}
        cached(_sch_store, spec,
               lambda s=s, l=l, L=L, sd=sd: deps(
                   train(task, L, lam=l, seed=sd, steps=3000, schedule=s)[0],
                   task, L))
    SCH = _sch_store.frame(); SCH.to_csv(csv("p7_schedule"), index=False)
    cr = {}
    for s, g in SCH.groupby("sched"):
        t_ = boundary_report(g, f"§4.7 schedule={s}", fit_lam_min=0.01)
        if t_ is not None:
            cr[s] = t_[t_.usable].set_index("lam").crossing
    if len(cr) == 2:
        w = pd.DataFrame(cr).dropna()
        if len(w):
            print(f"\n  crossing ratio constant/cosine on matched lambdas: "
                  f"{(w['constant'] / w['cosine']).mean():.2f}× over {len(w)} points")
            print("  This belongs in the limitations section: the boundary's")
            print("  coefficients depend on the schedule as well as the budget.")

    conds = [{"lam": l, "load": L} for l in [0.005, 0.01, 0.03, 0.05, 0.1]
             for L in [4, 8, 12, 16, 24]]
    EQ = sweep("equilibrium", conds, SEEDS_AUX, LONG, schedule="constant",
               out=csv("p7_equilibrium"), log_every=2000, ckpt=True)
    print("\nequilibrium context_share:")
    print(EQ.pivot_table(index="lam", columns="load", values="context_share").round(4).to_string())
    boundary_report(EQ, "§4.7 equilibrium (constant LR, long budget)")


# ---- one comparison across every arm, on one criterion ----------------------
if ALL_FITS:
    _b = csv("p2_boundary")
    if os.path.exists(_b):
        boundary_report(pd.read_csv(_b), "§4.2 boundary (selective_recall · gru_attn)")
    F = pd.DataFrame(ALL_FITS)
    F.to_csv(csv("p6_fit_summary"), index=False)
    print("\n" + "=" * 78)
    print("every arm, same criterion:")
    print(F[["arm", "n_points", "lam_lo", "lam_hi", "slope_full", "r2_full",
             "slope_lower_half", "slope_upper_half", "log_linear",
             "slope_common", "n_common"]].round(3).to_string(index=False))
    bent = F[~F.log_linear]
    if len(bent):
        print(f"\n  {len(bent)}/{len(F)} arms are not log-linear across their full swept")
        print("  range. Quote `slope_common`, state the range, and report the")
        print("  curvature rather than hiding it:")
        print(bent[["arm", "slope_lower_half", "slope_upper_half", "half_gap",
                    "slope_common"]].round(3).to_string(index=False))
    ok = F[F.log_linear]
    if len(ok):
        print(f"\n  {len(ok)}/{len(F)} arms are consistent with log-linear:")
        print(ok[["arm", "slope_full", "seed_slope", "seed_lo", "seed_hi"]]
              .round(3).to_string(index=False))
    print("\n  The claim §4.7 supports is that the boundary EXISTS and is")
    print("  log-linear over the range where the load grid resolves it, on three")
    print("  tasks, three recurrent cores and two attention architectures. The")
    print("  coefficients vary with task and with storage capacity; they are not")
    print("  invariant, and the paper should not say they are.")
    print("=" * 78)

## §5.1 The causal route inverts at fixed accuracy  **(headline)**

Constant learning rate so the drift is not frozen, five seeds, dependencies
logged throughout. The high-load condition is the **boundary control**: above
the boundary there should be no inversion at the same ceiling accuracy, which
is what makes this a dissociation rather than generic drift.

In [ ]:
if RUN["inversion"]:
    conds = [{"lam": 0.01, "load": 8}, {"lam": 0.03, "load": 8},
             {"lam": 0.1, "load": 12}, {"lam": 0.01, "load": 16}]
    INV = sweep("inversion", conds, SEEDS_MAIN, LONG, schedule="constant",
                out=csv("p8_inversion"), log_every=1000, ckpt=True)
    T = pd.read_csv(csv("p8_inversion").replace(".csv", "_traces.csv"))

    # The claim is that route dependence inverts AFTER behaviour has converged,
    # so every measurement here starts from the converged step. At step 1000
    # accuracy is still at chance, the dependence denominator is the model's
    # headroom above chance, and dividing by ~0 produces both the huge seed SDs
    # of the first version and the spurious "inverted at step 1" entries.
    CONV = 0.99

    # Behaviour is called flat if it stays within EVAL_TOL of ceiling. `deps`
    # evaluates on 8 batches of 128, about 1000 trials, so a single logged point
    # at 0.9795 is roughly 20 errors out of 1000 — evaluation noise on a
    # converged model, not a behavioural change. A criterion of "never below
    # 0.99 at any of 15 logged points" rejects models whose accuracy is 1.0000
    # at both endpoints, which is the wrong test.
    EVAL_TOL = 0.97

    rows, conv_step = [], {}
    print(f"\nroute dependence from the first step at ≥{CONV} accuracy → last "
          f"(mean ± sd over seeds):\n")
    for (lam, L), g in T.groupby(["lam", "load"]):
        # convergence on the seed-averaged curve, so all seeds are compared at
        # the same step rather than each at its own
        curve = g.groupby("step").acc_clean.mean()
        hit = curve.index[curve >= CONV]
        if not len(hit):
            print(f"  λ={lam:<5} L={int(L):>2}  never reached {CONV} accuracy "
                  f"(peak {curve.max():.3f}) — excluded")
            continue
        s0, s1 = int(hit.min()), int(g.step.max())
        conv_step[(lam, L)] = s0
        a, b = g[g.step == s0], g[g.step == s1]
        win = g[g.step >= s0]
        inv = (a.attn_dep.mean() > a.hidden_dep.mean()) and \
              (b.hidden_dep.mean() > b.attn_dep.mean())
        rows.append({"lam": lam, "load": L, "converged_step": s0, "final_step": s1,
                     "window": s1 - s0,
                     "hidden_0": a.hidden_dep.mean(), "hidden_0_sd": a.hidden_dep.std(),
                     "hidden_1": b.hidden_dep.mean(), "hidden_1_sd": b.hidden_dep.std(),
                     "attn_0": a.attn_dep.mean(), "attn_0_sd": a.attn_dep.std(),
                     "attn_1": b.attn_dep.mean(), "attn_1_sd": b.attn_dep.std(),
                     "acc_0": a.acc_clean.mean(), "acc_1": b.acc_clean.mean(),
                     "acc_mean_window": win.acc_clean.mean(),
                     "acc_sd_window": win.acc_clean.std(),
                     "acc_min_window": win.acc_clean.min(),
                     "inverted": inv})
        print(f"  λ={lam:<5} L={int(L):>2}  converged at step {s0:>5}, "
              f"{s1 - s0:>5} further steps")
        print(f"           hidden {a.hidden_dep.mean():.3f}±{a.hidden_dep.std():.3f}"
              f" → {b.hidden_dep.mean():.3f}±{b.hidden_dep.std():.3f}   "
              f"attn {a.attn_dep.mean():.3f}±{a.attn_dep.std():.3f}"
              f" → {b.attn_dep.mean():.3f}±{b.attn_dep.std():.3f}")
        print(f"           accuracy {a.acc_clean.mean():.4f} → {b.acc_clean.mean():.4f}"
              f"   across the window {win.acc_clean.mean():.4f}±{win.acc_clean.std():.4f}"
              f" (min {win.acc_clean.min():.4f})"
              f"{'   INVERTS' if inv else ''}")

    if rows:
        IV = pd.DataFrame(rows)
        IV.to_csv(csv("p8_inversion_summary"), index=False)

        flat = IV.acc_min_window >= EVAL_TOL
        print(f"\n  behaviour flat across the post-convergence window "
              f"(min ≥ {EVAL_TOL}): {int(flat.sum())}/{len(IV)} conditions")
        print(f"  the tolerance accommodates evaluation noise: each accuracy is "
              f"~1000 trials, so ±0.02 is a handful of errors, not drift")
        if not flat.all():
            print("  !! dipped further than that after convergence in:")
            print(IV[~flat][["lam", "load", "acc_mean_window", "acc_min_window"]]
                  .round(4).to_string(index=False))

        # Crossing search restricted to the converged window, for the same
        # reason the endpoints are: before convergence both dependencies are
        # degenerate and any sign change there is an artefact, not an inversion.
        print("\n  step at which hidden_dep overtakes attn_dep, per seed "
              "(searched from the converged step onward):")
        cross_rows = []
        for (lam, L), g in T.groupby(["lam", "load"]):
            s0 = conv_step.get((lam, L))
            if s0 is None:
                continue
            steps = []
            for sd_, gs in g[g.step >= s0].groupby("seed"):
                gs = gs.sort_values("step")
                d = (gs.hidden_dep - gs.attn_dep).values
                idx = np.where(d > 0)[0]
                steps.append(int(gs.step.values[idx[0]]) if len(idx) else np.nan)
            ok = sorted(s for s in steps if s == s)
            cross_rows.append({"lam": lam, "load": L, "n_inverted": len(ok),
                               "n_seeds": len(steps),
                               "median_step": float(np.median(ok)) if ok else np.nan,
                               "min_step": ok[0] if ok else np.nan,
                               "max_step": ok[-1] if ok else np.nan})
            if ok:
                print(f"    λ={lam:<5} L={int(L):>2}  {ok}   "
                      f"({len(ok)}/{len(steps)} seeds, median {np.median(ok):.0f})")
            else:
                print(f"    λ={lam:<5} L={int(L):>2}  no seed inverted "
                      f"(0/{len(steps)})")
        CR = pd.DataFrame(cross_rows)
        CR.to_csv(csv("p8_inversion_crossings"), index=False)

        n_inv = int(IV.inverted.sum())
        print(f"\n  {n_inv}/{len(IV)} conditions inverted.")
        print("  The high-load condition at the same price is the control: it")
        print("  should not invert while the low-load one does, which is what")
        print("  makes the effect the boundary rather than elapsed training.")
        if len(CR) and CR.median_step.notna().any():
            print("\n  Lower price, later inversion, wider spread across seeds —")
            print("  the crossing step is itself a measurement, not just a yes/no:")
            print(CR[["lam", "load", "n_inverted", "n_seeds", "min_step",
                      "median_step", "max_step"]].to_string(index=False))

## §5.2 The cost is what drives the reorganisation

Four arms switching at step 2000, long after accuracy converges. `never` is the
arm that makes the experiment interpretable — with no cost there should be no
drift, and weight decay is 0 here, ruling out the obvious alternative.

In [ ]:
if RUN["switch"]:
    LAM, SW, ST = 0.03, 2000, S(12000, 4000, 400)
    ARMS = {"maintained": (LAM, LAM), "withdrawn": (LAM, 0.0),
            "never": (0.0, 0.0), "late": (0.0, LAM)}
    st = Store("switch")
    for (arm, (lb, la)), L, sd in itertools.product(ARMS.items(), [8, 12], SEEDS_MAIN):
        spec = {"block": "switch", "arm": arm, "load": L, "seed": sd,
                "lam": LAM, "switch": SW, "steps": ST}

        def _go(arm=arm, lb=lb, la=la, L=L, sd=sd, spec=spec):
            task = make_task("selective_recall")
            m, tr = train(task, L, lam=lb, lam_after=la, switch=SW, seed=sd, steps=ST,
                          schedule="constant", log_every=500, ckpt_key=keyof(spec))
            at = tr[tr.step <= SW].context_share.iloc[-1]
            return {"share_at_switch": float(at), **deps(m, task, L),
                    "trace": tr.to_dict("records")}

        cached(st, spec, _go, label=f"{arm:<11} load {L} seed {sd}")

    SWI = st.frame(); SWI.to_csv(csv("p9_switch"), index=False)
    TR = st.traces(); TR.to_csv(csv("p9_switch_traces"), index=False)
    print("\nretrieval share, at the switch -> at the end (mean over seeds):")
    piv = SWI.pivot_table(index="arm", columns="load",
                          values=["share_at_switch", "context_share"]).round(3)
    print(piv.to_string())
    print("\naccuracy in every cell: "
          f"min {SWI.acc_clean.min():.3f}, max {SWI.acc_clean.max():.3f}")
    print("  never   -> no drift without a cost (necessary; weight decay is 0)")
    print("  late    -> full handover even when the shortcut is established (sufficient)")
    print("  withdrawn -> reverses, so the cost is required continuously")


In [ ]:
SWI = pd.read_csv(csv("p9_switch"))
print("end-of-run accuracy, worst cells first:")
print(SWI.nsmallest(8, "acc_clean")[["arm", "load", "seed", "acc_clean",
                                     "share_at_switch", "context_share"]]
      .round(4).to_string(index=False))
print("\nby arm and load:")
print(SWI.pivot_table(index="arm", columns="load", values="acc_clean",
                      aggfunc=["mean", "min"]).round(4).to_string())

In [ ]:
TR = pd.read_csv(csv("p9_switch_traces"))
print("\nminimum accuracy over the whole trajectory, by arm and load:")
print(TR.pivot_table(index="arm", columns="load", values="acc_clean",
                     aggfunc="min").round(4).to_string())
low = TR[TR.acc_clean < 0.99]
if len(low):
    print(f"\n{len(low)} logged points below 0.99, by arm and phase:")
    low = low.assign(phase=np.where(low.step <= 2000, "A (pre-switch)", "B (post)"))
    print(low.groupby(["arm", "load", "phase"]).acc_clean
          .agg(["count", "min"]).round(4).to_string())

## §5.3 The mechanism is rotation, not removal

`proj_c` is `Sequential(Linear, GELU)`, so pre- and post-activation can be
inspected separately. A collapse at the activation with a *growing*
pre-activation norm is saturation gating, not shrinkage — and nothing being
destroyed is why §5.4 recovers.

In [ ]:
if RUN["silencing"]:
    @torch.no_grad()
    def readout_path(m, task, load, n=4):
        m.eval(); acc = {k: [] for k in ["ctx", "pre", "post", "lc", "dead", "pre_mean"]}
        lin, act = m.proj_c[0], m.proj_c[1]
        for _ in range(n):
            b = task.generate(CFG.batch_size, load)
            o = m(b["x"], return_internals=True)
            pre = lin(o["ctx"]); post = act(pre); lc = m.w_c(post)
            acc["ctx"].append(o["ctx"].norm(dim=-1)); acc["pre"].append(pre.norm(dim=-1))
            acc["post"].append(post.norm(dim=-1)); acc["lc"].append(lc.norm(dim=-1))
            acc["dead"].append((pre < -2.0).float().mean(dim=-1))
            acc["pre_mean"].append(pre.mean(dim=-1))
        m.train()
        out = {k: torch.cat(v).mean().item() for k, v in acc.items()}
        out["w_c"] = m.w_c.weight.norm().item()
        out["proj_c_w"] = lin.weight.norm().item()
        out["proj_c_b"] = lin.bias.mean().item()
        return out

    NS = S(8000, 3000, 400)
    st = Store("silencing")
    for sd in SEEDS_MAIN:
        spec = {"block": "silencing", "seed": sd, "steps": NS, "lam": 0.1, "load": 4}

        def _go(sd=sd):
            task = make_task("selective_recall")
            set_seed(sd); m = build("gru_attn", n_out=task.n_out).to(DEVICE)
            opt = torch.optim.AdamW(m.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
            rows = []
            for step in range(1, NS + 1):
                m.train(); b = task.generate(CFG.batch_size, 4); o = m(b["x"])
                loss = F.cross_entropy(o["logits"], b["y"]) + \
                    0.1 * o["parts"]["context"].norm(dim=-1).mean()
                opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(m.parameters(), CFG.grad_clip); opt.step()
                if step % 250 == 0:
                    rows.append({"step": step, **readout_path(m, task, 4)})
            return {"rows": rows}

        cached(st, spec, _go, label=f"seed {sd}")

    SIL = st.frame(); SIL.to_csv(csv("p10_readout_path"), index=False)
    t = SIL.groupby("step")[["ctx", "pre", "post", "lc", "dead", "pre_mean",
                             "w_c", "proj_c_w", "proj_c_b"]].mean()
    f0, l0 = t.iloc[0], t.iloc[-1]
    print("\nfold-change along the retrieval readout:")
    for c in ["ctx", "pre", "post", "lc", "w_c", "proj_c_w"]:
        print(f"  {c:<10} {f0[c]:>9.4f} -> {l0[c]:>9.4f}   ({l0[c]/max(f0[c],1e-12):>8.4f}x)")
    print(f"  dead fraction (pre < -2): {f0.dead:.3f} -> {l0.dead:.3f}")
    print(f"  mean pre-activation:      {f0.pre_mean:+.3f} -> {l0.pre_mean:+.3f}")
    print(f"  proj_c bias mean:         {f0.proj_c_b:+.3f} -> {l0.proj_c_b:+.3f}")


## §5.4 Reversibility

Snapshots at share *thresholds* rather than fixed steps, so the dose–response is
on displacement rather than training time. The second block repeats the staged
walk-down that originally produced an "irreversible" claim and gives it a long
λ=0 tail; if share keeps climbing past 1500 steps, that claim was a budget
artefact.

In [ ]:
if RUN["reversibility"]:
    ON, OFF = S(12000, 4000, 400), S(10000, 3000, 300)
    st = Store("reversibility")
    for sd in SEEDS_MAIN:
        spec = {"block": "reversibility", "seed": sd, "on": ON, "off": OFF,
                "lam": 0.1, "load": 4}

        def _go(sd=sd):
            task = make_task("selective_recall")
            set_seed(sd); m = build("gru_attn", n_out=task.n_out).to(DEVICE)
            opt = torch.optim.AdamW(m.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
            rem, snaps = [0.30, 0.10, 0.03, 1e-3, 1e-5], {}
            for step in range(1, ON + 1):
                m.train(); b = task.generate(CFG.batch_size, 4); o = m(b["x"])
                loss = F.cross_entropy(o["logits"], b["y"]) + \
                    0.1 * o["parts"]["context"].norm(dim=-1).mean()
                opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(m.parameters(), CFG.grad_clip); opt.step()
                if step % 250 == 0:
                    sh = deps(m, task, 4, n_batches=4)["context_share"]
                    while rem and sh <= rem[0]:
                        thr = rem.pop(0)
                        snaps[thr] = (copy.deepcopy(m.state_dict()), sh, step)
            rows = []
            for thr, (state, sh, stp) in snaps.items():
                set_seed(sd + 900)
                m2 = build("gru_attn", n_out=task.n_out).to(DEVICE); m2.load_state_dict(state)
                o2 = torch.optim.AdamW(m2.parameters(), lr=CFG.lr,
                                       weight_decay=CFG.weight_decay)
                for _ in range(OFF):
                    m2.train(); b = task.generate(CFG.batch_size, 4)
                    loss = F.cross_entropy(m2(b["x"])["logits"], b["y"])
                    o2.zero_grad(set_to_none=True); loss.backward()
                    nn.utils.clip_grad_norm_(m2.parameters(), CFG.grad_clip); o2.step()
                fin = deps(m2, task, 4)
                rows.append({"threshold": thr, "share_snap": sh, "snap_step": stp,
                             "share_final": fin["context_share"],
                             "recovery": fin["context_share"] - sh,
                             "acc_final": fin["acc_clean"]})
                print(f"    [rev] s{sd} thr {thr:<8} {sh:.2e} -> {fin['context_share']:.4f}")
            return {"rows": rows}

        cached(st, spec, _go, label=f"seed {sd}")

    REV = st.frame(); REV.to_csv(csv("p11_reversibility"), index=False)
    if len(REV):
        print("\n", REV.groupby("threshold")[["share_snap", "share_final", "recovery",
                                              "acc_final"]].agg(["mean", "std"]).round(4).to_string())

    TAIL = S(15000, 5000, 400)
    st2 = Store("reversibility_tail")
    for sd in SEEDS_AUX[:2]:
        spec = {"block": "reversibility_tail", "seed": sd, "tail": TAIL, "load": 8}

        def _go(sd=sd, spec=None):
            task = make_task("selective_recall")
            set_seed(sd); m = build("gru_attn", n_out=task.n_out).to(DEVICE)
            for lam in [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.86, 1.5]:
                m, _ = train(task, 8, lam=lam, seed=sd, steps=1500, model=m,
                             schedule="constant")
            d0 = deps(m, task, 8)
            print(f"    [tail] seed {sd}: after up-leg share {d0['context_share']:.2e} "
                  f"acc {d0['acc_clean']:.3f}")
            opt = torch.optim.AdamW(m.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
            rows = []
            for step in range(1, TAIL + 1):
                m.train(); b = task.generate(CFG.batch_size, 8)
                loss = F.cross_entropy(m(b["x"])["logits"], b["y"])
                opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(m.parameters(), CFG.grad_clip); opt.step()
                if step % 500 == 0:
                    rows.append({"tail_step": step, **deps(m, task, 8, n_batches=4)})
            return {"share_after_upleg": d0["context_share"], "rows": rows}

        cached(st2, spec, _go, label=f"tail seed {sd}")

    TL = st2.frame(); TL.to_csv(csv("p11_long_tail"), index=False)
    if len(TL):
        tt = TL.groupby("tail_step")[["context_share", "acc_clean"]].mean()
        if 1500 in tt.index:
            print(f"\n  share at 1500 steps of lambda=0: {tt.loc[1500].context_share:.4f}")
        print(f"  share at {int(tt.index.max())}: {tt.context_share.iloc[-1]:.4f}")
        print("  Still climbing means the irreversibility claim was a budget artefact.")


In [ ]:
TL = pd.read_csv(csv("p11_long_tail"))

# The tail arm is priced up to lambda=1.5, which is far past the point where the
# model stops solving the task (accuracy 0.660 and 0.716 after the up-leg). So
# the recovery curve starts from a model that is broken in two ways at once: the
# retrieval route is suppressed AND the task is no longer solved. Whether the
# rising share means "the route is coming back" or "the model is relearning the
# task from scratch" depends on which recovers first, and by how much.
tt = TL.groupby("tail_step")[["context_share", "acc_clean", "hidden_dep",
                              "attn_dep"]].agg(["mean", "std"])
print("tail recovery — share and accuracy together:\n")
print(f"  {'step':>6}  {'share':>16}  {'accuracy':>16}  {'hidden_dep':>8}  {'attn_dep':>8}")
for s, r in tt.iterrows():
    print(f"  {int(s):>6}  "
          f"{r[('context_share','mean')]:>7.4f}±{r[('context_share','std')]:<8.4f}  "
          f"{r[('acc_clean','mean')]:>7.4f}±{r[('acc_clean','std')]:<8.4f}  "
          f"{r[('hidden_dep','mean')]:>8.3f}  {r[('attn_dep','mean')]:>8.3f}")

start = TL[TL.tail_step == TL.tail_step.min()]
end = TL[TL.tail_step == TL.tail_step.max()]
up = TL.groupby("seed").share_after_upleg.first()
print(f"\n  after the up-leg:  share {up.mean():.2e},  accuracy "
      f"{TL.groupby('seed').acc_clean.first().mean():.3f} (reported by the block)")
print(f"  first logged step: share {start.context_share.mean():.4f}, "
      f"accuracy {start.acc_clean.mean():.4f}")
print(f"  last logged step:  share {end.context_share.mean():.4f}, "
      f"accuracy {end.acc_clean.mean():.4f}")

# The question the block cannot answer on its own: does accuracy return to
# ceiling, and does it do so before or after the share starts climbing?
acc_ok = end.acc_clean.mean() >= 0.99
print(f"\n  accuracy recovered to ceiling: {acc_ok}")
if acc_ok:
    first_ok = tt[tt[("acc_clean", "mean")] >= 0.99].index
    s_at = tt.loc[first_ok.min(), ("context_share", "mean")] if len(first_ok) else np.nan
    print(f"  accuracy first reached 0.99 at step {int(first_ok.min())}, "
          f"where the share was {s_at:.4f}")
    print("  If accuracy recovers early and the share keeps climbing afterwards,")
    print("  the share is not a by-product of relearning the task — the route is")
    print("  reopening on a model that already works.")
else:
    print("  !! accuracy did NOT return to ceiling. The rising share is measured")
    print("     on a model that is not solving the task, so it cannot be read as")
    print("     the retrieval route recovering. Report the threshold table (where")
    print("     accuracy is 1.000 at every row) and drop the tail arm, or rerun it")
    print("     with a gentler up-leg that stops before accuracy collapses.")

# correlation across the trajectory, as a second read on the same question
if len(TL) > 3:
    r = TL.context_share.corr(TL.acc_clean)
    print(f"\n  share-accuracy correlation across the trajectory: {r:+.2f}")
    print("  Strongly positive means the two are recovering together and cannot")
    print("  be separated; near zero means the share moves independently of")
    print("  whether the task is solved.")

## §6 What does not explain the boundary

Five failed predictors, reported because the pattern is the point: each is a
property of *solutions*, and the endpoint is a stopping point in a slow drift.

In [ ]:
if RUN["negatives"]:
    task = make_task("selective_recall")
    LOADS = [2, 4, 6, 8, 12]

    # 6.1 — the accuracy retrieval buys, against training budget.
    # Three budgets, so the "Delta vanishes with budget" claim can be checked at
    # the long end rather than extrapolated from two points.
    st = Store("neg_delta")
    for L, budget, sd in itertools.product([8, 12], [3000, LONG, XLONG], SEEDS_AUX):
        spec = {"block": "neg_delta", "load": L, "budget": budget, "seed": sd, "lam": 0.1}

        def _go(L=L, budget=budget, sd=sd, spec=spec):
            m, _ = train(task, L, lam=0.1, seed=sd, steps=budget, schedule="constant",
                         ckpt_key=keyof(spec))
            af = accuracy(m, task, L)
            ar = accuracy(m, task, L, {"context": ("zero", 1.0)})
            return {"acc_full": af, "acc_recurrent_only": ar, "delta": af - ar}

        cached(st, spec, _go, label=f"delta load {L} budget {budget} seed {sd}")
    D = st.frame(); D.to_csv(csv("p12_delta"), index=False)
    print("S6.1 Delta against training budget (mean over seeds):")
    print(D.pivot_table(index="load", columns="budget", values="delta").round(3).to_string())

    # 6.2 — optimisation cost of the recurrent solution
    st = Store("neg_optcost")
    for L, sd in itertools.product(LOADS, SEEDS_AUX):
        spec = {"block": "neg_optcost", "load": L, "seed": sd, "steps": 3 * STEPS}

        def _go(L=L, sd=sd):
            set_seed(sd); m = build("gru_attn", n_out=task.n_out).to(DEVICE)
            opt = torch.optim.AdamW(m.parameters(), lr=CFG.lr); hit = None
            for step in range(1, 3 * STEPS + 1):
                m.train(); b = task.generate(CFG.batch_size, L)
                loss = F.cross_entropy(
                    m(b["x"], intervene={"context": ("zero", 1.0)})["logits"], b["y"])
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
                if hit is None and step % 50 == 0 and \
                        accuracy(m, task, L, {"context": ("zero", 1.0)}, 2) >= 0.90:
                    hit = step
            return {"steps_to_90": hit}

        cached(st, spec, _go)
    OPT = st.frame(); OPT.to_csv(csv("p12_optcost"), index=False)
    print("\nS6.2 steps for recurrence alone to reach 90%:")
    print(OPT.groupby("load").steps_to_90.mean().to_string())

    # 6.3 / 6.4 — full-ablation cross-entropy damage and the local derivative
    st = Store("neg_ce")
    for L, sd in itertools.product(LOADS, SEEDS_AUX):
        spec = {"block": "neg_ce", "load": L, "seed": sd, "steps": STEPS}

        def _go(L=L, sd=sd):
            m, _ = train(task, L, lam=0.0, seed=sd, steps=STEPS)
            m.eval()
            with torch.no_grad():
                ce0, cea, lcn, cee = [], [], [], []
                for _ in range(8):
                    b = task.generate(CFG.batch_size, L); o = m(b["x"])
                    ce0.append(F.cross_entropy(o["logits"], b["y"]).item())
                    lcn.append(o["parts"]["context"].norm(dim=-1).mean().item())
                    cea.append(F.cross_entropy(
                        m(b["x"], intervene={"context": ("shuffle", 1.0)})["logits"], b["y"]).item())
                    cee.append(F.cross_entropy(
                        m(b["x"], intervene={"context": ("zero", 0.05)})["logits"], b["y"]).item())
            m.train()
            base, lc0 = float(np.mean(ce0)), float(np.mean(lcn))
            return {"delta_ce": float(np.mean(cea)) - base, "lc0": lc0,
                    "pred_lam": (float(np.mean(cea)) - base) / lc0,
                    "local_deriv": (float(np.mean(cee)) - base) / (0.05 * lc0)}

        cached(st, spec, _go)
    CE = st.frame(); CE.to_csv(csv("p12_ce"), index=False)
    print("\nS6.3-6.4 full-ablation Delta-CE and the local derivative (mean over seeds):")
    print(CE.groupby("load")[["delta_ce", "lc0", "pred_lam", "local_deriv"]]
          .mean().round(5).to_string())
    print("  Delta-CE flat across loads = saturated; local derivative ~ 0 = the loss is")
    print("  flat to first order at the operating point. Both non-predictors.")

    # 6.5 — short-horizon route learnability
    def route_gain(state, load, which, batches, lam, steps=200):
        set_seed(0); m = build("gru_attn", n_out=task.n_out).to(DEVICE)
        m.load_state_dict(state)
        rec, att, shared = route_params(m)
        for p in (att if which == "recurrent" else rec):
            p.requires_grad_(False)
        tp = (rec if which == "recurrent" else att) + shared
        opt = torch.optim.AdamW(tp, lr=CFG.lr)

        def obj(b):
            o = m(b["x"])
            v = F.cross_entropy(o["logits"], b["y"])
            return v + lam * o["parts"]["context"].norm(dim=-1).mean() if lam > 0 else v

        with torch.no_grad():
            before = float(np.mean([obj(b).item() for b in batches]))
        for _ in range(steps):
            m.train(); b = task.generate(CFG.batch_size, load)
            loss = obj(b); opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        with torch.no_grad():
            return before - float(np.mean([obj(b).item() for b in batches]))

    CK = (100, 300, 500, 1000, 2000)
    st = Store("neg_learnability")
    for lam, L, sd in itertools.product([0.0, 0.02, 0.1], [4, 8], SEEDS_AUX):
        spec = {"block": "neg_learnability", "lam": lam, "load": L, "seed": sd}

        def _go(lam=lam, L=L, sd=sd):
            set_seed(sd); m = build("gru_attn", n_out=task.n_out).to(DEVICE)
            opt = torch.optim.AdamW(m.parameters(), lr=CFG.lr); saved = {}
            for step in range(1, max(CK) + 1):
                m.train(); b = task.generate(CFG.batch_size, L); o = m(b["x"])
                loss = F.cross_entropy(o["logits"], b["y"])
                if lam > 0:
                    loss = loss + lam * o["parts"]["context"].norm(dim=-1).mean()
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
                if step in CK:
                    saved[step] = copy.deepcopy(m.state_dict())
            mf, _ = train(task, L, lam=lam, seed=sd, steps=STEPS)
            final = deps(mf, task, L)["context_share"]
            set_seed(1234); batches = [task.generate(CFG.batch_size, L) for _ in range(8)]
            rows = []
            for t_, stt in saved.items():
                rows.append({"step": t_, "final_share": final,
                             "gain_attn": route_gain(stt, L, "attention", batches, lam),
                             "gain_rec": route_gain(stt, L, "recurrent", batches, lam)})
            for r in rows:
                r["A"] = r["gain_attn"] - r["gain_rec"]
            return {"rows": rows}

        cached(st, spec, _go, label=f"learnability lam {lam} load {L} seed {sd}")
    A = st.frame(); A.to_csv(csv("p12_learnability"), index=False)
    print("\nS6.5 does sign(A(t)) predict the final strategy?")
    for t_, g in A.groupby("step"):
        print(f"  step {int(t_):>5}: sign accuracy "
              f"{((g.A > 0) == (g.final_share > 0.5)).mean():.2f}  (chance 0.50, n={len(g)})"
              f"   mean gains: attn {g.gain_attn.mean():+.3f} rec {g.gain_rec.mean():+.3f}")


In [ ]:
# §6.5 with intervals. The section claims no predictor works, so the one
# checkpoint that looks predictive (0.83 at step 100) has to be shown to be
# within noise rather than left for a reviewer to query. n=18 is small: a
# Wilson interval at that size is wide enough that 0.83 and 0.39 both include
# values close to chance.
A = pd.read_csv(csv("p12_learnability"))


def wilson(k, n, z=1.96):
    """Wilson score interval — behaves sensibly at small n and near 0 or 1,
    unlike the normal approximation."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return (max(0.0, c - h), min(1.0, c + h))


print("§6.5 does sign(A(t)) predict the final strategy?\n")
print(f"  {'step':>6}  {'correct':>8}  {'accuracy':>8}  {'95% CI':>16}  "
      f"{'gain attn':>10}  {'gain rec':>9}  {'|A|':>8}  {'|A|/gain':>9}")
rows = []
for t_, g in A.groupby("step"):
    correct = ((g.A > 0) == (g.final_share > 0.5))
    k, n = int(correct.sum()), len(correct)
    lo, hi = wilson(k, n)
    ga, gr = g.gain_attn.mean(), g.gain_rec.mean()
    absA = g.A.abs().mean()
    common = (abs(ga) + abs(gr)) / 2          # how much both routes gained
    ratio = absA / common if common > 1e-9 else np.nan
    rows.append({"step": int(t_), "k": k, "n": n, "acc": k / n, "lo": lo, "hi": hi,
                 "gain_attn": ga, "gain_rec": gr, "mean_abs_A": absA,
                 "A_over_gain": ratio, "excludes_chance": lo > 0.5 or hi < 0.5})
    print(f"  {int(t_):>6}  {k:>3}/{n:<4}  {k/n:>8.2f}  "
          f"[{lo:>5.2f}, {hi:>5.2f}]  {ga:>+10.3f}  {gr:>+9.3f}  "
          f"{absA:>8.4f}  {ratio:>9.2f}")

L65 = pd.DataFrame(rows)
L65.to_csv(csv("p12_learnability_ci"), index=False)

sig = L65[L65.excludes_chance]
print(f"\n  checkpoints whose interval excludes chance: {len(sig)}/{len(L65)}")
if len(sig):
    print(sig[["step", "acc", "lo", "hi"]].round(3).to_string(index=False))
    print("  Those need explaining, not ignoring — see the ratio column.")
else:
    print("  None. At n=18 the intervals are wide enough that every checkpoint")
    print("  is consistent with chance, including the 0.83 at step 100.")

print("\n  |A|/gain is the size of the route DIFFERENCE against the size of what")
print("  both routes gained. Where it is small, the two routes are improving")
print("  together and sign(A) is the sign of a difference between two nearly")
print("  equal large numbers — which is why an early checkpoint can look")
print("  predictive without any route actually outcompeting the other.")

# Does A(t) predict anything about the final share, beyond its sign?
if len(A) > 3:
    print("\n  correlation between A(t) and the final retrieval share:")
    for t_, g in A.groupby("step"):
        r = g.A.corr(g.final_share)
        rows_ = len(g.dropna(subset=["A", "final_share"]))
        print(f"    step {int(t_):>5}:  r = {r:+.2f}  (n={rows_})")
    print("  A predictor should show a consistent, sizeable correlation at some")
    print("  checkpoint. Sign accuracy near chance AND correlations near zero")
    print("  together are what make this a negative result rather than an")
    print("  underpowered one.")

# per-condition breakdown: is the failure uniform, or driven by one cell?
print("\n  by condition, at each checkpoint (sign correct / n):")
piv = (A.assign(correct=(A.A > 0) == (A.final_share > 0.5))
        .pivot_table(index=["lam", "load"], columns="step", values="correct",
                     aggfunc="mean"))
print(piv.round(2).to_string())
print("\n  If some cells are always right and others always wrong, the predictor")
print("  is capturing the condition rather than the dynamics, and the pooled")
print("  number hides it.")

In [ ]:
# =============================================================================
# §6.5 extended — is the step-500 predictor real, and does it fail at the
# boundary?
#
# The first pass found r = +0.84 between A(500) and the final retrieval share
# (n=18), with sign accuracy above chance at two checkpoints. It also found one
# condition — lambda=0.02, load 8 — where every seed was predicted wrong. That
# condition is the one closest to its boundary, which is where a predictor of
# "which route wins" should break down if the competition there is genuinely
# balanced.
#
# Two things are needed to say either way. More seeds, because n=18 across six
# conditions cannot separate a real effect from a lucky one. And a lambda grid
# placed relative to each load's boundary rather than shared across loads, so
# "distance from the boundary" is a measured variable instead of an accident of
# which lambdas happened to be swept.
#
# lambda* from the 5-seed boundary fit, load* = 14.01 + 1.46 log2(lambda):
#   load 4 -> 0.0086     load 8 -> 0.0577   (width block gives 0.078 at load 8;
# the grids below bracket both estimates)
# =============================================================================

LAM_STAR = {4: 0.0086, 8: 0.0577}
LAM_GRID = {4: [0.0, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1],
            8: [0.0, 0.01, 0.03, 0.06, 0.1, 0.2]}
CK = (100, 300, 500, 1000, 2000)

task = make_task("selective_recall")


def route_gain(state, load, which, batches, lam, steps=200):
    """Train one route for a short horizon with the other frozen, and report how
    much it reduced the objective. The shared encoder trains either way, so this
    measures how much a route can improve given the current state, not how good
    it already is."""
    set_seed(0)
    m = build("gru_attn", n_out=task.n_out).to(DEVICE)
    m.load_state_dict(state)
    rec, att, shared = route_params(m)
    for p in (att if which == "recurrent" else rec):
        p.requires_grad_(False)
    tp = (rec if which == "recurrent" else att) + shared
    opt = torch.optim.AdamW(tp, lr=CFG.lr)

    def obj(b):
        o = m(b["x"])
        v = F.cross_entropy(o["logits"], b["y"])
        return v + lam * o["parts"]["context"].norm(dim=-1).mean() if lam > 0 else v

    with torch.no_grad():
        before = float(np.mean([obj(b).item() for b in batches]))
    for _ in range(steps):
        m.train(); b = task.generate(CFG.batch_size, load)
        loss = obj(b); opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    with torch.no_grad():
        return before - float(np.mean([obj(b).item() for b in batches]))


st = Store("neg_learnability")
JOBS = [(lam, L) for L in LAM_GRID for lam in LAM_GRID[L]]
n_units = len(JOBS) * len(SEEDS_MAIN)
print(f"{n_units} units ({len(JOBS)} conditions x {len(SEEDS_MAIN)} seeds); "
      f"the 18 from the first pass are cached")

for (lam, L), sd in itertools.product(JOBS, SEEDS_MAIN):
    spec = {"block": "neg_learnability", "lam": lam, "load": L, "seed": sd}

    def _go(lam=lam, L=L, sd=sd):
        set_seed(sd)
        m = build("gru_attn", n_out=task.n_out).to(DEVICE)
        opt = torch.optim.AdamW(m.parameters(), lr=CFG.lr)
        saved = {}
        for step in range(1, max(CK) + 1):
            m.train(); b = task.generate(CFG.batch_size, L); o = m(b["x"])
            loss = F.cross_entropy(o["logits"], b["y"])
            if lam > 0:
                loss = loss + lam * o["parts"]["context"].norm(dim=-1).mean()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            if step in CK:
                saved[step] = copy.deepcopy(m.state_dict())
        mf, _ = train(task, L, lam=lam, seed=sd, steps=STEPS)
        fin = deps(mf, task, L)
        set_seed(1234)
        batches = [task.generate(CFG.batch_size, L) for _ in range(8)]
        rows = []
        for t_, stt in saved.items():
            rows.append({"step": t_, "final_share": fin["context_share"],
                         "final_acc": fin["acc_clean"],
                         "final_hidden_dep": fin["hidden_dep"],
                         "final_attn_dep": fin["attn_dep"],
                         "gain_attn": route_gain(stt, L, "attention", batches, lam),
                         "gain_rec": route_gain(stt, L, "recurrent", batches, lam)})
        for r in rows:
            r["A"] = r["gain_attn"] - r["gain_rec"]
        return {"rows": rows}

    cached(st, spec, _go, label=f"lam {lam:<6g} load {L} seed {sd}")

A = st.frame()
A = A[A.load.isin(LAM_STAR)]
A["dist"] = [np.log2(l / LAM_STAR[L]) if l > 0 else -np.inf
             for l, L in zip(A.lam, A.load)]        # doublings from the boundary
A.to_csv(csv("p12_learnability"), index=False)


def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p, d = k / n, 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return (max(0.0, c - h), min(1.0, c + h))


def corr_ci(x, y, n_boot=10000, seed=0):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 4:
        return (np.nan, np.nan, np.nan, len(x))
    r = np.corrcoef(x, y)[0, 1]
    rng = np.random.default_rng(seed)
    bs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(x), len(x))
        if np.std(x[i]) > 1e-12 and np.std(y[i]) > 1e-12:
            bs.append(np.corrcoef(x[i], y[i])[0, 1])
    return (r, float(np.quantile(bs, .025)), float(np.quantile(bs, .975)), len(x))


print("\n" + "=" * 78)
print("§6.5  short-horizon route learnability against the final strategy\n")
print(f"  {'step':>6}  {'sign acc':>18}  {'r(A, final share)':>26}  {'|A|/gain':>9}")
summ = []
for t_, g in A.groupby("step"):
    correct = (g.A > 0) == (g.final_share > 0.5)
    k, n = int(correct.sum()), len(correct)
    lo, hi = wilson(k, n)
    r, rl, rh, nr = corr_ci(g.A, g.final_share)
    common = (g.gain_attn.abs().mean() + g.gain_rec.abs().mean()) / 2
    ratio = g.A.abs().mean() / common if common > 1e-9 else np.nan
    summ.append({"step": int(t_), "k": k, "n": n, "sign_acc": k / n,
                 "sign_lo": lo, "sign_hi": hi, "r": r, "r_lo": rl, "r_hi": rh,
                 "A_over_gain": ratio})
    print(f"  {int(t_):>6}  {k:>3}/{n:<3} {k/n:.2f} [{lo:.2f},{hi:.2f}]  "
          f"r={r:+.2f} [{rl:+.2f},{rh:+.2f}] n={nr:<3}  {ratio:>9.2f}")
S65 = pd.DataFrame(summ)
S65.to_csv(csv("p12_learnability_summary"), index=False)

best = S65.loc[S65.r.idxmax()]
print(f"\n  strongest checkpoint: step {int(best.step)}, "
      f"r = {best.r:+.2f} [{best.r_lo:+.2f}, {best.r_hi:+.2f}]")
print(f"  its interval {'excludes' if best.r_lo > 0 else 'includes'} zero")

# ---- the boundary question -------------------------------------------------
print("\n" + "-" * 78)
print("does the predictor fail near the boundary?\n")
NEAR = 1.0          # within one doubling of lambda* either side
for t_ in sorted(A.step.unique()):
    g = A[(A.step == t_) & np.isfinite(A.dist)]
    if not len(g):
        continue
    near, far = g[g.dist.abs() <= NEAR], g[g.dist.abs() > NEAR]
    out = [f"  step {int(t_):>5}"]
    for name, sub in [("near", near), ("far", far)]:
        if len(sub) >= 4:
            k = int(((sub.A > 0) == (sub.final_share > 0.5)).sum())
            r, rl, rh, _ = corr_ci(sub.A, sub.final_share)
            out.append(f"{name}: sign {k}/{len(sub)}={k/len(sub):.2f}  "
                       f"r={r:+.2f} [{rl:+.2f},{rh:+.2f}]")
        else:
            out.append(f"{name}: n={len(sub)} too few")
    print("   ".join(out))

print("\n  by distance from the boundary, at the strongest checkpoint:")
g = A[(A.step == int(best.step)) & np.isfinite(A.dist)]
tab = (g.assign(correct=(g.A > 0) == (g.final_share > 0.5))
        .groupby(["load", "lam", "dist"])
        .agg(correct=("correct", "mean"), n=("correct", "size"),
             mean_A=("A", "mean"), final_share=("final_share", "mean"),
             final_acc=("final_acc", "mean")).reset_index())
tab["|dist|"] = tab.dist.abs().round(2)
print(tab.sort_values("|dist|").round(3).to_string(index=False))

print("\n  A predictor that works away from the boundary and fails at it is not")
print("  a failed predictor — it is one whose failure is located where the two")
print("  routes are closely matched, which is what the boundary means. Report")
print("  the near/far split rather than the pooled number either way.")
print("=" * 78)

In [ ]:
g = A[(A.step == 300) & np.isfinite(A.dist) & (A.lam > 0)]
print("r(A, log2 lam)        ", g.A.corr(np.log2(g.lam)).round(2))
print("r(A, final_share)     ", g.A.corr(g.final_share).round(2))
resid_A = g.A - np.polyval(np.polyfit(np.log2(g.lam), g.A, 1), np.log2(g.lam))
resid_S = g.final_share - np.polyval(
    np.polyfit(np.log2(g.lam), g.final_share, 1), np.log2(g.lam))
print("partial r, lam removed", np.corrcoef(resid_A, resid_S)[0, 1].round(2))

In [ ]:
# The partial correlation is now the load-bearing number in §6.5, so it needs an
# interval — and it needs computing at every checkpoint, not just the one that
# looked best. Reporting a single r from the checkpoint chosen after seeing the
# results is how a window gets invented; showing the whole profile is how it
# gets established.
#
# lambda enters the probe's own objective, so part of r(A, final_share) is
# lambda predicting itself. Removing lambda linearly in log2 leaves the
# correlation that is not attributable to that.

def partial_r(x, y, z):
    """Correlation between x and y with the linear effect of z removed from both."""
    x, y, z = (np.asarray(v, float) for v in (x, y, z))
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    x, y, z = x[ok], y[ok], z[ok]
    if len(x) < 4 or np.std(z) < 1e-12:
        return np.nan, len(x)
    rx = x - np.polyval(np.polyfit(z, x, 1), z)
    ry = y - np.polyval(np.polyfit(z, y, 1), z)
    if np.std(rx) < 1e-12 or np.std(ry) < 1e-12:
        return np.nan, len(x)
    return float(np.corrcoef(rx, ry)[0, 1]), len(x)


def partial_r_ci(x, y, z, n_boot=10000, seed=0):
    r, n = partial_r(x, y, z)
    if not np.isfinite(r):
        return r, np.nan, np.nan, n
    x, y, z = (np.asarray(v, float) for v in (x, y, z))
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    x, y, z = x[ok], y[ok], z[ok]
    rng = np.random.default_rng(seed)
    bs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(x), len(x))
        rb, _ = partial_r(x[i], y[i], z[i])
        if np.isfinite(rb):
            bs.append(rb)
    if len(bs) < 100:
        return r, np.nan, np.nan, n
    return r, float(np.quantile(bs, .025)), float(np.quantile(bs, .975)), n


AA = A[(A.lam > 0) & np.isfinite(A.dist)].copy()
AA["log2lam"] = np.log2(AA.lam)

print("§6.5  A(t) against the final strategy, before and after removing lambda\n")
print(f"  {'step':>6}  {'raw r':>22}  {'partial r (lambda out)':>30}  "
      f"{'r(A, log2 lam)':>15}  {'n':>4}")
prof = []
for t_ in sorted(AA.step.unique()):
    g = AA[AA.step == t_]
    r_raw, rl, rh, n = partial_r_ci(g.A, g.final_share, np.zeros(len(g)) + 0.0) \
        if False else (*corr_ci(g.A, g.final_share)[:3], len(g))
    r_p, pl, ph, npp = partial_r_ci(g.A, g.final_share, g.log2lam)
    r_lam = g.A.corr(g.log2lam)
    prof.append({"step": int(t_), "r_raw": r_raw, "raw_lo": rl, "raw_hi": rh,
                 "r_partial": r_p, "partial_lo": pl, "partial_hi": ph,
                 "r_A_lam": r_lam, "n": npp,
                 "excludes_zero": np.isfinite(pl) and (pl > 0 or ph < 0)})
    print(f"  {int(t_):>6}  {r_raw:+.2f} [{rl:+.2f},{rh:+.2f}]  "
          f"{r_p:+.2f} [{pl:+.2f},{ph:+.2f}]{'  *' if (np.isfinite(pl) and pl > 0) else '   '}  "
          f"{r_lam:>+15.2f}  {npp:>4}")
P65 = pd.DataFrame(prof)
P65.to_csv(csv("p12_partial_profile"), index=False)

print("\n  * = the partial correlation's 95% interval excludes zero.")
print("  A window is credible if the profile rises to a peak and falls away on")
print("  both sides. A single isolated checkpoint is a multiple-comparisons")
print("  artefact and should be reported as one.")

# ---- the same profile, split at the boundary -------------------------------
print("\n" + "-" * 78)
print("partial correlation near vs far from lambda*  "
      f"(near = within {1.0:g} doubling)\n")
NEAR = 1.0
rows = []
for t_ in sorted(AA.step.unique()):
    g = AA[AA.step == t_]
    line = [f"  step {int(t_):>5}"]
    rec = {"step": int(t_)}
    for name, sub in [("near", g[g.dist.abs() <= NEAR]), ("far", g[g.dist.abs() > NEAR])]:
        if len(sub) >= 8:
            r_, l_, h_, n_ = partial_r_ci(sub.A, sub.final_share, sub.log2lam)
            line.append(f"{name}: r={r_:+.2f} [{l_:+.2f},{h_:+.2f}] n={n_:<3}")
            rec |= {f"{name}_r": r_, f"{name}_lo": l_, f"{name}_hi": h_, f"{name}_n": n_}
        else:
            line.append(f"{name}: n={len(sub)} too few")
            rec |= {f"{name}_r": np.nan, f"{name}_lo": np.nan,
                    f"{name}_hi": np.nan, f"{name}_n": len(sub)}
    rows.append(rec)
    print("   ".join(line))
N65 = pd.DataFrame(rows)
N65.to_csv(csv("p12_partial_nearfar"), index=False)

best = P65.loc[P65.r_partial.idxmax()]
b = N65[N65.step == best.step].iloc[0] if len(N65[N65.step == best.step]) else None
print(f"\n  strongest checkpoint: step {int(best.step)}, partial r = "
      f"{best.r_partial:+.2f} [{best.partial_lo:+.2f}, {best.partial_hi:+.2f}]")
if b is not None and np.isfinite(b.get("near_r", np.nan)) and np.isfinite(b.get("far_r", np.nan)):
    near_null = b.near_lo <= 0 <= b.near_hi
    far_sig = b.far_lo > 0
    print(f"  far from the boundary  r = {b.far_r:+.2f} [{b.far_lo:+.2f}, {b.far_hi:+.2f}]"
          f"  ({'excludes' if far_sig else 'includes'} zero)")
    print(f"  near the boundary      r = {b.near_r:+.2f} [{b.near_lo:+.2f}, {b.near_hi:+.2f}]"
          f"  ({'includes' if near_null else 'excludes'} zero)")
    if far_sig and near_null:
        print("\n  The predictor works away from the boundary and is indistinguishable")
        print("  from zero at it. That is the claim §6.5 can make: the quantity")
        print("  predicts the outcome where the outcome was not in question, and")
        print("  stops predicting exactly where the two routes are matched.")
    else:
        print("\n  The near/far split does not separate. Report the pooled partial")
        print("  correlation and drop the boundary-localisation claim.")

print("\n  Also state plainly: mean A is negative in every condition, so the SIGN")
print("  of A carries no information and the sign-accuracy framing in the draft")
print("  does not survive. Only the magnitude predicts.")

In [ ]:
# Testing the DIFFERENCE, not comparing two significance verdicts.
#
# "near includes zero, far excludes zero" is not evidence that the correlation
# is lower near the boundary — with n=25 the near interval [-0.12, +0.70] easily
# contains the far estimate of +0.46. The claim needs the difference itself
# bootstrapped, so the two groups are resampled independently and the interval
# on (r_far - r_near) is what decides it.
#
# Two further checks, because both are things a reviewer would raise:
#   - a permutation test on the same difference, which makes no distributional
#     assumption about correlations
#   - the multiple-comparisons picture across the five checkpoints, since step
#     1000 is significant while the checkpoints either side of it are not, and
#     one significant result in five at alpha=0.05 is what chance produces

def _partial(x, y, z):
    x, y, z = (np.asarray(v, float) for v in (x, y, z))
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    x, y, z = x[ok], y[ok], z[ok]
    if len(x) < 4 or np.std(z) < 1e-12:
        return np.nan
    rx = x - np.polyval(np.polyfit(z, x, 1), z)
    ry = y - np.polyval(np.polyfit(z, y, 1), z)
    if np.std(rx) < 1e-12 or np.std(ry) < 1e-12:
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def diff_test(near, far, n_boot=10000, n_perm=10000, seed=0):
    """Bootstrap CI and permutation p for (r_far - r_near), both partialling
    out log2(lambda) within each resample."""
    rng = np.random.default_rng(seed)
    r_n = _partial(near.A, near.final_share, near.log2lam)
    r_f = _partial(far.A, far.final_share, far.log2lam)
    if not (np.isfinite(r_n) and np.isfinite(r_f)):
        return dict(r_near=r_n, r_far=r_f, diff=np.nan, lo=np.nan, hi=np.nan,
                    p=np.nan, n_near=len(near), n_far=len(far))
    obs = r_f - r_n

    bs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(near), len(near))
        j = rng.integers(0, len(far), len(far))
        a = _partial(near.A.values[i], near.final_share.values[i], near.log2lam.values[i])
        b = _partial(far.A.values[j], far.final_share.values[j], far.log2lam.values[j])
        if np.isfinite(a) and np.isfinite(b):
            bs.append(b - a)

    # permutation: shuffle the near/far label, keeping group sizes fixed
    pool = pd.concat([near, far], ignore_index=True)
    n_near = len(near)
    perm = []
    for _ in range(n_perm):
        idx = rng.permutation(len(pool))
        pn, pf = pool.iloc[idx[:n_near]], pool.iloc[idx[n_near:]]
        a = _partial(pn.A, pn.final_share, pn.log2lam)
        b = _partial(pf.A, pf.final_share, pf.log2lam)
        if np.isfinite(a) and np.isfinite(b):
            perm.append(b - a)
    p = (np.sum(np.abs(perm) >= abs(obs)) + 1) / (len(perm) + 1) if perm else np.nan

    return dict(r_near=r_n, r_far=r_f, diff=obs,
                lo=float(np.quantile(bs, .025)) if len(bs) > 100 else np.nan,
                hi=float(np.quantile(bs, .975)) if len(bs) > 100 else np.nan,
                p=p, n_near=len(near), n_far=len(far))


AA = A[(A.lam > 0) & np.isfinite(A.dist)].copy()
AA["log2lam"] = np.log2(AA.lam)
NEAR = 1.0

print("difference in partial correlation, far minus near the boundary")
print(f"(near = within {NEAR:g} doubling of lambda*; lambda partialled out in "
      f"every resample)\n")
print(f"  {'step':>6}  {'r near':>7}  {'r far':>7}  {'far - near':>26}  {'perm p':>7}")
rows = []
for t_ in sorted(AA.step.unique()):
    g = AA[AA.step == t_]
    d = diff_test(g[g.dist.abs() <= NEAR].reset_index(drop=True),
                  g[g.dist.abs() > NEAR].reset_index(drop=True))
    d["step"] = int(t_); rows.append(d)
    ci = (f"[{d['lo']:+.2f}, {d['hi']:+.2f}]" if np.isfinite(d["lo"]) else "  --  ")
    star = " *" if np.isfinite(d["lo"]) and (d["lo"] > 0 or d["hi"] < 0) else "  "
    print(f"  {int(t_):>6}  {d['r_near']:>+7.2f}  {d['r_far']:>+7.2f}  "
          f"{d['diff']:>+6.2f} {ci}{star}  {d['p']:>7.3f}")
D65 = pd.DataFrame(rows)
D65.to_csv(csv("p12_nearfar_difference"), index=False)

print("\n  * = the interval on the DIFFERENCE excludes zero. Only then can the")
print("    section say the correlation is lower near the boundary. Otherwise the")
print("    honest statement is that it is not detectable there, with the two")
print("    intervals overlapping.")

sig = D65[(D65.lo > 0) | (D65.hi < 0)]
if len(sig):
    print(f"\n  {len(sig)}/{len(D65)} checkpoints show a difference whose interval")
    print("  excludes zero:")
    print(sig[["step", "r_near", "r_far", "diff", "lo", "hi", "p"]]
          .round(3).to_string(index=False))
    print("\n  The boundary-localisation claim is supported at those checkpoints.")
else:
    print("\n  No checkpoint shows a difference whose interval excludes zero.")
    print("  Write the weaker sentence: the correlation is detectable away from")
    print("  the boundary and not detectable at it, and we do not claim the two")
    print("  differ. That costs nothing and removes the one objection a careful")
    print("  reviewer will raise.")

# ---- multiple comparisons across the five checkpoints ----------------------
print("\n" + "-" * 78)
print("five checkpoints were tested, so the pooled profile needs correcting.\n")
P = pd.read_csv(csv("p12_partial_profile"))


def _p_from_ci(lo, hi):
    """Two-sided p implied by a symmetric 95% bootstrap interval."""
    if not (np.isfinite(lo) and np.isfinite(hi)):
        return np.nan
    r, se = (lo + hi) / 2, (hi - lo) / (2 * 1.96)
    if se <= 0:
        return 0.0
    from math import erfc, sqrt
    return float(erfc(abs(r / se) / sqrt(2)))


P["p_approx"] = [_p_from_ci(l, h) for l, h in zip(P.partial_lo, P.partial_hi)]
P = P.sort_values("p_approx").reset_index(drop=True)
m = P.p_approx.notna().sum()
P["holm_thresh"] = [0.05 / (m - i) for i in range(len(P))]
P["survives_holm"] = P.p_approx <= P.holm_thresh
# Holm is step-down: once one fails, everything after it fails too
first_fail = P.index[~P.survives_holm].min() if (~P.survives_holm).any() else len(P)
P.loc[first_fail:, "survives_holm"] = False
print(P[["step", "r_partial", "partial_lo", "partial_hi", "p_approx",
         "holm_thresh", "survives_holm"]].round(4).to_string(index=False))
print(f"\n  {int(P.survives_holm.sum())}/{m} checkpoints survive Holm correction.")
print("  Step 1000 being significant while 500 and 2000 are not is the pattern")
print("  chance produces when five tests are run. If it does not survive")
print("  correction, say so and rest the window on 100-300, where the profile")
print("  is contiguous.")

## §6 The price axis: λ*(load), measured rather than inverted

§6 argues about λ*(L) — the price at which retrieval is dislodged at a fixed
load — while the boundary sweep measures L*(λ). Reading one off the other across
a coarse grid is exactly the cross-grid comparison that produced a retracted
claim, so λ*(L) is measured directly here on a dense price grid.

The grid runs to 0.5 for two reasons: §6.2's extrapolated prediction has to be
checkable at high λ, and the accuracy collapse — which is not the boundary, and
happens well after it — needs to be located rather than asserted.


In [ ]:
if RUN["lamstar"]:
    # S6 quotes lambda*(L) — the price at which retrieval is dislodged at a fixed
    # load — but the boundary sweep crosses on the load axis and gives L*(lambda).
    # Inverting one to get the other across a coarse grid is what produced the
    # earlier cross-grid error, so lambda*(L) is measured directly here. The grid
    # runs past 0.35 because that is where S6.2's prediction has to be checked and
    # where the accuracy collapse sets in.
    LAM_GRID = [0.0, 0.002, 0.005, 0.01, 0.02, 0.035, 0.05, 0.075,
                0.1, 0.15, 0.2, 0.25, 0.35, 0.5]
    LS_LOADS = [2, 4, 6, 8, 12, 16]
    conds = [{"lam": l, "load": L} for L in LS_LOADS for l in LAM_GRID]
    LS = sweep("lamstar", conds, SEEDS_AUX, STEPS, out=csv("p14_lamstar"))

    print("\ncontext_share (rows load, cols lambda):")
    print(LS.pivot_table(index="load", columns="lam", values="context_share").round(3).to_string())
    print("\naccuracy:")
    ACC = LS.pivot_table(index="load", columns="lam", values="acc_clean")
    print(ACC.round(3).to_string())

    CL = report_crossings(LS, "lam", "S6 lambda*(load)", keys=["load"], acc_floor=ACC_FLOOR)
    CL.to_csv(csv("p14_lamstar_crossings"), index=False)

    # where the model stops solving the task at all — the collapse, not the boundary
    coll = []
    for L in LS_LOADS:
        g = LS[LS.load == L].groupby("lam").acc_clean.mean()
        bad = g[g < 0.90]
        coll.append({"load": L, "lam_collapse": float(bad.index.min()) if len(bad) else np.nan,
                     "min_acc": float(g.min())})
    COL = pd.DataFrame(coll); COL.to_csv(csv("p14_collapse"), index=False)
    print("\nsmallest lambda at which accuracy falls below 0.90 (accuracy collapse,")
    print("distinct from the route boundary, which happens at ceiling):")
    print(COL.round(3).to_string(index=False))


## §6.6 Swap errors do not dissociate the algorithms

If recurrence fails by losing the binding between item and position, its errors
should be other items from the sequence. That is a real signature, and it is
also what attention produces when it mis-points: another position holds another
sequence item.

The test needs a symbol set large relative to the load, otherwise landing on a
presented item is near-certain by chance. The two arms are brought to comparable
error rates by different means — the recurrence arm by being priced at a load it
cannot hold, the retrieval arm by raising the load — so the comparison is between
error *compositions* at matched error rates, not between error rates.


In [ ]:
if RUN["swap"] and not IS_VISUAL:
    # §6.6 — do the two configurations produce different error signatures?
    #
    # This question has now been asked three ways and the answer changed with
    # the chance baseline, which is the finding as much as anything else.
    #
    # At 32 symbols both arms showed a large excess of sequence-item errors over
    # chance and the signature looked uninformative. But with 12 items drawn
    # from 32 symbols a third of the alphabet is present in any sequence, so the
    # baseline is 0.30 and rises to 0.78 by load 48 — an in-sequence rate of
    # 1.000 there is close to arithmetically forced and means almost nothing.
    #
    # At 128 symbols the baseline drops to 0.06-0.31 and the arms separate: the
    # recurrent arm's excess sits at zero across four loads while retrieval's
    # reaches 0.74. What was missing was a MATCHED pair at 128 — retrieval jumps
    # from a 0.000 error rate at load 20 to 0.107 at load 24, with nothing in
    # between. Loads 21-23 fill that gap.
    SWAP_ALPHABETS = [32, 128]
    REC_LOADS = [8, 9, 10, 11, 12]
    RET_LOADS = [16, 20, 21, 22, 23, 24, 32, 48]
    ACC_GATE, MIN_ERR = 0.95, 30

    @torch.no_grad()
    def swap_stats(m, task, load, cfg, n_batches=40):
        """Among error trials: how often is the prediction another item that was
        actually presented, against the per-trial chance of hitting one of the
        presented items by picking a wrong symbol uniformly."""
        S_ = cfg.n_symbols
        m.eval()
        n_err = in_seq = 0.0; chance = []; n_tot = 0
        for _ in range(n_batches):
            b = task.generate(cfg.batch_size, load)
            pred = m(b["x"])["logits"].argmax(-1)
            y, items = b["y"], b["items"]
            B = y.shape[0]; ar = torch.arange(B, device=y.device)
            present = torch.zeros(B, S_, device=y.device)
            present.scatter_(1, items, 1.0)
            present[ar, y] = 0.0                       # non-target items only
            wrong = pred != y
            n_tot += B; n_err += wrong.sum().item()
            if wrong.any():
                in_seq += present[ar, pred][wrong].sum().item()
                chance.append((present.sum(1) / (S_ - 1))[wrong])
        m.train()
        if n_err == 0:
            return {"err_rate": 0.0, "in_seq_rate": np.nan, "chance": np.nan,
                    "excess": np.nan, "n_err": 0}
        ch = torch.cat(chance).mean().item()
        rate = in_seq / n_err
        return {"err_rate": n_err / n_tot, "in_seq_rate": rate, "chance": ch,
                "excess": rate - ch, "n_err": int(n_err)}

    st = Store("swap")
    for S_ in SWAP_ALPHABETS:
        cfg_s = replace(CFG, n_symbols=S_, max_query_pos=64)
        for arm, loads in [("recurrent", REC_LOADS), ("retrieval", RET_LOADS)]:
            for L, sd in itertools.product(loads, SEEDS_AUX):
                spec = {"block": "swap", "arm": arm, "load": L, "seed": sd,
                        "n_symbols": S_, "steps": STEPS}

                def _go(arm=arm, L=L, sd=sd, cfg_s=cfg_s):
                    task = make_task("selective_recall", cfg_s)
                    if arm == "recurrent":
                        m, lam_u = anneal(task, L, seed=sd, steps=STEPS, cfg=cfg_s)
                    else:
                        m, lam_u = train(task, L, lam=0.0, seed=sd, steps=STEPS,
                                         cfg=cfg_s)[0], 0.0
                    return {"build_lam": lam_u, **deps(m, task, L, cfg_s),
                            **swap_stats(m, task, L, cfg_s)}

                cached(st, spec, _go,
                       label=f"S={S_:<4} {arm:<10} load {L} seed {sd}")

    SW = st.frame(); SW.to_csv(csv("p15_swap"), index=False)

    print("\n§6.6 error composition (mean over seeds):")
    print(SW.groupby(["n_symbols", "arm", "load"])
            [["acc_clean", "context_share", "err_rate", "in_seq_rate",
              "chance", "excess"]].mean().round(3).to_string())

    # A cell carries a claim only if the model still does the task AND makes
    # enough errors to characterise. Both halves matter: a model at 0.15
    # accuracy tells us about the alphabet, and one with two errors tells us
    # nothing at all.
    G = SW.groupby(["n_symbols", "arm", "load"]).agg(
        acc=("acc_clean", "mean"), err=("err_rate", "mean"),
        n_err=("n_err", "mean"), excess=("excess", "mean"),
        excess_sd=("excess", "std"), chance=("chance", "mean"))
    G["usable"] = (G.acc >= ACC_GATE) & (G.n_err >= MIN_ERR)
    print(f"\ncell usability (accuracy ≥ {ACC_GATE} and ≥ {MIN_ERR} error trials):")
    print(G.round(3).to_string())

    USE = set(G[G.usable].index)
    OK = SW[[(r.n_symbols, r.arm, r.load) in USE for r in SW.itertuples()]]

    # ---- single-arm summary, which does not need a pair ---------------------
    # The recurrent arm at 128 symbols is well sampled across four loads and its
    # excess sits at zero. That statement stands on its own and does not depend
    # on finding a matched retrieval cell.
    print("\nexcess over chance per arm, usable cells only:")
    for (S_, arm), g in G[G.usable].groupby(level=[0, 1]):
        vals = g.excess.values
        mu, lo, hi = boot_ci(vals) if len(vals) > 1 else (vals[0], np.nan, np.nan)
        ci = f" 95% CI [{lo:+.3f}, {hi:+.3f}]" if np.isfinite(lo) else ""
        loads = list(g.index.get_level_values(2))
        print(f"  S={S_:<4} {arm:<10} loads {loads}  excess {mu:+.3f}{ci}")
    print("  An arm whose interval contains zero is not producing sequence-item")
    print("  errors above chance; one whose interval is clear of zero is.")

    # ---- matched pairs ------------------------------------------------------
    print("\nmatched pairs (same alphabet, closest error rates, both usable):")
    pairs = []
    for S_ in sorted(SW.n_symbols.unique()):
        sub = OK[OK.n_symbols == S_]
        rec, ret = sub[sub.arm == "recurrent"], sub[sub.arm == "retrieval"]
        if not len(rec) or not len(ret):
            print(f"  S={S_}: no usable pair ({rec.load.nunique()} recurrent, "
                  f"{ret.load.nunique()} retrieval cells survive)")
            continue
        ret_err = ret.groupby("load").err_rate.mean()
        for Lr, g in rec.groupby("load"):
            e = g.err_rate.mean()
            Lt = (ret_err - e).abs().idxmin()
            t = ret[ret.load == Lt]
            pairs.append({
                "n_symbols": S_, "rec_load": Lr, "ret_load": Lt,
                "rec_err": e, "ret_err": ret_err[Lt],
                "err_ratio": max(e, ret_err[Lt]) / max(min(e, ret_err[Lt]), 1e-9),
                "rec_chance": g.chance.mean(), "ret_chance": t.chance.mean(),
                "rec_excess": g.excess.mean(), "rec_sd": g.excess.std(),
                "ret_excess": t.excess.mean(), "ret_sd": t.excess.std(),
                "excess_gap": t.excess.mean() - g.excess.mean()})
    if pairs:
        PR = pd.DataFrame(pairs)
        PR["separated"] = (PR.excess_gap.abs()
                           > PR.rec_sd.fillna(0) + PR.ret_sd.fillna(0))
        # a pair whose error rates differ by more than 3x is not "matched"
        PR["well_matched"] = PR.err_ratio <= 3.0
        PR.to_csv(csv("p15_swap_pairs"), index=False)
        print(PR.round(3).to_string(index=False))
        good = PR[PR.well_matched & PR.separated]
        print(f"\n  {int(PR.separated.sum())}/{len(PR)} pairs separate; "
              f"{int(PR.well_matched.sum())}/{len(PR)} have error rates within 3×; "
              f"{len(good)} are both")
        if len(good):
            print("\n  well-matched pairs that separate:")
            print(good[["n_symbols", "rec_load", "ret_load", "rec_err", "ret_err",
                        "rec_excess", "ret_excess", "excess_gap"]]
                  .round(3).to_string(index=False))
            print("\n  Retrieval's errors are sequence items; recurrence's are not.")
            print("  The swap signature DOES dissociate the two algorithms once the")
            print("  chance baseline is low enough to see through it. §6.6 is a")
            print("  positive result, and the 32-symbol version was a measurement")
            print("  artefact rather than a negative finding.")
        else:
            print("\n  No pair is both well matched and separated. State the")
            print("  single-arm result above instead, which does not require a")
            print("  pair, and say the matched comparison was not achievable.")
    else:
        print("  no usable pair in either alphabet")

    print("\n  Report the chance baseline alongside every in-sequence rate. At 32")
    print("  symbols and load 48 the baseline is 0.78, so a raw rate of 1.000")
    print("  sounds dramatic and carries almost no information. That is a")
    print("  methodological point about swap-error analyses generally, not just")
    print("  about this experiment.")
elif RUN["swap"]:
    print("swap-error analysis is defined on the symbolic alphabet; skipped for "
          f"MODALITY={MODALITY}")

## §7 Variance share is biased relative to causal necessity

A methodological finding, not just a correction: three claims retracted during
this project were read off the share rather than the ablations.

In [ ]:
if RUN["sharebias"]:
    # Every file produced by this notebook is recomputed both ways, so the S7
    # table is generated rather than transcribed. PAPER_LABEL maps the file
    # names to the labels used in the text; earlier drafts named files from an
    # older layout, which is the mismatch this fixes.
    PAPER_LABEL = {
        "p2_boundary": "boundary (S4.2)",
        "p3_costforms": "cost forms (S4.4)",
        "p3_availability": "availability (S4.4)",
        "p4_width": "decoupled widths (S4.5)",
        "p6_replication": "tasks x architectures (S4.7)",
        "p6_cores": "recurrent cores (S4.7)",
        "p7_schedule": "schedule (S4.7)",
        "p7_equilibrium": "equilibrium (S4.7)",
        "p14_lamstar": "price axis (S6)",
        "p1_free": "free retrieval (S4.1)",
    }

    def share_crossing(x, share, thr=0.5):
        o = np.argsort(x); x, y = np.asarray(x, float)[o], np.asarray(share, float)[o]
        for i in range(len(y) - 1):
            a, b = y[i] - thr, y[i + 1] - thr
            if a == 0:
                return float(x[i])
            if a * b < 0:
                f = abs(a) / (abs(a) + abs(b))
                return float(x[i] + f * (x[i + 1] - x[i]))
        return np.nan

    summ = []
    for fn in sorted(glob.glob(str(RESDIR / f"*__{TAG}.csv"))):
        stem = os.path.basename(fn).split("__")[0]
        if "_traces" in fn or stem.endswith("_crossings"):
            continue
        df = pd.read_csv(fn)
        if not {"hidden_dep", "attn_dep", "context_share"} <= set(df.columns):
            continue
        if "step" in df.columns:
            df = df[df.step == df.step.max()]
        axis = "load" if "load" in df and df.load.nunique() >= 3 else "lam"
        if axis not in df or df[axis].nunique() < 3:
            continue
        keys = [c for c in ["lam", "task", "model", "form", "p_drop", "d_h", "d_c"]
                if c in df and c != axis and df[c].nunique() > 1]
        rows = []
        for k, g in (df.groupby(keys) if keys else [((), df)]):
            gm = g.groupby(axis)[["hidden_dep", "attn_dep", "context_share"]].mean().reset_index()
            rows.append({"abl": crossing_any(gm[axis], gm.hidden_dep, gm.attn_dep),
                         "share": share_crossing(gm[axis], gm.context_share)})
        O_ = pd.DataFrame(rows).dropna()
        if len(O_) > 1:
            d = O_.abl - O_.share
            summ.append({"result": PAPER_LABEL.get(stem, stem), "file": stem,
                         "swept axis": axis, "n": len(O_),
                         "mean (abl - share)": d.mean(), "max abs": d.abs().max()})
    SB = pd.DataFrame(summ)
    if len(SB):
        SB = SB.sort_values("mean (abl - share)")
        SB.to_csv(csv("p13_share_bias"), index=False)
        print(SB.round(3).to_string(index=False))
        lo = SB[SB["swept axis"] == "load"]
        if len(lo):
            print(f"\n  pooled mean (ablation - share) on load axes: "
                  f"{lo['mean (abl - share)'].mean():+.2f} loads over {len(lo)} files")
            print("  Consistently negative: where the routes are equally necessary,")
            print("  retrieval still contributes less than half the logit variance.")
    else:
        print("  no files with both measures yet — run the sweeps first")


## Figures and LaTeX tables

Reads only CSVs, so it can be rerun while the writing changes. Figures are
written as PDF (vector, for LaTeX) and PNG; tables as `booktabs` fragments.
Anything found for other modalities is merged into the cross-modality figure.

In [ ]:
if RUN["figures"]:
    mpl.rcParams.update({
        "figure.dpi": 130, "savefig.dpi": 300, "savefig.bbox": "tight",
        "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8.5,
        "legend.fontsize": 7, "xtick.labelsize": 7, "ytick.labelsize": 7,
        "axes.spines.top": False, "axes.spines.right": False,
        "lines.linewidth": 1.4, "lines.markersize": 3.5,
        "figure.constrained_layout.use": True})
    C_REC, C_ATT, C_ACC = "#1b6ca8", "#d1495b", "#8d99ae"
    ONE, TWO = 3.35, 6.9

    def savefig(fig, name):
        for ext in ("pdf", "png"):
            fig.savefig(os.path.join(FIGDIR, f"{name}_{TAG}.{ext}"))
        plt.show(); print(f"  {FIGDIR}/{name}_{TAG}.pdf")

    def rd(fn):
        return pd.read_csv(fn) if os.path.exists(fn) else None

    def latex(df, name, caption, label, ff="%.3f"):
        body = df.to_latex(index=False, escape=False, float_format=ff,
                           column_format="l" + "r" * (df.shape[1] - 1))
        with open(os.path.join(TABDIR, f"{name}_{TAG}.tex"), "w") as f:
            f.write("\\begin{table}[t]\n\\centering\n\\small\n"
                    f"\\caption{{{caption}}}\n\\label{{{label}}}\n{body}\\end{{table}}\n")
        print(f"\n--- {name} ---"); print(df.to_string(index=False))

    B = rd(csv("p2_boundary"))
    if B is not None:
        fig, ax = plt.subplots(1, 2, figsize=(TWO, 2.5))
        p = B.pivot_table(index="lam", columns="load", values="context_share")
        im = ax[0].imshow(p.values, aspect="auto", origin="lower", vmin=0, vmax=1, cmap="RdBu_r")
        ax[0].set_xticks(range(len(p.columns)), p.columns)
        ax[0].set_yticks(range(len(p.index)), [f"{v:g}" for v in p.index])
        ax[0].set_xlabel("memory load"); ax[0].set_ylabel(r"retrieval price $\lambda$")
        ax[0].set_title("retrieval share of readout"); fig.colorbar(im, ax=ax[0], fraction=.046)
        a = B.pivot_table(index="lam", columns="load", values="acc_clean")
        lo = min(0.99, float(a.values.min()))
        im2 = ax[1].imshow(a.values, aspect="auto", origin="lower", vmin=lo, vmax=1.0, cmap="viridis")
        ax[1].set_xticks(range(len(a.columns)), a.columns)
        ax[1].set_yticks(range(len(a.index)), [f"{v:g}" for v in a.index])
        ax[1].set_xlabel("memory load"); ax[1].set_title(f"accuracy ({lo:.3f}–1.000)")
        fig.colorbar(im2, ax=ax[1], fraction=.046)
        savefig(fig, "fig1_boundary")

    T = rd(csv("p8_inversion").replace(".csv", "_traces.csv"))
    if T is not None:
        picks = [(l, L) for l, L in [(0.01, 8), (0.03, 8), (0.1, 12), (0.01, 16)]
                 if ((T.lam == l) & (T.load == L)).any()]
        fig, axes = plt.subplots(1, len(picks), figsize=(TWO, 2.2), sharey=True)
        for ax, (lam, L) in zip(np.atleast_1d(axes), picks):
            g = T[(T.lam == lam) & (T.load == L)]
            for col, lbl, cc in [("hidden_dep", "ablate recurrence", C_REC),
                                 ("attn_dep", "ablate attention", C_ATT)]:
                s = g.groupby("step")[col].agg(["mean", "std"])
                ax.plot(s.index, s["mean"], "-o", color=cc, label=lbl)
                ax.fill_between(s.index, s["mean"]-s["std"], s["mean"]+s["std"],
                                color=cc, alpha=.18, lw=0)
            s = g.groupby("step").acc_clean.mean()
            ax.plot(s.index, s.values, ":", color=C_ACC, lw=1.6, label="accuracy")
            ax.set_ylim(-.05, 1.08); ax.set_xlabel("training step")
            ax.set_title(rf"$\lambda$={lam}, load={L}" +
                         ("  (control)" if (lam, L) == (0.01, 16) else ""))
        np.atleast_1d(axes)[0].set_ylabel("accuracy lost when ablated")
        np.atleast_1d(axes)[0].legend(loc="center right", frameon=False)
        savefig(fig, "fig2_inversion")

    CF, AV = rd(csv("p3_costforms")), rd(csv("p3_availability"))
    panels = []
    if CF is not None:
        for f in [x for x in ["l2", "l2sq", "gate"] if x in set(CF.form)]:
            panels.append((f, CF[CF.form == f], "lam", r"$\lambda/\lambda_{\max}$"))
    if AV is not None:
        panels.append(("availability (no penalty)", AV, "p_drop", "p(retrieval absent)"))
    if panels:
        fig, axes = plt.subplots(1, len(panels), figsize=(TWO, 2.0), sharey=True)
        cm = plt.cm.viridis
        for ax, (title, g, xc, xl) in zip(np.atleast_1d(axes), panels):
            loads = sorted(g.load.unique())
            for i, L in enumerate(loads):
                s = g[g.load == L].groupby(xc).context_share.mean()
                xs = s.index / max(s.index.max(), 1e-9) if xc == "lam" else s.index
                ax.plot(xs, s.values, "-o", color=cm(i / max(len(loads)-1, 1)), label=f"{int(L)}")
            ax.set_xlabel(xl); ax.set_title(title)
        np.atleast_1d(axes)[0].set_ylabel("retrieval share")
        np.atleast_1d(axes)[-1].legend(title="load", frameon=False, ncol=2)
        savefig(fig, "fig3_costforms")

    W = rd(csv("p4_width_crossings"))
    if W is not None and "crossing" in W:
        fig, ax = plt.subplots(1, 2, figsize=(TWO*.72, 2.2), sharey=True)
        for a_, (fix, var, lbl) in zip(ax, [("d_h", "d_c", r"retrieval width $d_c$"),
                                            ("d_c", "d_h", r"storage width $d_h$")]):
            sub = W[W[fix] == 52].dropna(subset=["crossing"])
            if not len(sub):
                a_.set_axis_off(); continue
            s = sub.groupby(var).crossing.agg(["mean", "std"])
            a_.errorbar(s.index, s["mean"], yerr=s["std"], fmt="-o", capsize=2.5,
                        color=C_REC if var == "d_h" else C_ATT)
            a_.set_xscale("log"); a_.set_yscale("log"); a_.set_xlabel(lbl)
        ax[0].set_ylabel(r"$\lambda$ at crossing"); savefig(fig, "fig4_width")

    Z = rd(csv("p9_switch_traces"))
    if Z is not None:
        cols = {"never": "#2a9d8f", "withdrawn": "#e76f51",
                "maintained": "#264653", "late": "#e9c46a"}
        loads = sorted(Z.load.unique())
        fig, axes = plt.subplots(1, len(loads), figsize=(TWO*.8, 2.2), sharey=True, squeeze=False)
        for ax, L in zip(axes[0], loads):
            for arm in ["never", "withdrawn", "maintained", "late"]:
                g = Z[(Z.load == L) & (Z.arm == arm)]
                if not len(g):
                    continue
                s = g.groupby("step").context_share.agg(["mean", "std"])
                ax.plot(s.index, s["mean"], color=cols[arm], label=arm)
                ax.fill_between(s.index, s["mean"]-s["std"], s["mean"]+s["std"],
                                color=cols[arm], alpha=.15, lw=0)
            ax.axvline(2000, color="gray", ls="--", lw=.9)
            ax.set_xlabel("training step"); ax.set_title(f"load {int(L)}")
        axes[0][0].set_ylabel("retrieval share"); axes[0][0].legend(frameon=False)
        savefig(fig, "fig5_switch")

    O = rd(csv("p5_ood"))
    if O is not None:
        loads = sorted(O.load.unique())
        fig, axs = plt.subplots(len(loads), 3, figsize=(TWO, 1.7*len(loads)),
                                sharey=True, squeeze=False)
        for i, L in enumerate(loads):
            for j, axn in enumerate(["delay", "interference", "noise"]):
                ax = axs[i][j]
                g = O[(O.load == L) & (O.axis == axn) & O.arm.isin(["retrieval", "recurrent"])]
                for arm, c in [("retrieval", C_ATT), ("recurrent", C_REC)]:
                    s = g[g.arm == arm].groupby("level").acc.agg(["mean", "std"])
                    ax.errorbar(s.index, s["mean"], yerr=s["std"], fmt="-o",
                                capsize=2, color=c, label=arm)
                ax.set_ylim(0, 1.05)
                if i == len(loads)-1: ax.set_xlabel(axn)
                if j == 0: ax.set_ylabel(f"load {int(L)}\naccuracy")
                if i == 0 and j == 0: ax.legend(frameon=False)
        savefig(fig, "fig6_ood")

    S = rd(csv("p10_readout_path"))
    if S is not None:
        t = S.groupby("step")[["ctx", "pre", "post", "lc", "dead", "w_c"]].mean()
        fig, ax = plt.subplots(1, 2, figsize=(TWO*.8, 2.2))
        for c, lbl in [("ctx", "context vector"), ("pre", "pre-activation"),
                       ("post", "post-activation"), ("lc", "readout contribution")]:
            ax[0].plot(t.index, t[c]/max(t[c].iloc[0], 1e-12), label=lbl)
        ax[0].set_yscale("log"); ax[0].set_xlabel("training step")
        ax[0].set_ylabel("norm, relative to start"); ax[0].legend(frameon=False)
        ax[1].plot(t.index, t.dead, color=C_ATT, label="fraction in dead zone")
        ax[1].plot(t.index, t.w_c/max(t.w_c.iloc[0], 1e-12), color=C_REC, label=r"$\|W_c\|$ rel.")
        ax[1].set_xlabel("training step"); ax[1].legend(frameon=False)
        savefig(fig, "fig7_silencing")

    # cross-modality: merges any p2_boundary__*.csv found
    files = sorted(glob.glob(str(RESDIR / "p2_boundary__*.csv")))
    if len(files) > 1:
        fig, ax = plt.subplots(figsize=(ONE, 2.3))
        for fn, mk in zip(files, ["o", "s", "^", "v"]):
            lbl = os.path.basename(fn).split("__")[1].replace(".csv", "")
            C = crossings_by_seed(pd.read_csv(fn), "load", ["lam"], acc_floor=ACC_FLOOR).dropna()
            if not len(C): continue
            s = C.groupby("lam").crossing.agg(["mean", "std"]); s = s[s.index > 0]
            ax.errorbar(np.log2(s.index), s["mean"], yerr=s["std"], fmt=f"-{mk}",
                        capsize=2.5, label=lbl)
        ax.set_xlabel(r"$\log_2\lambda$"); ax.set_ylabel("load at crossing")
        ax.legend(frameon=False); savefig(fig, "fig8_modalities")

    # tables
    rows = []
    for fn in sorted(glob.glob(str(RESDIR / "p2_boundary__*.csv"))):
        lbl = os.path.basename(fn).split("__")[1].replace(".csv", "")
        C = crossings_by_seed(pd.read_csv(fn), "load", ["lam"], acc_floor=ACC_FLOOR)
        for lam, g in C.groupby("lam"):
            if lam == 0: continue
            mu, lo, hi = boot_ci(g.crossing.values)
            rows.append({"stimuli": lbl, r"$\lambda$": lam, "crossing": mu,
                         "CI low": lo, "CI high": hi,
                         "n": f"{int(g.crossing.notna().sum())}/{len(g)}"})
    if rows:
        latex(pd.DataFrame(rows), "tab1_crossings",
              "Load at which attention dependence overtakes recurrent dependence. "
              "Crossings computed per seed and bootstrapped (10{,}000 resamples).",
              "tab:crossings")

    D5 = rd(csv("p5_dissociation"))
    if D5 is not None:
        d = D5.copy()
        d["retrieval"] = d.apply(lambda r: f"{r.retrieval:.3f}$\\pm${r.retrieval_sd:.3f}", axis=1)
        d["recurrent"] = d.apply(lambda r: f"{r.recurrent:.3f}$\\pm${r.recurrent_sd:.3f}", axis=1)
        d["separated"] = d.separated.map({True: "yes", False: "no"})
        latex(d[["load", "axis", "retrieval", "recurrent", "advantage", "separated"]],
              "tab2_ood",
              "Out-of-distribution accuracy at the hardest level of each axis, for "
              "pairs matched in-distribution. Positive favours retrieval.", "tab:ood")

    print(f"\nfigures in ./{FIGDIR}, LaTeX fragments in ./{TABDIR}")
    print(f"Rerun with MODALITY set to another value to add it to fig8_modalities.")

## Figures and tables for the added blocks


In [ ]:
if RUN["figures"]:
    # figures for the blocks added after the first draft
    def rd2(fn):
        p = csv(fn)
        return pd.read_csv(p) if os.path.exists(p) else None

    LS = rd2("p14_lamstar")
    if LS is not None:
        fig, ax = plt.subplots(1, 3, figsize=(TWO, 2.3))
        loads = sorted(LS.load.unique()); cm = plt.cm.viridis
        for i, L in enumerate(loads):
            g = LS[(LS.load == L) & (LS.lam > 0)]
            s = g.groupby("lam")[["hidden_dep", "attn_dep", "acc_clean"]].mean()
            col = cm(i / max(len(loads) - 1, 1))
            ax[0].plot(s.index, s.attn_dep - s.hidden_dep, "-o", color=col, label=f"{int(L)}")
            ax[2].plot(s.index, s.acc_clean, "-o", color=col)
        ax[0].axhline(0, color="k", lw=.8, ls="--")
        ax[0].set_xscale("log"); ax[0].set_xlabel(r"$\lambda$")
        ax[0].set_ylabel("attn dep $-$ hidden dep")
        ax[0].set_title("crossing on the price axis")
        ax[0].legend(title="load", frameon=False, ncol=2, fontsize=6)
        C = rd2("p14_lamstar_crossings")
        if C is not None and "crossing" in C:
            s = C.dropna().groupby("load").crossing.agg(["mean", "std"])
            ax[1].errorbar(s.index, s["mean"], yerr=s["std"], fmt="-o", capsize=2.5,
                           color=C_REC)
            ax[1].set_yscale("log")
        ax[1].set_xlabel("memory load"); ax[1].set_ylabel(r"$\lambda^*$")
        ax[1].set_title(r"$\lambda^*(L)$, measured directly")
        ax[2].axhline(0.9, color="gray", lw=.8, ls=":")
        ax[2].set_xscale("log"); ax[2].set_xlabel(r"$\lambda$")
        ax[2].set_ylabel("accuracy"); ax[2].set_title("collapse, not boundary")
        savefig(fig, "fig9_lamstar")

    SW = rd2("p15_swap")
    if SW is not None and len(SW):
        fig, ax = plt.subplots(figsize=(ONE * 1.25, 2.3))
        for arm, c in [("recurrent", C_REC), ("retrieval", C_ATT)]:
            g = SW[SW.arm == arm].groupby("load")[["excess", "err_rate"]].agg(["mean", "std"])
            if not len(g):
                continue
            ax.errorbar(g[("err_rate", "mean")], g[("excess", "mean")],
                        yerr=g[("excess", "std")], fmt="o", capsize=2.5, color=c, label=arm)
            for L, r in g.iterrows():
                ax.annotate(f"L={int(L)}", (r[("err_rate", "mean")], r[("excess", "mean")]),
                            fontsize=6, xytext=(3, 3), textcoords="offset points")
        ax.axhline(0, color="k", lw=.8, ls="--")
        ax.set_xlabel("error rate"); ax.set_ylabel("excess sequence-item errors over chance")
        ax.set_title("swap signature does not dissociate the algorithms")
        ax.legend(frameon=False)
        savefig(fig, "fig10_swap")

    CAP = rd2("p0_capacity")
    if CAP is not None:
        latex(CAP, "tab5_capacity",
              "Width search placing every architecture near the reference "
              "attention-free GRU parameter count.", "tab:capacity", ff="%.0f")

    SBt = rd2("p13_share_bias")
    if SBt is not None:
        latex(SBt.drop(columns=[c for c in ["file"] if c in SBt.columns]),
              "tab4_sharebias",
              "Crossings computed from ablations against the variance-share "
              "estimate, per result file.", "tab:sharebias")

    SWt = rd2("p15_swap")
    if SWt is not None and len(SWt):
        t = (SWt.groupby(["arm", "load"])[["acc_clean", "err_rate", "in_seq_rate",
                                           "chance", "excess"]].mean().reset_index())
        latex(t, "tab6_swap",
              "Error composition for the two configurations at comparable error "
              "rates. Both show a large excess of sequence items over chance.",
              "tab:swap")


## Derived quantities

Numbers the paper quotes that are functions of the sweeps rather than direct
outputs: the per-seed boundary fit with a spread on its coefficients, the
cost-form-specific cliff, the width fold-ranges, the Δ-versus-λ* comparison, the
power fit in §6.2 and its failure at high load, and the schedule ratio on
matched grids.

Computing them here rather than by hand is the point: no figure in the text
should be transcribed from a printed table.


In [ ]:
if RUN["derived"]:
    # Numbers the paper quotes that are functions of the sweeps rather than
    # outputs of them. Computing them here rather than by hand means no figure
    # in the text is transcribed from a printed table.
    def rdc(fn):
        p = csv(fn)
        return pd.read_csv(p) if os.path.exists(p) else None

    derived = {}

    # --- S4.2 boundary fit, per seed, with a spread on the coefficients -------
    B = rdc("p2_boundary")
    if B is not None:
        C = crossings_by_seed(B, "load", ["lam"], acc_floor=ACC_FLOOR).dropna()
        FIT = pd.DataFrame([
            dict(zip(("seed", "intercept", "slope", "r2"),
                     (sd, *boundary_fit(g.set_index("lam").crossing.to_dict()))))
            for sd, g in C.groupby("seed")])
        derived["boundary_fit"] = FIT
        print("S4.2 boundary fit, per seed  (L* = a + b*log2 lambda):")
        print(FIT.round(3).to_string(index=False))
        print(f"  a = {FIT.intercept.mean():.2f} +/- {FIT.intercept.std():.2f}   "
              f"b = {FIT.slope.mean():.2f} +/- {FIT.slope.std():.2f}   "
              f"R^2 = {FIT.r2.mean():.3f}")
        print(f"  minimum accuracy over {len(B)} runs: {B.acc_clean.min():.4f}")

    # --- S4.4 is the boundary invariant to cost form? ------------------------
    # It is not, and the pairwise test below is what establishes that. The
    # earlier claim of invariance came from eyeballing 1.46 against 1.61 without
    # a spread; with per-seed fits and bootstrap intervals, four of six pairs
    # separate. The defensible claim is that the boundary is log-linear in every
    # cost formulation, with form-dependent coefficients — the functional form
    # is the finding, the coefficients are not.
    CFd = rdc("p3_costforms")
    FITS = [("boundary (S4.2)", B)]
    if CFd is not None:
        FITS += [(f"cost form: {f}", g) for f, g in CFd.groupby("form")]
    rows = []
    for _name, _df in FITS:
        if _df is None or "lam" not in _df:
            continue
        Cs = crossings_by_seed(_df, "load", ["lam"], acc_floor=ACC_FLOOR).dropna()
        for sd, g in Cs.groupby("seed"):
            a, b, r2 = boundary_fit(g.set_index("lam").crossing.to_dict())
            if b == b:
                rows.append({"source": _name, "seed": sd, "intercept": a,
                             "slope": b, "r2": r2})
    FIT2 = pd.DataFrame(rows)
    if len(FIT2):
        derived["form_fits"] = FIT2
        srows = []
        for s, g in FIT2.groupby("source"):
            mb, lb, hb = boot_ci(g.slope.values)
            ma, la, ha = boot_ci(g.intercept.values)
            srows.append({"source": s, "n seeds": len(g), "slope": mb,
                          "slope lo": lb, "slope hi": hb, "intercept": ma,
                          "int lo": la, "int hi": ha, "R^2": g.r2.mean()})
        SF = pd.DataFrame(srows)
        derived["form_fit_ci"] = SF
        print("\nS4.4 per-seed fits, load* = a + b log2(lambda):")
        print(SF.round(3).to_string(index=False))
        print("\n  pairwise: do the slope CIs overlap?")
        n_sep = 0
        for i_ in range(len(SF)):
            for j_ in range(i_ + 1, len(SF)):
                a_, b_ = SF.iloc[i_], SF.iloc[j_]
                ov = (a_["slope lo"] <= b_["slope hi"]) and (b_["slope lo"] <= a_["slope hi"])
                n_sep += (not ov)
                print(f"    {a_['source'][:26]:<26} vs {b_['source'][:26]:<26} "
                      f"{'overlap' if ov else 'SEPARATE'}   ({a_.slope:.2f} vs {b_.slope:.2f})")
        n_pair = len(SF) * (len(SF) - 1) // 2
        print(f"\n  {n_sep}/{n_pair} pairs separate. The boundary is NOT invariant to")
        print("  cost form. Write it as: log-linear in every formulation, with")
        print("  form-dependent coefficients. Note also that the gate's lambda is")
        print("  on a different scale (it prices a bounded gate value, the others")
        print("  an unbounded norm), so its intercept is not comparable and only")
        print("  the slope — loads per doubling — is.")
        print("\n  Availability is excluded: its axis is a drop probability, not a")
        print("  price, so a log2(lambda) slope is not defined for it.")

    # --- S4.4 the cliff IS cost-form specific --------------------------------
    if CFd is not None:
        rows = []
        for f, g in CFd.groupby("form"):
            for L, gg in g.groupby("load"):
                s = gg.groupby("lam").acc_clean.mean().sort_index()
                s = s[s.index > 0]
                if len(s) < 2:
                    continue
                dl = np.diff(np.log2(s.index.values))
                drop = -np.diff(s.values) / np.maximum(dl, 1e-9)
                rows.append({"form": f, "load": L,
                             "max_drop_per_log2lam": float(drop.max())})
        CLIFF = pd.DataFrame(rows)
        derived["cliff"] = CLIFF
        print("\nS4.4 sharpness — maximum accuracy drop per doubling of lambda:")
        print(CLIFF.pivot_table(index="load", columns="form",
                                values="max_drop_per_log2lam").round(3).to_string())
        nz = CFd[CFd.context_share > 0].groupby("form").context_share.min()
        print("\n  smallest nonzero retrieval share by cost form:")
        print(nz.apply(lambda v: f"{v:.2e}").to_string())
        print("  Both the boundary coefficients and the accuracy cliff depend on")
        print("  the cost form. What survives across all four is the log-linear")
        print("  relation itself.")

    # --- S4.5 width dissociation ---------------------------------------------
    W = rdc("p4_width_crossings")
    if W is not None and "crossing" in W:
        sub = W.dropna(subset=["crossing"])
        a = sub[sub.d_h == 52].groupby("d_c").crossing.mean()
        b = sub[sub.d_c == 52].groupby("d_h").crossing.mean()
        if len(a) > 1 and len(b) > 1:
            fa, fb = a.max() / a.min(), b.max() / b.min()
            print(f"\nS4.5 varying retrieval width d_c: {fa:.2f}x change in lambda*")
            print(f"     varying storage   width d_h: {fb:.2f}x change in lambda*")
            print(f"     ratio {fb/fa:.1f}x — storage capacity sets the boundary")
            derived["width_folds"] = pd.DataFrame(
                [{"axis": "retrieval (d_c)", "fold": fa},
                 {"axis": "storage (d_h)", "fold": fb}])

    # --- lambda*(load), measured on the price axis ---------------------------
    LSC = rdc("p14_lamstar_crossings")
    lamstar = None
    if LSC is not None and "crossing" in LSC:
        lamstar = LSC.dropna().groupby("load").crossing.mean()
        print("\nlambda*(load), measured directly on the price axis:")
        print(lamstar.round(4).to_string())

    # --- S6.1 does Delta predict lambda*? ------------------------------------
    D = rdc("p12_delta")
    if D is not None:
        print("\nS6.1 Delta by budget:")
        print(D.pivot_table(index="load", columns="budget",
                            values="delta").round(3).to_string())
        if lamstar is not None:
            j = (D.groupby("load").delta.mean().to_frame("delta")
                  .join(lamstar.to_frame("lam_star"), how="inner").dropna())
            print("\n  Delta against lambda* where both exist:")
            print(j.round(4).to_string())
            if len(j) > 2:
                print(f"  correlation {j.delta.corr(j.lam_star):+.2f} over {len(j)} loads")
            else:
                # a correlation on two points is 1.00 by construction; printing it
                # invites a reviewer to stop reading at that number
                rat_d = j.delta.max() / j.delta.min()
                rat_l = j.lam_star.max() / j.lam_star.min()
                print(f"  only {len(j)} loads have both; no correlation is reported.")
                print(f"  Over that range Delta changes {rat_d:.1f}x while lambda* "
                      f"changes {rat_l:.1f}x — the ordering agrees, the scale does")
                print("  not, which is the sense in which Delta fails as a predictor.")
                print("  Add loads to neg_delta if a correlation is wanted.")

    # --- S6.2 optimisation cost as a predictor -------------------------------
    OPT = rdc("p12_optcost")
    if OPT is not None and lamstar is not None:
        s90 = OPT.dropna().groupby("load").steps_to_90.mean()
        j = s90.to_frame("steps").join(lamstar.to_frame("lam_star"), how="inner").dropna()
        fit_loads = [L for L in [2, 4, 6] if L in j.index]
        if len(fit_loads) >= 2 and len(j) > len(fit_loads):
            sub = j.loc[fit_loads]
            k, c = np.polyfit(np.log(sub.steps), np.log(sub.lam_star), 1)
            j["predicted"] = np.exp(c) * j.steps ** k
            j["error"] = j.predicted / j.lam_star - 1
            print(f"\nS6.2 power fit on loads {fit_loads}: lambda* ~ steps^{k:.2f}")
            print(j.round(4).to_string())
            held = j.drop(index=fit_loads)
            if len(held):
                print("  held-out loads, over-prediction:")
                for L, r in held.iterrows():
                    print(f"    load {int(L):>2}: predicted {r.predicted:.4f} vs "
                          f"measured {r.lam_star:.4f}  ({r.error:+.0%})")
                print("  The fit does not fail flatly; it degrades systematically")
                print("  with extrapolation distance. Report it that way — a")
                print("  predictor that is 20% out at one load and 50% at the next")
                print("  is more informative than one that is simply wrong.")
            derived["optcost"] = j.reset_index()

    # --- S4.7 schedule artefact ----------------------------------------------
    SCH = rdc("p7_schedule")
    if SCH is not None:
        cr = {}
        for s, g in SCH.groupby("sched"):
            Cc = crossings_by_seed(g, "load", ["lam"], acc_floor=ACC_FLOOR).dropna()
            cr[s] = Cc.groupby("lam").crossing.mean()
        if len(cr) == 2:
            w = pd.DataFrame(cr).dropna()
            r = w["constant"] / w["cosine"]
            print(f"\nS4.7 crossing ratio constant/cosine: {r.mean():.2f}x "
                  f"(range {r.min():.2f}-{r.max():.2f}, matched load grids)")
            print("  The ratio is not uniform: larger at low price, converging at")
            print("  high price, which is what cosine decay freezing the drift")
            print("  partway would produce. The per-seed slope intervals for the")
            print("  two schedules overlap, so report the shift, not a difference")
            print("  in coefficients.")
            derived["schedule_ratio"] = w.assign(ratio=r).reset_index()

    # --- S5.1 the inversion, measured from convergence -----------------------
    # The claim is that route dependence inverts AFTER behaviour converges, so
    # the "before" reading has to be taken at convergence. Taken at the first
    # logged step the model is still at chance, the dependence denominator is
    # its headroom above chance, and dividing by ~0 gives seed SDs of 0.3-0.55
    # and spurious "no inversion" verdicts.
    _tp = csv("p8_inversion").replace(".csv", "_traces.csv")
    T = pd.read_csv(_tp) if os.path.exists(_tp) else None
    if T is not None:
        CONV, EVAL_TOL = 0.99, 0.97
        print(f"\nS5.1 route dependence from the first step at >={CONV} accuracy:")
        rows = []
        for (lam, L), g in T.groupby(["lam", "load"]):
            curve = g.groupby("step").acc_clean.mean()
            hit = curve.index[curve >= CONV]
            if not len(hit):
                print(f"  lam={lam:<5} L={int(L):>2}  never converged "
                      f"(peak {curve.max():.3f})")
                continue
            s0, s1 = int(hit.min()), int(g.step.max())
            a_, b_ = g[g.step == s0], g[g.step == s1]
            win = g[g.step >= s0]
            inv = (a_.attn_dep.mean() > a_.hidden_dep.mean()) and \
                  (b_.hidden_dep.mean() > b_.attn_dep.mean())
            n_seed_inv = 0
            for _, gs in g[g.step >= s0].groupby("seed"):
                gs = gs.sort_values("step")
                n_seed_inv += int(((gs.hidden_dep - gs.attn_dep) > 0).any())
            rows.append({"lam": lam, "load": L, "converged_step": s0,
                         "window": s1 - s0,
                         "hidden_0": a_.hidden_dep.mean(), "hidden_1": b_.hidden_dep.mean(),
                         "attn_0": a_.attn_dep.mean(), "attn_1": b_.attn_dep.mean(),
                         "acc_mean": win.acc_clean.mean(), "acc_min": win.acc_clean.min(),
                         "seeds_inverted": n_seed_inv, "n_seeds": g.seed.nunique(),
                         "inverted": inv})
            print(f"  lam={lam:<5} L={int(L):>2}  converged {s0:>5}, +{s1-s0:>5} steps  "
                  f"hidden {a_.hidden_dep.mean():.3f}->{b_.hidden_dep.mean():.3f}  "
                  f"attn {a_.attn_dep.mean():.3f}->{b_.attn_dep.mean():.3f}  "
                  f"acc {win.acc_clean.mean():.4f}+/-{win.acc_clean.std():.4f}  "
                  f"{n_seed_inv}/{g.seed.nunique()} seeds"
                  f"{'   INVERTS' if inv else ''}")
        if rows:
            IV = pd.DataFrame(rows)
            derived["inversion"] = IV
            print(f"\n  behaviour flat across the window (min >= {EVAL_TOL}): "
                  f"{int((IV.acc_min >= EVAL_TOL).sum())}/{len(IV)} conditions")
            print("  The tolerance accommodates evaluation noise: each accuracy is")
            print("  ~1000 trials, so 0.98 is about twenty errors, not drift.")
            print("  The high-load condition at the same price is the control; it")
            print("  should show 0 seeds inverting while the others show all.")

    for k, v in derived.items():
        v.to_csv(csv(f"d_{k}"), index=False)
    print(f"\nderived quantities written to {RESDIR}/d_*__{TAG}.csv")

## Appendix A: reproducibility, generated

Seed counts, run counts and measured compute come from the results store rather
than a hand-maintained table, so they cannot drift from what was actually run.
This also writes `requirements.txt`, an environment record, and a manifest of
every artefact on disk.

Blocks that finished with fewer than three seeds are flagged explicitly.


In [ ]:
if RUN["appendix"]:
    # Appendix A is generated, not written: whatever is actually on disk when the
    # run finishes is what the appendix reports.
    REQS = f"""torch=={torch.__version__.split('+')[0]}
numpy=={np.__version__}
pandas=={pd.__version__}
matplotlib=={mpl.__version__}
"""
    if IS_VISUAL:
        try:
            import torchvision as _tv
            REQS += f"torchvision=={_tv.__version__.split('+')[0]}\n"
        except Exception:
            pass
    (WORK / "requirements.txt").write_text(REQS)

    env = {"python": platform.python_version(), "platform": platform.platform(),
           "torch": torch.__version__, "cuda": torch.version.cuda,
           "device": str(DEVICE),
           "gpu": (torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None),
           "modality": MODALITY, "preset": PRESET,
           "seeds_main": SEEDS_MAIN, "seeds_aux": SEEDS_AUX,
           "steps": STEPS, "long": LONG, "xlong": XLONG,
           "generated": time.strftime("%Y-%m-%d %H:%M:%S")}
    (WORK / f"environment__{TAG}.json").write_text(json.dumps(env, indent=2))

    # seed and compute accounting straight from the stores
    BLOCKS = [("free", "Free retrieval", "4.1"),
              ("control", "Positive control", "3.4"),
              ("boundary", "Boundary", "4.2-4.3"),
              ("costforms", "Cost formulations", "4.4"),
              ("availability", "Availability control", "4.4"),
              ("width", "Width dissociation", "4.5"),
              ("ood_pairs", "Matched pairs", "4.6"),
              ("ood_eval", "OOD dissociation", "4.6"),
              ("replication", "Tasks x architectures", "4.7"),
              ("cores", "Recurrent cores", "4.7"),
              ("schedule", "Schedule artefact", "4.7"),
              ("equilibrium", "Equilibrium boundary", "4.7"),
              ("inversion", "Causal inversion", "5.1"),
              ("switch", "Four-arm lambda switch", "5.2"),
              ("silencing", "Silencing mechanism", "5.3"),
              ("reversibility", "Reversibility", "5.4"),
              ("reversibility_tail", "Long-tail recovery", "5.4"),
              ("lamstar", "lambda*(load)", "6"),
              ("neg_delta", "Delta vs budget", "6.1"),
              ("neg_optcost", "Optimisation cost", "6.2"),
              ("neg_ce", "Ablation CE and derivative", "6.3-6.4"),
              ("neg_learnability", "Route learnability", "6.5"),
              ("swap", "Swap errors", "6.6"),
              ("mamba", "Reference Mamba core", "4.7")]

    rows, total_h = [], 0.0
    for blk, label, sec in BLOCKS:
        st = Store(blk)
        if not st.recs:
            continue
        df = st.frame()
        hrs = st.elapsed() / 3600
        total_h += hrs
        rows.append({"result": label, "section": sec, "block": blk,
                     "seeds": int(df.seed.nunique()) if "seed" in df else 0,
                     "runs": len(st.recs), "gpu hours": round(hrs, 2)})
    SEEDTAB = pd.DataFrame(rows)
    SEEDTAB.to_csv(csv("p16_seed_table"), index=False)
    print("seed and compute accounting (replaces the hand-maintained Table 7):")
    print(SEEDTAB.to_string(index=False))
    print(f"\ntotal measured compute for MODALITY={MODALITY}: {total_h:.1f} hours "
          f"on {env['gpu'] or env['device']}")
    thin = SEEDTAB[SEEDTAB.seeds < 3]
    if len(thin):
        print("\n  !! blocks with fewer than three seeds — do not submit these as claims:")
        print(thin[["result", "seeds"]].to_string(index=False))

    if RUN["figures"]:
        latex(SEEDTAB, "tab3_seeds",
              "Seed counts, run counts and measured compute per experiment.",
              "tab:seeds", ff="%.2f")

    # manifest of everything on disk
    man = ["# Appendix A: reproducibility artefacts", "",
           f"Generated {env['generated']} — modality `{MODALITY}`, preset `{PRESET}`.", "",
           "All experiments generate their data online, so there is no dataset to "
           "release for the symbolic modality and no train/test split to leak "
           "through. Every run is reproducible from its seed; the results store "
           "records the exact specification of each run alongside its outputs.", "",
           "## Environment", "", "```", json.dumps(env, indent=2), "```", "",
           f"Total measured compute for this modality: **{total_h:.1f} hours**.", "",
           "## Running it", "",
           "One notebook, top to bottom. `PRESET = \"smoke\"` runs every block on "
           "the full grid with tiny budgets as a pipeline check; `PRESET = \"full\"` "
           "reproduces the paper. `RESUME = True` means an interrupted session "
           "restarts where it stopped: finished runs live in `results/*.jsonl` and "
           "are skipped, and long runs checkpoint mid-flight into `checkpoints/`. "
           "Set `MODALITY` and rerun to add a stimulus type.", "",
           "## Files", ""]
    for pat, what in [("results/*.csv", "result tables"),
                      ("results/*.jsonl", "raw run records"),
                      ("figures/*.pdf", "figures"),
                      ("tables/*.tex", "LaTeX table fragments")]:
        fs = sorted(glob.glob(str(WORK / pat)))
        man.append(f"### {what} ({len(fs)})")
        man.append("")
        for f in fs:
            n = ""
            if f.endswith(".csv"):
                try:
                    n = f" — {len(pd.read_csv(f))} rows"
                except Exception:
                    pass
            man.append(f"- `{os.path.relpath(f, WORK)}`{n}")
        man.append("")
    (WORK / f"APPENDIX_A__{TAG}.md").write_text("\n".join(man))
    print(f"\nwrote {WORK}/APPENDIX_A__{TAG}.md, requirements.txt, "
          f"environment__{TAG}.json")


## §4.7 addendum: the real selective state-space block

The two SSM arms above are stand-ins — a diagonal linear recurrence, and that
recurrence plus input-dependent gating. This cell runs the actual Mamba block:
depthwise causal convolution, input-dependent discretisation step Δ, a full
`d_inner × d_state` state, and the gated branch.

It prefers the official `mamba-ssm` kernels and falls back to a reference
implementation of the S6 recurrence in plain PyTorch when those cannot be
installed. The fallback is the published recurrence with a sequential scan
instead of the fused kernel — slower, same function.

It is last because it is the one block with an external dependency, and nothing
else depends on it.


In [ ]:
if RUN["mamba"]:
    # S4.7 uses two stand-ins for a selective state-space core: a diagonal linear
    # recurrence (S4D/LRU) and that recurrence plus input-dependent gating. A
    # reviewer can reasonably ask whether the boundary survives the actual Mamba
    # block — depthwise causal convolution, input-dependent discretisation step,
    # a real (d_inner x d_state) state, and the gated branch — rather than a
    # sketch of its selectivity. This cell runs that.
    #
    # It prefers the official `mamba-ssm` kernels. Where those cannot be
    # installed (no CUDA, no compiler, no network) it falls back to a reference
    # implementation of the S6 recurrence written out below in plain PyTorch.
    # The fallback is mathematically the published block with a sequential scan
    # instead of the fused kernel: slower, same function.
    MAMBA_SRC = "fallback"
    try:
        from mamba_ssm import Mamba as _OfficialMamba
        MAMBA_SRC = "mamba-ssm"
    except Exception:
        try:
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "mamba-ssm", "causal-conv1d"], check=True, timeout=900)
            from mamba_ssm import Mamba as _OfficialMamba
            MAMBA_SRC = "mamba-ssm"
        except Exception as e:
            print(f"  mamba-ssm unavailable ({type(e).__name__}); using the "
                  f"reference implementation")

    class S6Reference(nn.Module):
        """Selective state-space block, reference implementation.

        h_t = exp(dt_t A) h_{t-1} + dt_t B_t x_t ;  y_t = C_t h_t + D x_t
        with dt, B and C produced from the input (selectivity), a depthwise
        causal convolution before the SSM, and a SiLU-gated skip branch.
        """

        def __init__(self, d_model, d_state=16, d_conv=4, expand=2, dt_rank=None):
            super().__init__()
            self.d_inner = expand * d_model
            self.d_state = d_state
            self.dt_rank = dt_rank or max(1, math.ceil(d_model / 16))
            self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)
            self.conv = nn.Conv1d(self.d_inner, self.d_inner, d_conv,
                                  groups=self.d_inner, padding=d_conv - 1)
            self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * d_state, bias=False)
            self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)
            A = torch.arange(1, d_state + 1, dtype=torch.float32)
            self.A_log = nn.Parameter(A.repeat(self.d_inner, 1).log())
            self.D = nn.Parameter(torch.ones(self.d_inner))
            self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
            with torch.no_grad():                     # standard dt initialisation
                self.dt_proj.bias.copy_(
                    torch.log(torch.expm1(torch.rand(self.d_inner) * 0.09 + 0.001)))

        def forward(self, u):
            B_, T_, _ = u.shape
            xz = self.in_proj(u)
            x, z = xz.chunk(2, dim=-1)
            x = self.conv(x.transpose(1, 2))[:, :, :T_].transpose(1, 2)
            x = F.silu(x)
            p = self.x_proj(x)
            dt, Bm, Cm = torch.split(p, [self.dt_rank, self.d_state, self.d_state], dim=-1)
            dt = F.softplus(self.dt_proj(dt))                       # (B,T,d_inner)
            A = -torch.exp(self.A_log)                              # (d_inner,d_state)
            dA = torch.exp(dt.unsqueeze(-1) * A)                    # (B,T,d_inner,d_state)
            dBx = dt.unsqueeze(-1) * Bm.unsqueeze(2) * x.unsqueeze(-1)
            h = x.new_zeros(B_, self.d_inner, self.d_state)
            ys = []
            for t in range(T_):
                h = dA[:, t] * h + dBx[:, t]
                ys.append(torch.einsum("bds,bs->bd", h, Cm[:, t]))
            y = torch.stack(ys, 1) + self.D * x
            return self.out_proj(y * F.silu(z))

    class MambaCore(nn.Module):
        """Adapts a Mamba block to the (H, h_T) interface the two-route readout
        expects, so nothing downstream changes."""

        def __init__(self, d, d_state=16, d_conv=4, expand=2):
            super().__init__()
            self.norm = nn.LayerNorm(d)
            if MAMBA_SRC == "mamba-ssm":
                self.block = _OfficialMamba(d_model=d, d_state=d_state,
                                            d_conv=d_conv, expand=expand)
            else:
                self.block = S6Reference(d, d_state=d_state, d_conv=d_conv,
                                         expand=expand)

        def forward(self, x):
            H = x + self.block(self.norm(x))     # residual, as in the Mamba stack
            return H, H[:, -1]

    # The official kernels require CUDA and a matching build. Importing them is
    # not proof they run here, so try one forward pass before committing to them.
    if MAMBA_SRC == "mamba-ssm":
        try:
            _probe = MambaCore(32).to(DEVICE)
            _probe(torch.zeros(2, 6, 32, device=DEVICE))
            del _probe
        except Exception as e:
            print(f"  mamba-ssm imported but does not run here ({type(e).__name__}: {e}); "
                  f"falling back to the reference implementation")
            MAMBA_SRC = "fallback"

    EXTRA_CORES["mamba"] = MambaCore
    CORE_OF["mamba_attn"] = "mamba"
    HAS_ATTN["mamba_attn"] = True

    # capacity-match against the same reference the other architectures use
    _target = count_params(TwoRoute(CFG, "gru", d_h=CFG.d_model, attention=False))
    DIMS["mamba_attn"] = min(range(8, 200, 4),
                             key=lambda d: abs(count_params(
                                 TwoRoute(CFG, "mamba", d_h=d, attention=True)) - _target))
    _mp = count_params(TwoRoute(CFG, "mamba", d_h=DIMS["mamba_attn"], attention=True))
    print(f"\nMamba core: {MAMBA_SRC}   d = {DIMS['mamba_attn']}   "
          f"{_mp:,} params against target {_target:,} ({_mp/_target:.2f}x)")

    conds = [{"model": "mamba_attn", "lam": l, "load": L}
             for l in [0.0, 0.005, 0.01, 0.02, 0.03, 0.05, 0.1]
             for L in [2, 4, 8, 12, 16]]
    MB = sweep("mamba", conds, SEEDS_AUX, STEPS, out=csv("p17_mamba"))
    print("\naccuracy — mamba_attn:")
    print(MB.pivot_table(index="lam", columns="load", values="acc_clean").round(3).to_string())
    print("\ncontext_share:")
    print(MB.pivot_table(index="lam", columns="load", values="context_share").round(3).to_string())
    CM = report_crossings(MB, "load", "S4.7 mamba_attn", acc_floor=ACC_FLOOR)
    CM.to_csv(csv("p17_mamba_crossings"), index=False)

    if RUN["figures"]:
        fig, ax = plt.subplots(figsize=(ONE * 1.3, 2.3))
        for fn, lbl, c in [("p6_cores", "linear / selective", C_ACC),
                           ("p17_mamba", f"Mamba ({MAMBA_SRC})", C_REC),
                           ("p2_boundary", "GRU + attention", C_ATT)]:
            p = csv(fn)
            if not os.path.exists(p):
                continue
            df = pd.read_csv(p)
            for mdl, g in (df.groupby("model") if "model" in df and df.model.nunique() > 1
                           else [(lbl, df)]):
                C = crossings_by_seed(g, "load", ["lam"], acc_floor=ACC_FLOOR).dropna()
                if not len(C):
                    continue
                s = C.groupby("lam").crossing.agg(["mean", "std"])
                s = s[s.index > 0]
                ax.errorbar(np.log2(s.index), s["mean"], yerr=s["std"], fmt="-o",
                            capsize=2.5, label=str(mdl))
        ax.set_xlabel(r"$\log_2\lambda$"); ax.set_ylabel("load at crossing")
        ax.set_title("the boundary across recurrent cores")
        ax.legend(frameon=False, fontsize=6)
        savefig(fig, "fig11_mamba")

    print("\n  A monotone boundary in the real selective-SSM block would say the")
    print("  phenomenon is a property of priced retrieval against a compressive")
    print("  recurrent state, not of the GRU or of the S4D sketch.")


In [ ]:
if RUN["reversibility"]:
    assert not IS_VISUAL, ("set MODALITY = 'abstract' and re-run the setup cell; "
                           "the existing reversibility store is tagged abstract "
                           "and running under another tag recomputes it")
    ON, OFF = S(12000, 4000, 400), S(10000, 3000, 300)
    st = Store("reversibility")
    for sd in SEEDS_MAIN:
        spec = {"block": "reversibility", "seed": sd, "on": ON, "off": OFF,
                "lam": 0.1, "load": 4}

        def _go(sd=sd):
            task = make_task("selective_recall")
            set_seed(sd); m = build("gru_attn", n_out=task.n_out).to(DEVICE)
            opt = torch.optim.AdamW(m.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
            rem, snaps = [0.30, 0.10, 0.03, 1e-3, 1e-5], {}
            for step in range(1, ON + 1):
                m.train(); b = task.generate(CFG.batch_size, 4); o = m(b["x"])
                loss = F.cross_entropy(o["logits"], b["y"]) + \
                    0.1 * o["parts"]["context"].norm(dim=-1).mean()
                opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(m.parameters(), CFG.grad_clip); opt.step()
                if step % 250 == 0:
                    sh = deps(m, task, 4, n_batches=4)["context_share"]
                    while rem and sh <= rem[0]:
                        thr = rem.pop(0)
                        snaps[thr] = (copy.deepcopy(m.state_dict()), sh, step)
            rows = []
            for thr, (state, sh, stp) in snaps.items():
                set_seed(sd + 900)
                m2 = build("gru_attn", n_out=task.n_out).to(DEVICE); m2.load_state_dict(state)
                o2 = torch.optim.AdamW(m2.parameters(), lr=CFG.lr,
                                       weight_decay=CFG.weight_decay)
                for _ in range(OFF):
                    m2.train(); b = task.generate(CFG.batch_size, 4)
                    loss = F.cross_entropy(m2(b["x"])["logits"], b["y"])
                    o2.zero_grad(set_to_none=True); loss.backward()
                    nn.utils.clip_grad_norm_(m2.parameters(), CFG.grad_clip); o2.step()
                fin = deps(m2, task, 4)
                rows.append({"threshold": thr, "share_snap": sh, "snap_step": stp,
                             "share_final": fin["context_share"],
                             "recovery": fin["context_share"] - sh,
                             "acc_final": fin["acc_clean"]})
                print(f"    [rev] s{sd} thr {thr:<8} {sh:.2e} -> {fin['context_share']:.4f}")
            return {"rows": rows}

        cached(st, spec, _go, label=f"seed {sd}")

    REV = st.frame(); REV.to_csv(csv("p11_reversibility"), index=False)
    if len(REV):
        agg = REV.groupby("threshold")[["share_snap", "share_final", "recovery",
                                        "acc_final"]].agg(["mean", "std", "count"])
        print("\nrecovery by suppression depth:")
        print(agg.round(4).to_string())
        thin = agg[("recovery", "count")] < 3
        if thin.any():
            print(f"\n  !! thresholds reached by fewer than three seeds: "
                  f"{list(agg.index[thin])}")
            print("     A depth only one or two seeds reach is not a claim; report")
            print("     it as the deepest observed, not as a population estimate.")
        print("\n  Recovery is monotone in depth and accuracy stays at ceiling at")
        print("  every threshold. The claim is graded reversibility: suppression")
        print("  reverses at every depth tested, at a rate that falls with depth.")
        print("  That follows from §5.3's mechanism — a projection driven into")
        print("  GELU's negative regime reopens, and the deeper it was driven the")
        print("  longer reopening takes.")

    # The long tail: suppressed eight orders of magnitude, then released. Two
    # seeds gave an endpoint SD of ±0.10 with the curve still climbing, which is
    # too thin for a claim the section rests on.
    TAIL = S(15000, 5000, 400)
    st2 = Store("reversibility_tail")
    for sd in SEEDS_MAIN:
        spec = {"block": "reversibility_tail", "seed": sd, "tail": TAIL, "load": 8}

        def _go(sd=sd):
            task = make_task("selective_recall")
            set_seed(sd); m = build("gru_attn", n_out=task.n_out).to(DEVICE)
            for lam in [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.86, 1.5]:
                m, _ = train(task, 8, lam=lam, seed=sd, steps=1500, model=m,
                             schedule="constant")
            d0 = deps(m, task, 8)
            print(f"    [tail] seed {sd}: after up-leg share {d0['context_share']:.2e} "
                  f"acc {d0['acc_clean']:.3f}")
            opt = torch.optim.AdamW(m.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
            rows = []
            for step in range(1, TAIL + 1):
                m.train(); b = task.generate(CFG.batch_size, 8)
                loss = F.cross_entropy(m(b["x"])["logits"], b["y"])
                opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(m.parameters(), CFG.grad_clip); opt.step()
                if step % 500 == 0:
                    rows.append({"tail_step": step, **deps(m, task, 8, n_batches=4)})
            return {"share_after_upleg": d0["context_share"], "rows": rows}

        cached(st2, spec, _go, label=f"tail seed {sd}")

    TL = st2.frame(); TL.to_csv(csv("p11_long_tail"), index=False)
    if len(TL):
        tt = TL.groupby("tail_step")[["context_share", "acc_clean",
                                      "hidden_dep", "attn_dep"]].agg(["mean", "std"])
        n_seed = TL.seed.nunique()
        print(f"\nlong-tail recovery, {n_seed} seeds — share and accuracy together:")
        print(f"  {'step':>6}  {'share':>17}  {'accuracy':>17}  "
              f"{'hidden_dep':>10}  {'attn_dep':>9}")
        for s_, r in tt.iterrows():
            print(f"  {int(s_):>6}  "
                  f"{r[('context_share','mean')]:>7.4f}±{r[('context_share','std')]:<9.4f}  "
                  f"{r[('acc_clean','mean')]:>7.4f}±{r[('acc_clean','std')]:<9.4f}  "
                  f"{r[('hidden_dep','mean')]:>10.3f}  {r[('attn_dep','mean')]:>9.3f}")

        up = TL.groupby("seed").share_after_upleg.first()
        first, last = tt.index.min(), tt.index.max()
        s0 = tt.loc[first, ("context_share", "mean")]
        s1 = tt.loc[last, ("context_share", "mean")]
        print(f"\n  after the up-leg:  share {up.mean():.2e} (min {up.min():.1e}, "
              f"max {up.max():.1e})")
        print(f"  step {int(first):>5}: share {s0:.4f}, "
              f"accuracy {tt.loc[first, ('acc_clean','mean')]:.4f}")
        print(f"  step {int(last):>5}: share {s1:.4f}, "
              f"accuracy {tt.loc[last, ('acc_clean','mean')]:.4f}")

        # Is the share recovering, or is the model simply relearning the task?
        # The two are separable only if accuracy returns to ceiling first and the
        # share keeps climbing afterwards.
        ok = tt[tt[("acc_clean", "mean")] >= 0.99].index
        if len(ok):
            a0 = int(ok.min())
            print(f"\n  accuracy reached 0.99 at step {a0}, where the share was "
                  f"{tt.loc[a0, ('context_share','mean')]:.4f}")
            after = TL[TL.tail_step >= a0]
            print(f"  after that point the share rises "
                  f"{tt.loc[a0, ('context_share','mean')]:.4f} -> {s1:.4f} while "
                  f"accuracy is flat at {after.acc_clean.mean():.4f}"
                  f"±{after.acc_clean.std():.4f}")
            print(f"  share-accuracy correlation after convergence: "
                  f"{after.context_share.corr(after.acc_clean):+.2f}")
            print("  A near-zero correlation there is what separates 'the route is")
            print("  reopening' from 'the model is relearning the task'.")
        else:
            print("\n  !! accuracy never reached 0.99; the rising share is measured")
            print("     on a model that is not solving the task and cannot be read")
            print("     as the route recovering. Rest §5.4 on the threshold table.")

        # has it plateaued, or is it still climbing at the end of the budget?
        tail_q = TL[TL.tail_step >= last - 3000]
        sl = np.polyfit(tail_q.tail_step, tail_q.context_share, 1)[0] * 1000
        print(f"\n  slope over the last 3000 steps: {sl:+.4f} share per 1000 steps")
        print("  " + ("still climbing — report as 'recovering, not yet plateaued'"
                      if sl > 0.002 else "flat — the recovery has plateaued"))
        print(f"  hidden_dep falls {tt.loc[first, ('hidden_dep','mean')]:.3f} -> "
              f"{tt.loc[last, ('hidden_dep','mean')]:.3f} while attn_dep rises "
              f"{tt.loc[first, ('attn_dep','mean')]:.3f} -> "
              f"{tt.loc[last, ('attn_dep','mean')]:.3f}: the causal measure runs")
        print("  backwards too, which is stronger evidence than the variance share.")

    import shutil
    shutil.make_archive(os.environ.get("BACKUP_ZIP", "results/results_backup"), "zip", RESDIR)
    print("\nbacked up")

In [ ]:
TL = pd.read_csv(csv("p11_long_tail"))
print(TL.groupby("seed").agg(
    upleg=("share_after_upleg", "first"),
    acc_min=("acc_clean", "min"), acc_end=("acc_clean", "last"),
    share_end=("context_share", "last")).round(4).to_string())

In [ ]:
TL = pd.read_csv(csv("p11_long_tail"))
ok = TL[TL.seed != 3]
tt = ok.groupby("tail_step")[["context_share", "acc_clean",
                              "hidden_dep", "attn_dep"]].agg(["mean", "std"])
print(f"four seeds, excluding seed 3 (accuracy 0.850, never recovered):\n")
print(f"  step {int(tt.index.min())}: share "
      f"{tt.iloc[0][('context_share','mean')]:.4f}, accuracy "
      f"{tt.iloc[0][('acc_clean','mean')]:.4f}")
print(f"  step {int(tt.index.max())}: share "
      f"{tt.iloc[-1][('context_share','mean')]:.4f}"
      f"±{tt.iloc[-1][('context_share','std')]:.4f}, accuracy "
      f"{tt.iloc[-1][('acc_clean','mean')]:.4f}"
      f"±{tt.iloc[-1][('acc_clean','std')]:.4f}")
print(f"  accuracy across the whole window: {ok.acc_clean.mean():.4f}"
      f"±{ok.acc_clean.std():.4f}, min {ok.acc_clean.min():.4f}")
print(f"  share-accuracy correlation: {ok.context_share.corr(ok.acc_clean):+.2f}")
print(f"  hidden_dep {tt.iloc[0][('hidden_dep','mean')]:.3f} -> "
      f"{tt.iloc[-1][('hidden_dep','mean')]:.3f}   "
      f"attn_dep {tt.iloc[0][('attn_dep','mean')]:.3f} -> "
      f"{tt.iloc[-1][('attn_dep','mean')]:.3f}")

In [ ]:
# The objection that decides the paper: after cross-entropy saturates, the only
# live gradient is the penalty, so "reorganisation" might be nothing but norm
# shrinkage under a still-active regulariser.
#
# Two things settle it, and both are measurements rather than arguments.
#
#   1. Track the PENALTY GRADIENT MAGNITUDE alongside route dependence. If the
#      penalty's gradient has gone flat while D_h is still climbing, the drift
#      is not being driven by the term that is supposedly driving it.
#   2. Price the RECURRENT route at matched penalty magnitude. If that produces
#      a mirror-image handover, the mechanism is route-symmetric — a property of
#      pricing a pathway, not an artefact of shrinking one particular norm.
assert not IS_VISUAL, "run this on MODALITY = 'abstract' to match §5.1"
LOAD, LAM_C, STEPS_D = 8, 0.03, LONG
LOG = 500

st = Store("deflation")


def route_norms(m, task, load, n=4):
    m.eval()
    with torch.no_grad():
        lh, lc = [], []
        for _ in range(n):
            o = m(task.generate(CFG.batch_size, load)["x"])
            lh.append(o["parts"]["hidden"].norm(dim=-1).mean())
            lc.append(o["parts"]["context"].norm(dim=-1).mean())
    m.train()
    return float(torch.stack(lh).mean()), float(torch.stack(lc).mean())


def grad_norm(m, task, load, which, lam):
    """Gradient of the penalty term alone, and of the task term alone, so the
    two can be compared on the same batch at the same point in training."""
    m.train()
    b = task.generate(CFG.batch_size, load)

    def gn(loss):
        m.zero_grad(set_to_none=True)
        loss.backward(retain_graph=True)
        g = torch.sqrt(sum((p.grad ** 2).sum() for p in m.parameters()
                           if p.grad is not None))
        return float(g)

    o = m(b["x"])
    part = o["parts"]["context" if which == "retrieval" else "hidden"]
    pen = lam * part.norm(dim=-1).mean()
    g_pen = gn(pen) if lam > 0 else 0.0
    g_task = gn(F.cross_entropy(o["logits"], b["y"]))
    m.zero_grad(set_to_none=True)
    return g_pen, g_task


# lambda for the mirror arm is set so the two penalties have the same magnitude
# at initialisation; the routes have different natural scales, so using the same
# lambda would not be a matched comparison
set_seed(0)
_m = build("gru_attn", n_out=make_task("selective_recall").n_out).to(DEVICE)
_h0, _c0 = route_norms(_m, make_task("selective_recall"), LOAD)
LAM_H = LAM_C * _c0 / max(_h0, 1e-9)
print(f"at init  ||l_h|| = {_h0:.3f}, ||l_c|| = {_c0:.3f}")
print(f"matched prices: retrieval lambda = {LAM_C}, recurrent lambda = {LAM_H:.4f}\n")

ARMS = [("retrieval", LAM_C), ("recurrent", LAM_H), ("none", 0.0)]
for (which, lam), sd in itertools.product(ARMS, SEEDS_AUX):
    spec = {"block": "deflation", "priced": which, "lam": lam, "load": LOAD,
            "seed": sd, "steps": STEPS_D}

    def _go(which=which, lam=lam, sd=sd, spec=spec):
        task = make_task("selective_recall")
        set_seed(sd)
        m = build("gru_attn", n_out=task.n_out).to(DEVICE)
        opt = torch.optim.AdamW(m.parameters(), lr=CFG.lr,
                                weight_decay=CFG.weight_decay)
        rows = []
        for step in range(1, STEPS_D + 1):
            m.train()
            b = task.generate(CFG.batch_size, LOAD)
            o = m(b["x"])
            loss = F.cross_entropy(o["logits"], b["y"])
            if lam > 0:
                part = o["parts"]["context" if which == "retrieval" else "hidden"]
                loss = loss + lam * part.norm(dim=-1).mean()
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), CFG.grad_clip); opt.step()
            if step % LOG == 0 or step == 1:
                h, c = route_norms(m, task, LOAD)
                gp, gt = grad_norm(m, task, LOAD, which, lam)
                rows.append({"step": step, "l_h": h, "l_c": c,
                             "pen_value": lam * (c if which == "retrieval" else h),
                             "grad_penalty": gp, "grad_task": gt,
                             **deps(m, task, LOAD, n_batches=4)})
        return {"rows": rows}

    cached(st, spec, _go, label=f"priced {which:<10} seed {sd}")

DF = st.frame(); DF.to_csv(csv("p18_deflation"), index=False)

# ---- does dependence keep moving after the penalty gradient goes flat? -----
print("\nis the drift still driven by the penalty when the penalty stops pulling?\n")
for which in ["retrieval", "recurrent"]:
    g = DF[DF.priced == which]
    if not len(g):
        continue
    t = g.groupby("step")[["grad_penalty", "grad_task", "hidden_dep", "attn_dep",
                           "l_c", "l_h", "acc_clean"]].mean()
    peak = t.grad_penalty.max()
    flat = t.index[t.grad_penalty <= 0.1 * peak]
    print(f"  priced {which}:")
    print(f"    penalty gradient peaks at {peak:.4f}, "
          f"falls below 10% of peak at step "
          f"{int(flat.min()) if len(flat) else 'never'}")
    if len(flat):
        s0, s1 = int(flat.min()), int(t.index.max())
        key = "hidden_dep" if which == "retrieval" else "attn_dep"
        d0, d1 = t.loc[s0, key], t.loc[s1, key]
        total = t[key].iloc[-1] - t[key].iloc[0]
        after = d1 - d0
        print(f"    {key} at that step {d0:.3f}, at the end {d1:.3f}")
        print(f"    {after/max(total, 1e-9):.0%} of the total change happens AFTER")
        print(f"    the penalty gradient has fallen to a tenth of its peak")
        print(f"    penalty-to-task gradient ratio there: "
              f"{t.loc[s0, 'grad_penalty']/max(t.loc[s0, 'grad_task'], 1e-9):.3f}")

# ---- is the mechanism route-symmetric? ------------------------------------
print("\nmirror arm — pricing the recurrent route at matched magnitude:\n")
piv = DF.groupby(["priced", "step"])[["hidden_dep", "attn_dep", "acc_clean"]].mean()
for which in ["retrieval", "recurrent", "none"]:
    if which not in piv.index.get_level_values(0):
        continue
    t = piv.loc[which]
    print(f"  priced {which:<10} hidden_dep {t.hidden_dep.iloc[0]:.3f} -> "
          f"{t.hidden_dep.iloc[-1]:.3f}   attn_dep {t.attn_dep.iloc[0]:.3f} -> "
          f"{t.attn_dep.iloc[-1]:.3f}   acc {t.acc_clean.iloc[-1]:.4f}")
print("\n  A mirror-image handover means the effect is a property of pricing a")
print("  route, not of shrinking the particular norm the retrieval route has.")
print("  The unpriced arm is the necessity control: no drift without a price.")

# ---- figure: the penalty goes quiet, the dependence keeps moving -----------
fig, axes = plt.subplots(1, 2, figsize=(6.9, 2.5))
for j, which in enumerate(["retrieval", "recurrent"]):
    g = DF[DF.priced == which]
    if not len(g):
        axes[j].set_visible(False); continue
    t = g.groupby("step")[["grad_penalty", "hidden_dep", "attn_dep"]].mean()
    ax = axes[j]
    ax.plot(t.index, t.hidden_dep, color="#1b6ca8", label="recurrent dep")
    ax.plot(t.index, t.attn_dep, color="#d1495b", label="retrieval dep")
    ax.set_xlabel("training step"); ax.set_ylim(0, 1.05)
    ax.set_title(f"priced: {which}")
    a2 = ax.twinx()
    a2.plot(t.index, t.grad_penalty / max(t.grad_penalty.max(), 1e-9),
            color="#8d99ae", ls="--", lw=1.1)
    a2.axhline(0.1, color="#8d99ae", lw=.7, ls=":")
    a2.set_ylim(0, 1.05)
    if j == 1:
        a2.set_ylabel("penalty gradient (normalised)", color="#8d99ae")
axes[0].set_ylabel("accuracy lost when ablated")
axes[0].legend(frameon=False, loc="center right")
for ext in ("pdf", "png"):
    fig.savefig(os.path.join(FIGDIR, f"figX4_deflation_{TAG}.{ext}"))
plt.show()

import shutil
shutil.make_archive(os.environ.get("BACKUP_ZIP", "results/results_backup"), "zip", RESDIR)
print(f"\n{FIGDIR}/figX4_deflation_{TAG}.pdf   backed up")

In [ ]:
DF = pd.read_csv(csv("p18_deflation"))
for w in ["retrieval", "recurrent"]:
    t = DF[DF.priced == w].groupby("step")[
        ["grad_penalty", "grad_task", "hidden_dep", "attn_dep", "l_c", "l_h"]].mean()
    print(f"\npriced {w}:")
    print(f"  {'step':>6} {'g_pen':>9} {'g_task':>9} {'ratio':>8} "
          f"{'hid_dep':>8} {'attn_dep':>9}")
    for s in t.index[::max(len(t)//10, 1)]:
        r = t.loc[s]
        print(f"  {int(s):>6} {r.grad_penalty:>9.4f} {r.grad_task:>9.4f} "
              f"{r.grad_penalty/max(r.grad_task,1e-9):>8.2f} "
              f"{r.hidden_dep:>8.3f} {r.attn_dep:>9.3f}")

In [ ]:
t = DF.groupby(["priced", "step"])[["hidden_dep", "attn_dep"]].mean()
r, n = t.loc["recurrent"], t.loc["none"]
print("mirror arm vs unpriced control, per logged step:")
print((r - n).round(3).to_string())